# 🛠️ **Library**  
*Always run this block*


In [ ]:
import numpy as np
import pandas as pd

from scipy import ndimage

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.colors as mcolors
from matplotlib.pyplot import imsave

import torch
from IPython.display import clear_output
import os, subprocess, textwrap
from pathlib import Path
from tqdm import tqdm

from statistics import mode

import json
import os
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from google.colab import drive
import matplotlib
import skimage as ski
import seaborn as sns
drive.mount('/content/gdrive')
path="/content/gdrive/MyDrive/these/"


Mounted at /content/gdrive


In [ ]:
DONE_ASCII = r"""
 ____   ___  _   _  _____
|  _ \ / _ \| \ | || ____|
| | | | | | |  \| ||  _|
| |_| | |_| | |\  || |___
|____/ \___/|_| \_||_____|
"""



In [ ]:
def normalize(img):
  img=(img-np.min(img))/(np.max(img)-np.min(img))
  return img
def normalize_255(img):
  img=((img-np.min(img))/(np.max(img)-np.min(img)))*255
  return img


# 📂**Create the folders**
⚠️ USER INPUT REQUIRED  
*Always run this block*

## Function

In [ ]:
import os
from pathlib import Path


def setup_project_paths(
    drive_root: str = "/content/gdrive/MyDrive/",
    pipeline_folder: str = "pipeline",
    require_steps: bool = True,
    create_missing_folders: bool = True,
    ask_algo: bool = True,
    default_subpath_in_mydrive: str = "",  # if user presses Enter
):
    """
    Interactive project initializer for Google Colab.

    - Asks the user where the project is located under MyDrive (optional) + project name.
    - Checks that required folders exist (images/raw, segmentation masks, clustering/MFI).
    - Creates statistics folders if missing.
    - Asks for segmentation algorithm name and returns the filtered mask path.

    Returns
    -------
    paths : dict[str, str]
        Common pipeline paths (project root, images, masks, clustering, statistics, etc.)
    """

    drive_root = str(Path(drive_root))
    if not drive_root.endswith("/"):
        drive_root += "/"

    while True:
        name_path = input(
            "Enter the path where the project will be created "
            "(if the folder is in 'MyDrive' just press 'Enter'): "
        ).strip()
        if name_path == "":
            name_path = default_subpath_in_mydrive

        name_project = input("Enter the name of your project: ").strip()
        if name_project == "":
            print("❌ Project name cannot be empty.")
            continue

        project_root = Path(drive_root) / name_path / pipeline_folder / name_project
        project_root = project_root.resolve()

        if not project_root.is_dir():
            print(f"❌ The path does not exist: {project_root}")
            continue

        print(f"✅ Project: {project_root}/")
        break

    # Base folders
    path_raw = project_root / "images" / "raw"
    path_mcd = project_root / "images" / "mcd"
    path_img_segmentation=project_root / "images" / "images_segmentation"
    path_segmentation=project_root / "segmentation"
    path_segmentation_cells=path_segmentation / "cells"
    # Checks
    missing_msgs = []
    if not path_raw.is_dir():
        missing_msgs.append("❌ You have not completed the step of creating PNG images (missing: images/raw/)")
    # Create optional folders
    if create_missing_folders:
        if not path_raw.is_dir():
            path_raw.mkdir(parents=True, exist_ok=True)
            # keep same behavior message as your code
            print("✅ The raw image folder has been created")
        if not path_segmentation.is_dir():
            path_segmentation.mkdir(parents=True, exist_ok=True)
            print("✅ The segmentation folder has been created")
        if not path_segmentation_cells.is_dir():
            path_segmentation_cells.mkdir(parents=True, exist_ok=True)
            print("✅ The segmentation cells folder has been created")
        if not path_img_segmentation.is_dir():
            path_img_segmentation.mkdir(parents=True, exist_ok=True)
            print("✅ The images segmentation folder has been created")

    # Return as strings (easier to concatenate in notebooks)
    paths = {
        "project_root": str(project_root) + "/",
        "path_raw": str(path_raw) + "/",
        "path_segmentation": str(path_segmentation) + "/",
        "path_segmentation_cells": str(path_segmentation_cells) + "/",
        "path_img_segmentation": str(path_img_segmentation) + "/",
        "missing_required_steps": missing_msgs,  # empty list if all OK
    }

    return paths


## Execution

In [ ]:
paths = setup_project_paths()
path = paths["project_root"]
path_img_raw = paths["path_raw"]
path_img_segmentation = paths["path_img_segmentation"]
path_segm = paths["path_segmentation"]
path_segmentation_cells = paths["path_segmentation_cells"]


Enter the path where the project will be created (if the folder is in 'MyDrive' just press 'Enter'): these
Enter the name of your project: rejection
✅ Project: /content/gdrive/MyDrive/these/pipeline/rejection/


# 📊 **Images for segmentation**
⚠️ USER INPUT REQUIRED  
*Creating images combining multiple markers used for segmentation for different algorithms*

## 🛠️ Functions


In [ ]:
import os

def remove_image_extension(filename: str) -> str:
    """
    Remove image extension from filename, robust to multiple extensions
    (e.g. .ome.tif, .tar.gz-like patterns for images).
    """
    name = os.path.basename(filename)

    # Boucle tant que l'extension ressemble à une extension image
    image_exts = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp", ".ome"}

    while True:
        root, ext = os.path.splitext(name)
        if ext.lower() in image_exts:
            name = root
        else:
            break

    return name


In [ ]:
def normalize(img):
  img=(img-np.min(img))/(np.max(img)-np.min(img))
  return img
def normalize_255(img):
  img=((img-np.min(img))/(np.max(img)-np.min(img)))*255
  return img

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# variables globales (utilisées ensuite)
dna_marker = None
list_marker = []

def choose_markers_for_segmentation(list_all_markers):
    global dna_marker, list_marker

    dna_selector = widgets.Dropdown(
        options=list_all_markers,
        description="Nucleus:",
        layout=widgets.Layout(width="420px")
    )

    marker_selector = widgets.SelectMultiple(
        options=list_all_markers,
        description="Markers:",
        rows=min(12, len(list_all_markers)),
        layout=widgets.Layout(width="420px", height="220px")
    )

    confirm_btn = widgets.Button(description="✅ Confirm", button_style="success")
    out = widgets.Output()

    def on_confirm(_):
        global dna_marker, list_marker
        dna_marker = (dna_selector.value or "").strip()
        list_marker = [m.strip() for m in marker_selector.value if str(m).strip()]

        # optionnel: éviter que le marqueur noyau soit re-sélectionné
        list_marker = [m for m in list_marker if m != dna_marker]

        with out:
            out.clear_output(wait=True)
            print("✔ Nucleus marker:", dna_marker)
            print("✔ Markers used for segmentation:", list_marker)

    confirm_btn.on_click(on_confirm)

    display(
        widgets.HTML("<h3>🧬 Marker selection for cell segmentation</h3>"),
        dna_selector,
        marker_selector,
        confirm_btn,
        out
    )


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import tifffile as tiff
from tqdm.auto import tqdm

# Si tes fichiers sont de confiance, tu peux aussi activer ça :
Image.MAX_IMAGE_PIXELS = None

SUPPORTED_EXT = (".tif", ".tiff", ".png", ".jpeg", ".jpg")

def find_image_with_any_ext(folder, basename):
    for ext in SUPPORTED_EXT:
        p = folder / f"{basename}{ext}"
        if p.exists():
            return p
    return None

def read_image_float32(path: Path) -> np.ndarray:
    ext = path.suffix.lower()
    if ext in (".tif", ".tiff"):
        arr = tiff.imread(path)
    else:
        with Image.open(path) as im:
            arr = np.asarray(im)

    # si multi-canaux, prendre canal 0 (à adapter si besoin)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype(np.float32)

def build_segmentation_rgb_images(path_img_raw, path_img_segmentation, dna_marker, list_marker, strict=False):
    path_img_raw = Path(path_img_raw)
    path_out = Path(path_img_segmentation)
    path_out.mkdir(parents=True, exist_ok=True)

    rois = sorted([p for p in path_img_raw.iterdir() if p.is_dir()])

    for roi_dir in tqdm(rois, desc="Building segmentation composites", unit="ROI"):
        roi = roi_dir.name

        dna_path = find_image_with_any_ext(roi_dir, dna_marker)
        if dna_path is None:
            msg = f"[{roi}] DNA marker not found ({dna_marker}.*)"
            if strict:
                raise FileNotFoundError(msg)
            tqdm.write(msg + " -> skipped")
            continue

        dna = read_image_float32(dna_path)
        h, w = dna.shape[:2]
        img_tot = np.zeros((h, w, 3), dtype=np.float32)

        img_tot[:, :, 0] = normalize(dna)

        g_acc = np.zeros((h, w), dtype=np.float32)
        missing = []

        for marker in list_marker:
            m_path = find_image_with_any_ext(roi_dir, marker)
            if m_path is None:
                missing.append(marker)
                if strict:
                    raise FileNotFoundError(f"[{roi}] Marker not found: {marker}.*")
                continue

            m = read_image_float32(m_path)
            if m.shape[:2] != (h, w):
                msg = f"[{roi}] Size mismatch for {marker}"
                if strict:
                    raise ValueError(msg)
                tqdm.write(msg + " -> skipped")
                continue

            g_acc += normalize(m)

        img_tot[:, :, 1] = normalize(g_acc)

        img_u8 = normalize_255(img_tot).astype(np.uint8)

        # Écriture TIFF robuste (évite PIL pour l’écriture aussi)
        out_path = path_out / f"{roi}.tif"
        tiff.imwrite(out_path, img_u8, photometric="rgb")

        if missing:
            tqdm.write(f"[{roi}] Missing markers skipped: {', '.join(missing)}")

    print(f"✅ Images created for cell segmentation: {path_out}")


In [ ]:

# --------------------------------------------------
# Sélection interactive des marqueurs
# --------------------------------------------------

list_marker = []  # sera rempli après confirmation

def choose_markers_to_display_instanseg(list_all_markers):
    global list_marker

    selector = widgets.SelectMultiple(
        options=sorted(list_all_markers),
        description="Markers:",
        rows=min(12, len(list_all_markers)),
        layout=widgets.Layout(width="450px", height="240px")
    )

    confirm_btn = widgets.Button(
        description="✅ Confirm selection",
        button_style="success"
    )

    out = widgets.Output()

    def on_confirm(_):
        global list_marker
        list_marker = [m.strip() for m in selector.value if str(m).strip()]

        with out:
            out.clear_output(wait=True)
            print("✔ Selected markers:")
            for m in list_marker:
                print("  -", m)

    confirm_btn.on_click(on_confirm)

    display(
        widgets.HTML("<h3>🧬 Select the markers to display (including DNA)</h3>"),
        selector,
        confirm_btn,
        out
    )


In [ ]:
from pathlib import Path
import os
import numpy as np
import tifffile as tiff
from tqdm.auto import tqdm

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SUPPORTED_EXT = (".tif", ".tiff")  # on force TIFF pour workflow tiled robuste


# ------------------------------------------------------------
# I/O helpers
# ------------------------------------------------------------
def find_image_with_any_ext(folder: Path, basename: str) -> Path | None:
    """Retourne le premier fichier existant basename + extension supportée."""
    for ext in SUPPORTED_EXT:
        p = folder / f"{basename}{ext}"
        if p.exists():
            return p
    return None


def get_hw_tif(path: Path) -> tuple[int, int]:
    """Lit (H,W) via metadata TIFF, sans charger l'image."""
    with tiff.TiffFile(str(path)) as tf:
        page = tf.pages[0]
        shape = page.shape  # (H,W) ou (H,W,C)
        return int(shape[0]), int(shape[1])


def read_tile_any_tif(path: Path, y0: int, y1: int, x0: int, x1: int) -> np.ndarray:
    """
    Lit une tuile (y0:y1, x0:x1) d'un TIFF de façon compatible multi-versions tifffile.
    Stratégie:
      1) tिफffile.imread(..., key=(slice,slice)) si supporté (lecture partielle)
      2) tf.series[0].aszarr() puis slicing (souvent OK même compressé)
      3) fallback (lecture complète) -> à éviter, mais garantit de ne pas planter
    """
    # 1) Essai lecture partielle via key=
    try:
        arr = tiff.imread(str(path), key=(slice(y0, y1), slice(x0, x1)))
        if arr.ndim == 3:
            arr = arr[..., 0]
        return arr
    except TypeError:
        # key pas supporté par cette version/signature
        pass
    except Exception:
        # autre erreur -> on tente la méthode 2
        pass

    # 2) Essai via Zarr
    try:
        with tiff.TiffFile(str(path)) as tf:
            z = tf.series[0].aszarr()  # array-like lazy
            arr = np.asarray(z[y0:y1, x0:x1])
            if arr.ndim == 3:
                arr = arr[..., 0]
            return arr
    except Exception:
        pass

    # 3) Fallback complet
    arr = tiff.imread(str(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr[y0:y1, x0:x1]


# ------------------------------------------------------------
# Normalisation (tile-wise)
# ------------------------------------------------------------
def normalize_tile_percentile(x: np.ndarray, p_low=1.0, p_high=99.0) -> np.ndarray:
    """
    Normalisation robuste en [0,1] par percentiles sur LA TUILE.
    Avantage: 1 seule passe, RAM-safe.
    Inconvénient: petites variations possibles entre tuiles.
    """
    x = x.astype(np.float32, copy=False)
    if x.size == 0:
        return x

    lo = np.percentile(x, p_low)
    hi = np.percentile(x, p_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(x, dtype=np.float32)

    y = (x - lo) / (hi - lo)
    return np.clip(y, 0.0, 1.0)


def rgb01_to_u8(rgb01: np.ndarray) -> np.ndarray:
    return np.clip(rgb01 * 255.0, 0, 255).astype(np.uint8)


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def build_segmentation_rgb_images_tiled(
    path_img_raw,
    path_img_segmentation,
    dna_marker: str,
    list_marker: list[str],
    tile_size: int = 1024,
    strict: bool = False,
    p_low: float = 1.0,
    p_high: float = 99.0,
    compression: str | None = "deflate",  # None = pas de compression
):
    """
    Construit des composites RGB en mode tiled:

      R = DNA (normalisé)
      G = somme des markers (normalisés puis re-normalisés)
      B = 0

    Entrées: path_img_raw/<ROI>/<marker>.tif
    Sorties: path_img_segmentation/<ROI>.tif (RGB uint8)

    Paramètres:
    - tile_size: 512/1024/2048 selon disque/RAM
    - p_low/p_high: percentiles de normalisation (tuile)
    - compression: "deflate" (souvent bon), ou None
    """
    path_img_raw = Path(path_img_raw)
    path_out = Path(path_img_segmentation)
    path_out.mkdir(parents=True, exist_ok=True)

    rois = sorted([p for p in path_img_raw.iterdir() if p.is_dir()])

    for roi_dir in tqdm(rois, desc="Building tiled composites", unit="ROI"):
        roi = roi_dir.name

        # --- DNA path
        dna_path = find_image_with_any_ext(roi_dir, dna_marker)
        if dna_path is None:
            msg = f"[{roi}] DNA marker not found: {dna_marker}(.tif/.tiff)"
            if strict:
                raise FileNotFoundError(msg)
            tqdm.write(msg + " -> skipped")
            continue

        # dimensions
        try:
            H, W = get_hw_tif(dna_path)
        except Exception as e:
            msg = f"[{roi}] Cannot read DNA TIFF metadata: {dna_path.name} ({e})"
            if strict:
                raise
            tqdm.write(msg + " -> skipped")
            continue

        # resolve marker paths
        marker_paths: dict[str, Path] = {}
        missing: list[str] = []
        for m in list_marker:
            mp = find_image_with_any_ext(roi_dir, m)
            if mp is None:
                missing.append(m)
                if strict:
                    raise FileNotFoundError(f"[{roi}] Marker not found: {m}(.tif/.tiff)")
                continue

            # size check via metadata
            try:
                h2, w2 = get_hw_tif(mp)
                if (h2, w2) != (H, W):
                    msg = f"[{roi}] Size mismatch for {m}: {(h2,w2)} != {(H,W)}"
                    if strict:
                        raise ValueError(msg)
                    tqdm.write(msg + " -> skipped")
                    missing.append(m)
                    continue
            except Exception as e:
                msg = f"[{roi}] Cannot read marker metadata {m}: {mp.name} ({e})"
                if strict:
                    raise
                tqdm.write(msg + " -> skipped")
                missing.append(m)
                continue

            marker_paths[m] = mp

        # --- output temp raw buffer (memmap) to stay RAM-safe
        tmp_dat = path_out / f"{roi}__tmp_rgb_u8.dat"
        rgb_mm = np.memmap(str(tmp_dat), dtype=np.uint8, mode="w+", shape=(H, W, 3))

        # --- loop tiles
        for y0 in range(0, H, tile_size):
            y1 = min(y0 + tile_size, H)
            for x0 in range(0, W, tile_size):
                x1 = min(x0 + tile_size, W)

                # R: DNA
                dna_tile = read_tile_any_tif(dna_path, y0, y1, x0, x1)
                r01 = normalize_tile_percentile(dna_tile, p_low=p_low, p_high=p_high)

                # G: sum markers
                g_acc = np.zeros((y1 - y0, x1 - x0), dtype=np.float32)
                for m, mp in marker_paths.items():
                    mtile = read_tile_any_tif(mp, y0, y1, x0, x1)
                    g_acc += normalize_tile_percentile(mtile, p_low=p_low, p_high=p_high)

                g01 = normalize_tile_percentile(g_acc, p_low=p_low, p_high=p_high)

                # compose RGB
                rgb01 = np.zeros((y1 - y0, x1 - x0, 3), dtype=np.float32)
                rgb01[..., 0] = r01
                rgb01[..., 1] = g01
                # B stays 0

                rgb_mm[y0:y1, x0:x1, :] = rgb01_to_u8(rgb01)

        rgb_mm.flush()
        del rgb_mm  # ferme le memmap

        # --- write final tiled TIFF
        out_path = path_out / f"{roi}.tif"
        rgb_ro = np.memmap(str(tmp_dat), dtype=np.uint8, mode="r", shape=(H, W, 3))

        tiff.imwrite(
            str(out_path),
            rgb_ro,
            photometric="rgb",
            planarconfig="contig",
            tile=(tile_size, tile_size),
            compression=compression,
            bigtiff=True,
        )

        del rgb_ro
        os.remove(tmp_dat)

        if missing:
            tqdm.write(f"[{roi}] Missing/Skipped markers: {', '.join(sorted(set(missing)))}")

    print(f"✅ Tiled RGB composites saved in: {path_out}")


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Sequence, Dict, Any, Optional, Tuple, List

import numpy as np
from PIL import Image
import tifffile as tiff
from tqdm.auto import tqdm


SUPPORTED_EXT = (".tif", ".tiff", ".png", ".jpg", ".jpeg")


def _read_2d_image(path: Path, *, rgb_mode: str = "channel0") -> np.ndarray:
    """
    Read image and return a 2D array (H,W).
    rgb_mode:
      - "channel0": if RGB, take channel 0
      - "luma": convert RGB to luminance (0.299R+0.587G+0.114B)
    """
    ext = path.suffix.lower()
    if ext in (".tif", ".tiff"):
        img = tiff.imread(str(path))
    else:
        with Image.open(str(path)) as im:
            img = np.asarray(im)

    img = np.asarray(img)

    if img.ndim == 2:
        return img

    if img.ndim == 3:
        if img.shape[-1] in (3, 4):  # RGB/RGBA
            if rgb_mode == "luma":
                rgb = img[..., :3].astype(np.float32)
                y = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
                return y.astype(img.dtype) if np.issubdtype(img.dtype, np.integer) else y
            return img[..., 0]
        # other (H,W,C) but not RGB -> take first channel
        return img[..., 0]

    # multi-page tiff etc.: take first plane then ensure 2D
    if img.ndim > 3:
        img = img.reshape(-1, *img.shape[-2:])[0]
        if img.ndim != 2:
            raise RuntimeError(f"Unsupported image ndim after reshape: {path} -> {img.ndim}")
        return img

    raise RuntimeError(f"Unsupported image ndim={img.ndim} for {path}")


def build_instanseg_inputs(
    *,
    path_img_raw: str | Path,
    path_out: str | Path,
    list_marker: Sequence[str],
    extensions: Tuple[str, ...] = SUPPORTED_EXT,
    enforce_marker_order: bool = True,
    strict: bool = False,
    skip_if_missing_marker: bool = True,
    rgb_mode: str = "channel0",
    dtype_out: str = "float32",           # "float32" | "uint16" | "uint8"
    overwrite: bool = False,
    show_progress: bool = True,
) -> Dict[str, Any]:
    """
    Create one multi-channel TIFF per ROI for InstanSeg.

    Input layout:
      path_img_raw/
        ROI_001/
          CD3.tif
          CD4.png
          ...
        ROI_002/
          ...

    Output:
      path_out/ROI_001.tif  (C,H,W) with axes=CYX + Channel names metadata

    Parameters
    ----------
    list_marker:
      List of marker names WITHOUT extension. Example: ["DNA1", "CD3", "CD4", ...]
    enforce_marker_order:
      - True  -> channels are ordered exactly as list_marker (recommended for reproducibility)
      - False -> channels are ordered alphabetically by filename (your original behavior)
    strict:
      If True, raise on any issue (missing marker, shape mismatch, non-2D, etc.)
      If False, record issues and skip ROI or missing markers depending on flags.
    skip_if_missing_marker:
      If True, a ROI is skipped if any marker in list_marker is missing.
      If False, build with the subset found (still records missing markers).
    dtype_out:
      Output dtype; float32 is safe for ML pipelines.
    overwrite:
      If False, skip ROIs already written.

    Returns
    -------
    report dict with keys:
      - processed, skipped, failed
      - per_roi: list of {roi, out_path, channels, missing, reason}
    """
    path_img_raw = Path(path_img_raw)
    path_out = Path(path_out)
    path_out.mkdir(parents=True, exist_ok=True)

    ext_l = tuple(e.lower() for e in extensions)
    markers = list(list_marker)

    roi_dirs = sorted([p for p in path_img_raw.iterdir() if p.is_dir()])
    if not roi_dirs:
        raise RuntimeError(f"No ROI folders found in: {path_img_raw}")

    # dtype mapping
    dtype_map = {"float32": np.float32, "uint16": np.uint16, "uint8": np.uint8}
    if dtype_out not in dtype_map:
        raise ValueError(f"dtype_out must be one of {list(dtype_map)}")
    out_dtype = dtype_map[dtype_out]

    report = {"processed": 0, "skipped": 0, "failed": 0, "per_roi": []}

    iterator = tqdm(roi_dirs, desc="Building InstanSeg inputs", unit="ROI", ncols=110) if show_progress else roi_dirs

    for roi_dir in iterator:
        roi = roi_dir.name
        out_path = path_out / f"{roi}.tif"

        if out_path.exists() and not overwrite:
            report["skipped"] += 1
            report["per_roi"].append(
                {"roi": roi, "out_path": str(out_path), "channels": [], "missing": [], "reason": "exists"}
            )
            continue

        # list candidate files
        files = [p for p in roi_dir.iterdir() if p.is_file() and p.suffix.lower() in ext_l]
        by_stem = {p.stem: p for p in files}  # assumes unique stems; if not, you can adapt

        missing = [m for m in markers if m not in by_stem]
        if missing and skip_if_missing_marker:
            msg = f"missing markers: {missing}"
            if strict:
                raise FileNotFoundError(f"[{roi}] {msg}")
            report["skipped"] += 1
            report["per_roi"].append(
                {"roi": roi, "out_path": str(out_path), "channels": [], "missing": missing, "reason": msg}
            )
            continue

        # select files in desired order
        if enforce_marker_order:
            selected = [(m, by_stem[m]) for m in markers if m in by_stem]
        else:
            selected = sorted([(p.stem, p) for p in files if p.stem in set(markers)], key=lambda x: x[0].lower())

        if not selected:
            msg = "no valid marker files found after filtering"
            if strict:
                raise RuntimeError(f"[{roi}] {msg}")
            report["skipped"] += 1
            report["per_roi"].append(
                {"roi": roi, "out_path": str(out_path), "channels": [], "missing": missing, "reason": msg}
            )
            continue

        # read channels
        channels = []
        channel_names = []
        shapes = set()

        try:
            for name, p in selected:
                img = _read_2d_image(p, rgb_mode=rgb_mode)
                shapes.add(img.shape)
                channels.append(img)
                channel_names.append(name)

            if len(shapes) != 1:
                msg = f"shape mismatch across channels: {sorted(shapes)}"
                if strict:
                    raise RuntimeError(f"[{roi}] {msg}")
                report["failed"] += 1
                report["per_roi"].append(
                    {"roi": roi, "out_path": str(out_path), "channels": channel_names, "missing": missing, "reason": msg}
                )
                continue

            multiplex = np.stack(channels, axis=0)  # (C,H,W)

            # cast
            if out_dtype in (np.uint8, np.uint16):
                # if float input, clip
                if np.issubdtype(multiplex.dtype, np.floating):
                    multiplex = np.nan_to_num(multiplex, nan=0.0, posinf=0.0, neginf=0.0)
                    multiplex = np.clip(multiplex, 0, np.iinfo(out_dtype).max)
                multiplex = multiplex.astype(out_dtype, copy=False)
            else:
                multiplex = multiplex.astype(np.float32, copy=False)

            tiff.imwrite(
                str(out_path),
                multiplex,
                photometric="minisblack",
                metadata={"axes": "CYX", "Channel": {"Name": channel_names}},
            )

            report["processed"] += 1
            report["per_roi"].append(
                {"roi": roi, "out_path": str(out_path), "channels": channel_names, "missing": missing, "reason": "ok"}
            )

        except Exception as e:
            if strict:
                raise
            report["failed"] += 1
            report["per_roi"].append(
                {"roi": roi, "out_path": str(out_path), "channels": channel_names, "missing": missing, "reason": str(e)}
            )

    print(f"✅ Done. Output folder: {path_out}")
    print(f"Processed={report['processed']} | Skipped={report['skipped']} | Failed={report['failed']}")
    return report

## ⚙️ Execution
⚠️ USER INPUT REQUIRED

#### 📊 Image used for the segmentation by Mesmer and Cellpose
⚠️ USER INPUT REQUIRED  
*Images in 2 dimensions: the first one with the nucleus marker and the second one with the sum of the choosen membrane markers*

In [ ]:
path_img_segmentation_mesmer_cellpose=path_img_segmentation+"mesmer_cellpose/"
if os.path.isdir(path_img_segmentation_mesmer_cellpose)==False:
  os.mkdir(path_img_segmentation_mesmer_cellpose)

In [ ]:
roi=os.listdir(path_img_raw)[0]
list_all_markers=[remove_image_extension(f) for f in os.listdir(path_img_raw+roi)]
choose_markers_for_segmentation(list_all_markers)


HTML(value='<h3>🧬 Marker selection for cell segmentation</h3>')

Dropdown(description='Nucleus:', layout=Layout(width='420px'), options=('MPO', 'Ki67', 'Vimentin', 'CD14', 'Tb…

SelectMultiple(description='Markers:', layout=Layout(height='220px', width='420px'), options=('MPO', 'Ki67', '…

Button(button_style='success', description='✅ Confirm', style=ButtonStyle())

Output()

In [ ]:
build_segmentation_rgb_images(
    path_img_raw=path_img_raw,
    path_img_segmentation=path_img_segmentation_mesmer_cellpose,
    dna_marker=dna_marker,
    list_marker=list_marker,
    strict=False
)


Building segmentation composites:   0%|          | 0/89 [00:00<?, ?ROI/s]

✅ Images created for cell segmentation: /content/gdrive/MyDrive/these/pipeline/rejection/images/images_segmentation/mesmer_cellpose


In [ ]:
build_segmentation_rgb_images_tiled(
    path_img_raw=path_img_raw,
    path_img_segmentation=path_img_segmentation,
    dna_marker=dna_marker,
    list_marker=list_marker,
    tile_size=1024,   # 512/1024/2048 selon disque/RAM
    strict=False,
    p_low=1,
    p_high=99,
)


#### 📊 Image used for the segmentation by Instanseg
The images have as many dimensions as the number of markers chosen by the user

In [ ]:
path_img_segmentation_instanseg=path+"images/images_segmentation/instanseg/"
if os.path.isdir(path_img_segmentation_instanseg)==False:
  os.mkdir(path_img_segmentation_instanseg)

In [ ]:
roi=os.listdir(path_img_raw)[0]
list_all_markers=[remove_image_extension(f) for f in os.listdir(path_img_raw+roi)]
choose_markers_to_display_instanseg(list_all_markers)

HTML(value='<h3>🧬 Select the markers to display (including DNA)</h3>')

SelectMultiple(description='Markers:', layout=Layout(height='240px', width='450px'), options=('Aquaporin1', 'C…

Button(button_style='success', description='✅ Confirm selection', style=ButtonStyle())

Output()

In [ ]:
report = build_instanseg_inputs(
    path_img_raw=path_img_raw,
    path_out=path_img_segmentation_instanseg,
    list_marker=list_marker,
    enforce_marker_order=True,   # conseillé
    skip_if_missing_marker=True,
    strict=False,
    dtype_out="float32",
    overwrite=False,
)

Building InstanSeg inputs:   0%|                                                      | 0/89 [00:00<?, ?ROI/s]

✅ Done. Output folder: /content/gdrive/MyDrive/these/pipeline/rejection/images/images_segmentation/instanseg
Processed=89 | Skipped=0 | Failed=0


# ⚙️ **Segmentation of all the images**

In [ ]:
path_segmentation_cells_mask=path+"segmentation/cells/mask/"
path_img_segmentation_mesmer_cellpose=path_img_segmentation+"mesmer_cellpose/"
path_img_segmentation_instanseg=path+"images/images_segmentation/instanseg/"
path_segmentation_cells_mask_color=path+"segmentation/cells/mask_color/"
if os.path.isdir(path_segmentation_cells_mask)==False:
  os.mkdir(path_segmentation_cells_mask)
if not os.path.isdir(path_segmentation_cells_mask_color):
  os.mkdir(path_segmentation_cells_mask_color)

### 📊 Segmentation by mesmer
⚠️ USER INPUT REQUIRED

In [ ]:
 path_mask_mesmer=path_segmentation_cells_mask+"mesmer/"
 if os.path.isdir(path_mask_mesmer)==False:
    os.mkdir(path_mask_mesmer)
    print("✅ Folder for mesmer masks created")
 else:
    print("✅ Folder for mesmer masks already created")
 path_mask_color_mesmer=path_segmentation_cells_mask_color+"mesmer/"
 if os.path.isdir(path_mask_color_mesmer)==False:
    os.mkdir(path_mask_color_mesmer)
    print("✅ Folder for color mesmer masks created")
 else:
    print("✅ Folder for color mesmer masks already created")

✅ Folder for mesmer masks already created
✅ Folder for color mesmer masks already created


#### ⚙️ Segmentation
⚠️ USER INPUT REQUIRED

##### 🛠️ Functions

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


def export_colored_instance_masks_flat(
    *,
    path_mask: str,
    path_mask_color: str,
    seed: int = 0,
    use_all_matplotlib_colors: bool = True,
    exts_in: Tuple[str, ...] = (".tif", ".tiff", ".png"),
    progress: bool = True,
    progress_ncols: int = 110,
    background_color: Tuple[float, float, float] = (0.0, 0.0, 0.0),
) -> Dict[str, int]:
    """
    Colorize an *instance-labeled* mask folder (flat layout) WITHOUT re-labeling.

    Assumes:
      - mask pixels are integer labels
      - 0 = background
      - each object has its own label value (1..N, not necessarily contiguous)

    Output:
      - saves PNG RGB images with same stem into `path_mask_color`.

    Returns: processed / failed / output_root
    """
    path_mask = Path(path_mask)
    path_mask_color = Path(path_mask_color)
    path_mask_color.mkdir(parents=True, exist_ok=True)

    # Palette (RGB in [0,1])
    if use_all_matplotlib_colors:
        palette = [matplotlib.colors.to_rgb(hx) for hx in matplotlib.colors.cnames.values()]
    else:
        palette = [matplotlib.colors.to_rgb(hx) for hx in [
            "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
            "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
        ]]
    if not palette:
        raise RuntimeError("No colors available for palette.")
    palette = np.asarray(palette, dtype=np.float32)

    rng = np.random.default_rng(seed)

    # List mask files (flat)
    exts_l = tuple(e.lower() for e in exts_in)
    mask_files = sorted([p for p in path_mask.iterdir() if p.is_file() and p.suffix.lower() in exts_l])
    if not mask_files:
        raise RuntimeError(f"No mask files found in: {path_mask}")

    processed = 0
    failed = 0
    pbar = tqdm(total=len(mask_files), desc="Colorizing instance masks", unit="mask", ncols=progress_ncols) if progress else None

    for mask_path in mask_files:
        if pbar is not None:
            pbar.set_postfix_str(mask_path.name, refresh=False)

        try:
            mask = np.array(Image.open(mask_path))

            # Ensure integer labels
            if mask.dtype.kind not in ("u", "i"):
                # If mask was saved as float, cast safely (but ideally masks should be int)
                mask = mask.astype(np.int32)

            labels = np.unique(mask)
            labels = labels[labels != 0]  # exclude background

            H, W = mask.shape[:2]
            canvas = np.zeros((H, W, 3), dtype=np.float32)
            canvas[...] = background_color

            if labels.size > 0:
                # Pick a random color for each label present in this image (deterministic seed + RNG state)
                # We'll create a dict-like mapping using two arrays and vectorized indexing.
                # 1) draw colors for the present labels
                chosen_idx = rng.integers(0, len(palette), size=labels.size)
                chosen_colors = palette[chosen_idx]  # (n_labels, 3)

                # 2) map pixels: use an index image via searchsorted on sorted labels
                labels_sorted = np.sort(labels)
                colors_sorted = chosen_colors[np.argsort(labels)]  # align with labels_sorted

                # create an index for all pixels whose label != 0
                m = mask
                fg = (m != 0)
                idx = np.searchsorted(labels_sorted, m[fg])
                canvas[fg] = colors_sorted[idx]

            out_path = path_mask_color / f"{mask_path.stem}.png"
            plt.imsave(str(out_path), canvas)  # float in [0,1]
            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {mask_path.name}: {e}")
            else:
                print(f"[WARN] Failed {mask_path.name}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    print(f"✅ Folder with color masks available there: {path_mask_color}")

    return {"processed": processed, "failed": failed, "output_root": str(path_mask_color)}

In [ ]:
# Étape 1 : Installer micromamba (mini version d'Anaconda compatible Colab)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!mkdir -p /root/micromamba/envs

# Étape 2 : Créer un environnement Python 3.9 avec micromamba
!./bin/micromamba create -y -p /root/micromamba/envs/deepcell-env python=3.9

# Étape 3 : Activer l'environnement et installer DeepCell + dépendances
!./bin/micromamba run -p /root/micromamba/envs/deepcell-env pip install deepcell==0.12.6 scikit-image matplotlib

# Étape 4 : Démarrer Python dans ce nouvel environnement
import os
from IPython.display import clear_output

os.environ['PYTHONPATH'] = "/root/micromamba/envs/deepcell-env/lib/python3.9/site-packages"
os.environ['PATH'] = "/root/micromamba/envs/deepcell-env/bin:" + os.environ['PATH']
clear_output()
print("✅ Environnement Python 3.9 avec DeepCell prêt dans Colab")


✅ Environnement Python 3.9 avec DeepCell prêt dans Colab


##### Segmentation of small biopsies
⚠️ USER INPUT REQUIRED


In [ ]:
resolution=float(input("Enter the resolution in micrometer per pixel (depending on the size of the biopsie, 1.5 mpp by default):"))

Enter the resolution in micrometer per pixel (depending on the size of the biopsie, 1.5 mpp by default):1


In [ ]:
code = f"""
from deepcell.applications import Mesmer
from tifffile import imsave
from PIL import Image
import numpy as np
import os
import traceback

project = "rejection"
path = "{path}"
path_img = "{path_img_segmentation_mesmer_cellpose}"
path_mask_mesmer = "{path_mask_mesmer}"

# Vérifie que les dossiers existent
if not os.path.isdir(path_img):
    raise FileNotFoundError(f"📁 Dossier d'images introuvable : {{path_img}}")
if not os.path.isdir(path_mask_mesmer):
    os.makedirs(path_mask_mesmer)
    print(f"📂 Dossier créé : {{path_mask_mesmer}}")

# Charger le modèle
try:
    app = Mesmer()
    print("🧠 Modèle Mesmer chargé avec succès.")
except Exception as e:
    raise RuntimeError("❌ Échec du chargement de Mesmer : " + str(e))

# Traiter les images
for img_file in os.listdir(path_img):
    try:
        if not img_file.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
            continue

        output_file = os.path.join(path_mask_mesmer, img_file.rsplit(".", 1)[0] + ".tif")
        if os.path.isfile(output_file):
            print(f"⏭️ Déjà traité : {{img_file}}")
            continue

        img_path = os.path.join(path_img, img_file)
        img = np.array(Image.open(img_path))

        # Utiliser uniquement les deux premiers canaux
        img = img[:, :, :2] if img.ndim == 3 and img.shape[2] >= 2 else np.stack([img] * 2, axis=-1)
        img = np.expand_dims(img, axis=0)

        predictions = app.predict(img, image_mpp={resolution})
        mask = predictions[0, :, :, 0].astype(np.uint16)

        imsave(output_file, mask)
        print("✅ Traitée :", img_file)

    except Exception as e:
        print(f"❌ Erreur avec {{img_file}} : {{e}}")
        traceback.print_exc()
    print("✅ Folder for mesmer masks available there: output_file")

"""


In [ ]:
with open("run_mesmer.py", "w") as f:
    f.write(code)

# Exécution
!./bin/micromamba run -p /root/micromamba/envs/deepcell-env python run_mesmer.py


2026-03-06 14:06:17.770368: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /root/micromamba/envs/deepcell-env/lib/python3.9/site-packages/cv2/../../lib64:/usr/local/lib/python3.12/dist-packages/cv2/../../lib64:/usr/lib64-nvidia
2026-03-06 14:06:21.528223: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /root/micromamba/envs/deepcell-env/lib/python3.9/site-packages/cv2/../../lib64:/usr/local/lib/python3.12/dist-packages/cv2/../../lib64:/usr/lib64-nvidia
2026-03-06 14:06:21.528382: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file

##### Segmentation of big biopsies
⚠️ USER INPUT REQUIRED


In [ ]:

tile_size=int(input("Enter the size of the tiles (1024 or 512): "))
overlap=int(input("Enter the size of the overlap (256): "))
resolution=float(input("Enter the resolution of the images (0.5,1.5): "))

In [ ]:
code = f"""
from deepcell.applications import Mesmer
from tifffile import imwrite, memmap, TiffFile
from PIL import Image
import numpy as np
import os
import traceback
from tqdm.auto import tqdm

project = "rejection"
path = "{path}"
path_img = "{path_img_segmentation_mesmer_cellpose}"
path_mask_mesmer = "{path_mask_mesmer}"

# =========================
# Paramètres tiling + fusion
# =========================
TILE = {tile_size}            # ex: 512 ou 1024
OVERLAP = {overlap}           # ex: 128 (≈ 25% si TILE=512)
IMAGE_MPP = {resolution}      # ex: 0.5

# Fusion: seuil de chevauchement dans la zone de recouvrement
# ratio = intersection / min(area_tile, area_global_overlap)
FUSION_MIN_OVERLAP_RATIO = 0.30

# Optionnel: ignorer tuiles trop vides (accélération)
SKIP_EMPTY = True
EMPTY_THRESHOLD = 0.98  # fraction de pixels ~0 (sur canal 0) au-dessus de laquelle on skip

# =========================
# Utilitaires
# =========================
def ensure_two_channels(arr: np.ndarray) -> np.ndarray:
    \"\"\"Retourne un array HxWx2.\"\"\"
    if arr.ndim == 2:
        arr = np.stack([arr, arr], axis=-1)
    elif arr.ndim == 3:
        if arr.shape[2] >= 2:
            arr = arr[:, :, :2]
        else:
            arr = np.repeat(arr, 2, axis=2)
    else:
        raise ValueError(f"Format d'image inattendu: shape={{arr.shape}}")
    return arr

def load_image_lazy(img_path: str) -> np.ndarray:
    \"\"\"
    Pour TIFF: memmap (lecture à la demande, utile pour énormes images).
    Pour PNG/JPG: charge en RAM via PIL.
    \"\"\"
    ext = os.path.splitext(img_path)[1].lower()
    if ext in (".tif", ".tiff"):
        try:
            return memmap(img_path)
        except Exception:
            with TiffFile(img_path) as tif:
                return tif.asarray()
    else:
        return np.array(Image.open(img_path))

def iter_tiles(H: int, W: int, tile: int, overlap: int):
    \"\"\"Génère (y0,y1,x0,x1) couvrant toute l'image avec recouvrement.\"\"\"
    step = tile - overlap
    if step <= 0:
        raise ValueError("OVERLAP doit être strictement inférieur à TILE")

    ys = list(range(0, max(H - tile, 0) + 1, step))
    xs = list(range(0, max(W - tile, 0) + 1, step))

    if len(ys) == 0:
        ys = [0]
    if len(xs) == 0:
        xs = [0]
    if ys[-1] != max(H - tile, 0):
        ys.append(max(H - tile, 0))
    if xs[-1] != max(W - tile, 0):
        xs.append(max(W - tile, 0))

    for y0 in ys:
        y1 = min(y0 + tile, H)
        for x0 in xs:
            x1 = min(x0 + tile, W)
            yield y0, y1, x0, x1

def maybe_skip_tile(tile_img: np.ndarray) -> bool:
    if not SKIP_EMPTY:
        return False
    ch0 = tile_img[..., 0].astype(np.float32)
    frac_zero = np.mean(ch0 <= 0)
    return frac_zero >= EMPTY_THRESHOLD

def fuse_tile_into_global(full_mask: np.ndarray,
                          tile_mask_local: np.ndarray,
                          y0: int, y1: int, x0: int, x1: int,
                          next_id: int,
                          min_overlap_ratio: float):
    \"\"\"
    Fusionne les labels d'une tuile avec le masque global sur la zone (y0:y1, x0:x1).
    Retourne (next_id_updated, n_fused, n_new).
    \"\"\"
    region_global = full_mask[y0:y1, x0:x1]
    tile = tile_mask_local

    # Zone de conflit = overlap où les deux ont des labels
    overlap_mask = (region_global > 0) & (tile > 0)
    mapping = {{}}  # tile_label -> global_label
    fused = 0

    if np.any(overlap_mask):
        tg = tile[overlap_mask].astype(np.int64)
        gg = region_global[overlap_mask].astype(np.int64)

        # Pour chaque label tuile, trouver le global le plus fréquent en overlap
        # On calcule : counts[(tile_label, global_label)] = nb pixels
        pairs = np.stack([tg, gg], axis=1)
        # unique pairs + counts
        uniq_pairs, counts = np.unique(pairs, axis=0, return_counts=True)

        # area_tile_overlap[t] = pixels de t présents dans overlap
        tile_labels, tile_counts = np.unique(tg, return_counts=True)
        area_tile_overlap = {{int(t): int(c) for t, c in zip(tile_labels, tile_counts)}}

        # Pour estimer area_global_overlap[g] (pixels de g présents dans overlap)
        global_labels, global_counts = np.unique(gg, return_counts=True)
        area_global_overlap = {{int(g): int(c) for g, c in zip(global_labels, global_counts)}}

        # meilleur match par tile label
        best = {{}}
        for (t, g), c in zip(uniq_pairs, counts):
            t = int(t); g = int(g); c = int(c)
            if t not in best or c > best[t][1]:
                best[t] = (g, c)

        for t, (g, inter) in best.items():
            denom = min(area_tile_overlap.get(t, inter), area_global_overlap.get(g, inter))
            ratio = inter / max(denom, 1)
            if ratio >= min_overlap_ratio:
                mapping[t] = g
                fused += 1

    # Assigner IDs globaux pour labels non mappés
    tile_labels_all = np.unique(tile)
    tile_labels_all = tile_labels_all[tile_labels_all > 0]

    new_assigned = 0
    for t in tile_labels_all:
        t = int(t)
        if t in mapping:
            continue
        mapping[t] = next_id
        next_id += 1
        new_assigned += 1

    # Appliquer mapping
    # (lookup vectorisé via tableau de correspondance)
    max_t = int(tile.max())
    lut = np.zeros(max_t + 1, dtype=np.uint32)
    for t, g in mapping.items():
        if t <= max_t:
            lut[t] = np.uint32(g)

    tile_global = lut[tile.astype(np.int64)]

    # Écriture prudente : ne pas écraser un label global différent
    # On écrit:
    # - les pixels où global == 0
    # - ou les pixels où global == tile_global (cohérence)
    write_mask = (region_global == 0) | (region_global == tile_global)
    region_global[write_mask] = tile_global[write_mask]
    full_mask[y0:y1, x0:x1] = region_global

    return next_id, fused, new_assigned

# =========================
# Vérification dossiers
# =========================
if not os.path.isdir(path_img):
    raise FileNotFoundError(f"📁 Dossier d'images introuvable : {{path_img}}")
if not os.path.isdir(path_mask_mesmer):
    os.makedirs(path_mask_mesmer, exist_ok=True)
    print(f"📂 Dossier créé : {{path_mask_mesmer}}")

# =========================
# Charger Mesmer
# =========================
try:
    app = Mesmer()
    print("🧠 Modèle Mesmer chargé avec succès.")
except Exception as e:
    raise RuntimeError("❌ Échec du chargement de Mesmer : " + str(e))

# =========================
# Traitement images
# =========================
for img_file in os.listdir(path_img):
    try:
        if not img_file.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
            continue

        img_path = os.path.join(path_img, img_file)
        output_file = os.path.join(path_mask_mesmer, img_file)

        if os.path.isfile(output_file):
            print(f"⏭️ Déjà traité : {{img_file}}")
            continue

        img = load_image_lazy(img_path)
        if img.ndim == 4 and img.shape[0] == 1:
            img = img[0]
        img = ensure_two_channels(img)

        H, W, C = img.shape
        print(f"🧩 {{img_file}} | shape={{img.shape}} | TILE={{TILE}} OVERLAP={{OVERLAP}} | MPP={{IMAGE_MPP}}")

        full_mask = np.zeros((H, W), dtype=np.uint32)

        tiles = list(iter_tiles(H, W, TILE, OVERLAP))
        pbar = tqdm(tiles, desc=f"Mesmer+fusion tiles: {{img_file}}", unit="tile", leave=True)

        next_global_id = 1
        n_skipped = 0
        total_fused = 0
        total_new = 0

        for (y0, y1, x0, x1) in pbar:
            tile_img = img[y0:y1, x0:x1, :]

            # Pad si bord
            pad_h = TILE - (y1 - y0)
            pad_w = TILE - (x1 - x0)
            if pad_h > 0 or pad_w > 0:
                tile_padded = np.pad(
                    tile_img,
                    ((0, pad_h), (0, pad_w), (0, 0)),
                    mode="constant",
                    constant_values=0
                )
            else:
                tile_padded = tile_img

            if maybe_skip_tile(tile_padded):
                n_skipped += 1
                pbar.set_postfix(skipped=n_skipped, fused=total_fused, new=total_new, max_id=int(full_mask.max()))
                continue

            inp = np.expand_dims(tile_padded, axis=0)
            preds = app.predict(inp, image_mpp=IMAGE_MPP)

            tile_mask = preds[0, :, :, 0].astype(np.uint32)
            # enlever padding
            tile_mask = tile_mask[: (y1 - y0), : (x1 - x0)]

            # ✅ fusion dans le masque global sur la zone de la tuile
            next_global_id, n_fused, n_new = fuse_tile_into_global(
                full_mask, tile_mask, y0, y1, x0, x1, next_global_id, FUSION_MIN_OVERLAP_RATIO
            )
            total_fused += n_fused
            total_new += n_new

            pbar.set_postfix(skipped=n_skipped, fused=total_fused, new=total_new, max_id=int(full_mask.max()))

        max_id = int(full_mask.max())
        out = full_mask.astype(np.uint16) if max_id <= 65535 else full_mask.astype(np.uint32)
        imwrite(output_file, out, compression="zlib")
        print(f"✅ Traitée : {{img_file}} | tiles={{len(tiles)}} skipped={{n_skipped}} | fused={{total_fused}} | max_id={{max_id}}")
        print(f"📦 Masque : {{output_file}}")

    except Exception as e:
        print(f"❌ Erreur avec {{img_file}} : {{e}}")
        traceback.print_exc()

print("✅ Folder for Mesmer fused masks available there:", path_mask_mesmer)
"""


In [ ]:
with open("run_mesmer.py", "w") as f:
    f.write(code)

# Exécution
!./bin/micromamba run -p /root/micromamba/envs/deepcell-env python run_mesmer.py


#### 📊 Colored masks

In [ ]:
report = export_colored_instance_masks_flat(
     path_mask=path_mask_mesmer,
     path_mask_color=path_mask_color_mesmer,
     seed=0,
)
print(report)

Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/mesmer
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/mesmer'}


### 📊 Cellpose v3
It's necessary to restart the runtime after the mesmer segmentation (runtime --> restart runtime then execute the two first blocs)

In [ ]:
 path_mask_cellposev3=path_segmentation_cells_mask+"cellposev3/"
 if os.path.isdir(path_mask_cellposev3)==False:
    os.mkdir(path_mask_cellposev3)
    print("✅ Folder for cellpose v3 mask created")
 path_mask_color_cellposev3=path_segmentation_cells_mask_color+"cellposev3/"
 if os.path.isdir(path_mask_color_cellposev3)==False:
    os.mkdir(path_mask_color_cellposev3)
    print("✅ Folder for color cellpose v3 masks created")

#### 🛠️ Installation of cellpose

In [ ]:
%pip install cellpose
from cellpose import core, utils, io, models, metrics
model = models.CellposeModel(gpu=True, model_type="cyto3")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.1/213.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 106.1 MB/s eta 0:00:00


100%|██████████| 1.15G/1.15G [00:08<00:00, 144MB/s]


In [ ]:
# =========================================================
# Cellpose CYTO3 (membrane+nucleus) — Interactive Colab UI (CROP preview)
#
# ✅ Preview = crop only (user chooses crop size + mode)
# ✅ LEFT  = chosen DISPLAY marker (from display_root/<biopsy>/) with instance-accurate outlines
# ✅ RIGHT = colored instance mask of the crop (many colors)
# ✅ Batch = segment ALL images (full resolution)
#     - save raw masks (.tif)    -> mask_out_root/<seg_stem>__mask.tif   (NO subfolders)
#     - save colored masks (.png)-> color_out_root/<seg_stem>__mask.png  (NO subfolders)
#
# Folder layout:
# 1) seg_root (FLAT)
#    seg_root/
#      <biopsy_id>__something.tif   (recommended: multi-channel HxWxC or CxHxW)
#      ...
#
# 2) display_root (NESTED)
#    display_root/
#      <biopsy_id>/
#        DNA.png / Ir191_193.tif / any markers...
#
# Outputs:
# - mask_out_root/       (flat)
# - color_out_root/      (flat)
#
# Cellpose v3/v4 compatible.
# =========================================================

# If needed (Colab):
# %pip -q install "cellpose>=3" tifffile imageio scikit-image ipywidgets tqdm opencv-python
# Then Runtime -> Restart

from __future__ import annotations

from pathlib import Path
from typing import Optional, Iterable, Tuple, Dict, Any, List

import numpy as np
import tifffile as tiff
import imageio.v3 as iio
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

from tqdm.auto import tqdm
from skimage.morphology import remove_small_objects

import cv2
from cellpose import models


# -----------------------------
# Globals
# -----------------------------
IMG_EXTS = (".tif", ".tiff", ".ome.tif", ".ome.tiff", ".png", ".jpg", ".jpeg")


# =========================================================
# I/O helpers
# =========================================================
def list_files_flat(folder: str | Path, exts: Iterable[str] = IMG_EXTS) -> list[Path]:
    folder = Path(folder)
    files = [p for p in folder.iterdir() if p.is_file() and p.name.lower().endswith(tuple(exts))]
    return sorted(files, key=lambda p: p.name.lower())


def list_subdirs(folder: str | Path) -> list[str]:
    folder = Path(folder)
    return sorted([p.name for p in folder.iterdir() if p.is_dir()])


def list_marker_files(biopsy_dir: str | Path, exts: Iterable[str] = IMG_EXTS) -> list[str]:
    biopsy_dir = Path(biopsy_dir)
    files = [p.name for p in biopsy_dir.iterdir() if p.is_file() and p.name.lower().endswith(tuple(exts))]
    return sorted(files, key=lambda s: s.lower())


def read_image_any(path: str | Path) -> np.ndarray:
    """Read tif/png/jpg robustly."""
    path = Path(path)
    ext = path.suffix.lower()
    if ext in (".tif", ".tiff", ".ome.tif", ".ome.tiff"):
        return np.asarray(tiff.imread(str(path)))
    return np.asarray(iio.imread(str(path)))


def memmap_if_tiff(path: str | Path) -> np.ndarray:
    """Memory-map TIFF when possible (fast crop without full load)."""
    path = Path(path)
    ext = path.suffix.lower()
    if ext in (".tif", ".tiff", ".ome.tif", ".ome.tiff"):
        return tiff.memmap(str(path))
    return read_image_any(path)


def coerce_hw_c(img: np.ndarray) -> np.ndarray:
    """
    Ensure multi-channel arrays are (H,W,C) when possible.
    Supports:
      - (H,W)
      - (H,W,C)
      - (C,H,W) for small C
      - >3 dims: takes first plane then tries again
    """
    arr = np.asarray(img)

    if arr.ndim > 3:
        arr = arr.reshape(-1, *arr.shape[-2:])[0]

    if arr.ndim == 2:
        return arr

    if arr.ndim != 3:
        raise ValueError(f"Unsupported ndim={arr.ndim}")

    C, H, W = arr.shape
    if C <= 8 and H > 16 and W > 16 and (C < H and C < W):
        return np.transpose(arr, (1, 2, 0))

    return arr


def ensure_rgb_u8(img: np.ndarray) -> np.ndarray:
    """Convert grayscale or RGB-like to uint8 RGB for display."""
    arr = np.asarray(img)

    # grayscale
    if arr.ndim == 2:
        f = arr.astype(np.float32)
        lo, hi = np.percentile(f, (1, 99.8))
        f = np.clip((f - lo) / (hi - lo + 1e-8), 0, 1)
        u8 = (255 * f).astype(np.uint8)
        return np.stack([u8, u8, u8], axis=-1)

    # already RGB/RGBA-like
    if arr.ndim == 3 and arr.shape[-1] in (3, 4):
        if arr.dtype != np.uint8:
            f = arr.astype(np.float32)
            lo, hi = np.percentile(f, (1, 99.8))
            f = np.clip((f - lo) / (hi - lo + 1e-8), 0, 1)
            arr = (255 * f).astype(np.uint8)
        return arr[..., :3]

    # multi-channel: display channel 0
    if arr.ndim == 3:
        return ensure_rgb_u8(arr[..., 0])

    raise ValueError(f"Unsupported image ndim={arr.ndim}")


def save_mask_tiff(path_out: str | Path, mask: np.ndarray) -> None:
    path_out = Path(path_out)
    path_out.parent.mkdir(parents=True, exist_ok=True)
    tiff.imwrite(str(path_out), mask.astype(np.int32), compression="zlib")


def save_rgb_png(path_out: str | Path, rgb_u8: np.ndarray) -> None:
    path_out = Path(path_out)
    path_out.parent.mkdir(parents=True, exist_ok=True)
    iio.imwrite(str(path_out), rgb_u8)


# =========================================================
# Instance boundaries + outlines (instance-accurate)
# =========================================================
def instance_boundaries(mask: np.ndarray) -> np.ndarray:
    """
    Pixel-accurate boundaries for an instance mask (0 background, 1..N instances).
    Includes label-label and fg-bg boundaries.
    Returns (H,W) bool.
    """
    m = mask.astype(np.int32, copy=False)
    b = np.zeros_like(m, dtype=bool)

    b[1:, :] |= (m[1:, :] != m[:-1, :])
    b[:, 1:] |= (m[:, 1:] != m[:, :-1])

    return b


def outline_fast_instances(
    img_rgb_u8: np.ndarray,
    mask: np.ndarray,
    *,
    color: tuple[int, int, int] = (255, 0, 0),
    w: int = 1,
) -> np.ndarray:
    """Overlay instance boundaries on RGB uint8 image."""
    if img_rgb_u8.dtype != np.uint8:
        raise ValueError("img_rgb_u8 must be uint8")
    if img_rgb_u8.ndim != 3 or img_rgb_u8.shape[-1] != 3:
        raise ValueError("img_rgb_u8 must be (H,W,3)")

    b = instance_boundaries(mask)
    out = img_rgb_u8.copy()
    out[b] = np.array(color, dtype=np.uint8)

    if int(w) > 1:
        b8 = (b.astype(np.uint8) * 255)
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * w + 1, 2 * w + 1))
        b8 = cv2.dilate(b8, k)
        out[b8 > 0] = np.array(color, dtype=np.uint8)

    return out


def colorize_labels(mask: np.ndarray, seed: int = 0) -> np.ndarray:
    """Convert instance labels -> random RGB colors (deterministic with seed)."""
    m = mask.astype(np.int32, copy=False)
    n = int(m.max())
    if n <= 0:
        return np.zeros((m.shape[0], m.shape[1], 3), dtype=np.uint8)

    rng = np.random.default_rng(seed)
    lut = rng.random((n + 1, 3))
    lut[0] = 0.0
    return (255 * lut[m]).astype(np.uint8)


# =========================================================
# Biopsy ID mapping
# =========================================================
def extract_biopsy_id(filename: str, mode: str = "prefix_before__") -> str:
    stem = Path(filename).stem
    if mode == "prefix_before__":
        return stem.split("__")[0]
    if mode == "prefix_before_first_underscore":
        return stem.split("_")[0]
    return stem


def resolve_display_biopsy_folder(display_root: Path, biopsy_id: str) -> Optional[str]:
    dirs = set(list_subdirs(display_root))
    if biopsy_id in dirs:
        return biopsy_id
    matches = [d for d in dirs if d.startswith(biopsy_id) or biopsy_id.startswith(d)]
    if not matches:
        return None
    return sorted(matches, key=len)[0]


# =========================================================
# Cellpose cyto3 (v3/v4 compatible)
# =========================================================
def get_cellpose_cyto3_model(use_gpu: bool):
    try:
        return models.CellposeModel(gpu=bool(use_gpu), pretrained_model="cyto3")  # v4+
    except Exception:
        return models.Cellpose(gpu=bool(use_gpu), model_type="cyto3")            # v3


def cellpose_eval_robust(model, img_hw2: np.ndarray, *, diameter, cellprob_th: float, flow_th: float):
    out = model.eval(
        img_hw2,
        channels=[1, 2],  # membrane, nucleus
        diameter=diameter,
        cellprob_threshold=float(cellprob_th),
        flow_threshold=float(flow_th),
    )
    if not isinstance(out, tuple):
        raise RuntimeError(f"Unexpected model.eval output type: {type(out)}")
    if len(out) == 4:
        return out  # masks, flows, styles, diams
    if len(out) == 3:
        masks, flows, diams = out
        styles = None
        return masks, flows, styles, diams
    raise RuntimeError(f"Unexpected model.eval output length: {len(out)}")


# =========================================================
# Crop helpers
# =========================================================
def compute_crop_coords(H: int, W: int, crop_size: int, mode: str, rng: np.random.Generator) -> Tuple[int, int, int, int]:
    cs = int(max(64, crop_size))
    ph = min(cs, H)
    pw = min(cs, W)

    m = mode.lower().strip()
    if m == "center":
        y0 = max(0, (H - ph) // 2)
        x0 = max(0, (W - pw) // 2)
    elif m == "random":
        y0 = int(rng.integers(0, max(1, H - ph + 1)))
        x0 = int(rng.integers(0, max(1, W - pw + 1)))
    else:
        y0, x0 = 0, 0

    return y0, x0, ph, pw


def crop_hw(arr: np.ndarray, y0: int, x0: int, ph: int, pw: int) -> np.ndarray:
    if arr.ndim == 2:
        return np.asarray(arr[y0:y0 + ph, x0:x0 + pw])
    if arr.ndim == 3:
        return np.asarray(arr[y0:y0 + ph, x0:x0 + pw, :])
    raise ValueError(f"Unsupported ndim={arr.ndim} for crop")


# =========================================================
# Main UI
# =========================================================
def run_interactive_cellpose_cyto3_crop_ui(
    *,
    seg_root: str,
    display_root: str,
    mask_out_root: str,
    color_out_root: str,
    biopsy_id_mode: str = "prefix_before__",
    default_marker_contains: str = "dna",
    default_mem_ch: int = 0,
    default_nuc_ch: int = 1,
    use_gpu: bool = True,
    overwrite: bool = False,
):
    seg_root = Path(seg_root)
    display_root = Path(display_root)
    mask_out_root = Path(mask_out_root)
    color_out_root = Path(color_out_root)

    mask_out_root.mkdir(parents=True, exist_ok=True)
    color_out_root.mkdir(parents=True, exist_ok=True)

    seg_files = list_files_flat(seg_root)
    if not seg_files:
        raise ValueError(f"No images found in seg_root: {seg_root}")

    rng = np.random.default_rng(0)

    header = widgets.HTML("<h3 style='margin:0 0 8px 0;'>Cellpose CYTO3 — crop preview</h3>")

    seg_dd = widgets.Dropdown(
        options=[p.name for p in seg_files],
        value=seg_files[0].name,
        description="Seg img:",
        layout=widgets.Layout(width="720px"),
        style={"description_width": "90px"},
    )

    biopsy_info = widgets.HTML("<b>Biopsy:</b> -")

    marker_dd = widgets.Dropdown(
        options=[],
        value=None,
        description="Left img:",
        layout=widgets.Layout(width="720px"),
        style={"description_width": "90px"},
    )

    crop_mode_w = widgets.Dropdown(
        options=[("Center", "center"), ("Random", "random"), ("Top-left", "topleft")],
        value="center",
        description="Crop:",
        layout=widgets.Layout(width="360px"),
        style={"description_width": "90px"},
    )
    crop_size_w = widgets.IntText(
        value=300,
        description="crop_px:",
        layout=widgets.Layout(width="360px"),
        style={"description_width": "90px"},
    )

    mem_ch_w = widgets.IntText(value=int(default_mem_ch), description="mem_ch:", layout=widgets.Layout(width="360px"),
                               style={"description_width": "90px"})
    nuc_ch_w = widgets.IntText(value=int(default_nuc_ch), description="nuc_ch:", layout=widgets.Layout(width="360px"),
                               style={"description_width": "90px"})

    diameter_w = widgets.FloatText(value=0.0, description="diameter:", layout=widgets.Layout(width="360px"),
                                   style={"description_width": "90px"})
    cellprob_w = widgets.FloatText(value=0.0, description="cellprob:", layout=widgets.Layout(width="360px"),
                                   style={"description_width": "90px"})
    flow_w = widgets.FloatText(value=0.4, description="flow_th:", layout=widgets.Layout(width="360px"),
                               style={"description_width": "90px"})
    min_size_w = widgets.IntText(value=0, description="min_size:", layout=widgets.Layout(width="360px"),
                                 style={"description_width": "90px"})

    outline_w = widgets.IntSlider(value=1, min=1, max=5, step=1, description="outline_w:",
                                  layout=widgets.Layout(width="720px"),
                                  style={"description_width": "90px"})

    btn_preview = widgets.Button(description="Preview crop", button_style="primary", icon="eye")
    btn_batch = widgets.Button(description="Segment ALL + save", button_style="warning", icon="cogs")

    status = widgets.HTML("")
    out = widgets.Output()
    log = widgets.Output()

    state: Dict[str, Any] = {"biopsy_id": None, "display_folder": None}

    def refresh_marker_list():
        seg_filename = seg_dd.value
        biopsy_id = extract_biopsy_id(seg_filename, mode=biopsy_id_mode)
        disp = resolve_display_biopsy_folder(display_root, biopsy_id)

        state["biopsy_id"] = biopsy_id
        state["display_folder"] = disp

        if disp is None:
            biopsy_info.value = f"<b>Biopsy:</b> {biopsy_id} — <span style='color:#b00020'>no matching folder in display_root</span>"
            marker_dd.options = []
            marker_dd.value = None
            return

        biopsy_info.value = f"<b>Biopsy:</b> {biopsy_id} (display folder: {disp})"
        markers = list_marker_files(display_root / disp)
        marker_dd.options = markers

        if not markers:
            marker_dd.value = None
            return

        hint = default_marker_contains.lower().strip()
        pick = None
        for m in markers:
            if hint in Path(m).stem.lower():
                pick = m
                break
        marker_dd.value = pick or markers[0]

    def run_preview(_=None):
        with out:
            clear_output(wait=True)

            seg_filename = seg_dd.value
            seg_path = seg_root / seg_filename
            if not seg_path.exists():
                status.value = f"<b style='color:#b00020'>Missing:</b> {seg_path}"
                return

            biopsy_id = state["biopsy_id"]
            disp = state["display_folder"]
            left_marker = marker_dd.value

            seg_img = memmap_if_tiff(seg_path)
            seg_img = coerce_hw_c(seg_img)
            if seg_img.ndim != 3:
                status.value = "<b style='color:#b00020'>Error:</b> seg input must be (H,W,C>=2) for cyto3."
                return

            H, W, C = seg_img.shape
            mem_ch = int(mem_ch_w.value)
            nuc_ch = int(nuc_ch_w.value)
            if not (0 <= mem_ch < C and 0 <= nuc_ch < C):
                status.value = f"<b style='color:#b00020'>Error:</b> channel indices out of range (C={C})."
                return

            y0, x0, ph, pw = compute_crop_coords(H, W, int(crop_size_w.value), str(crop_mode_w.value), rng)

            mem_crop = np.asarray(seg_img[y0:y0 + ph, x0:x0 + pw, mem_ch]).astype(np.float32)
            nuc_crop = np.asarray(seg_img[y0:y0 + ph, x0:x0 + pw, nuc_ch]).astype(np.float32)
            cp_img = np.stack([mem_crop, nuc_crop], axis=-1)

            model = get_cellpose_cyto3_model(use_gpu)
            diameter = float(diameter_w.value)
            diameter = None if diameter <= 0 else diameter

            masks, _, _, _ = cellpose_eval_robust(
                model,
                cp_img,
                diameter=diameter,
                cellprob_th=float(cellprob_w.value),
                flow_th=float(flow_w.value),
            )

            ms = int(min_size_w.value)
            if ms > 0:
                masks = remove_small_objects(masks, min_size=ms).astype(np.int32)

            # LEFT: marker crop
            if disp is not None and left_marker is not None:
                left_path = display_root / disp / left_marker
                if left_path.exists():
                    left_img = memmap_if_tiff(left_path)
                    left_img = coerce_hw_c(left_img)
                    left_crop = crop_hw(left_img, y0, x0, ph, pw)
                else:
                    left_crop = mem_crop
            else:
                left_crop = mem_crop

            left_rgb = ensure_rgb_u8(left_crop)
            left_ov = outline_fast_instances(left_rgb, masks, w=int(outline_w.value))

            right_rgb = colorize_labels(masks, seed=abs(hash(seg_filename)) % (2**32))

            fig, axes = plt.subplots(1, 2, figsize=(14, 7))
            axes[0].imshow(left_ov)
            axes[0].set_title(f"LEFT: marker + outlines (crop)\n{biopsy_id} | {left_marker} | y={y0}:{y0+ph}, x={x0}:{x0+pw}")
            axes[0].axis("off")

            axes[1].imshow(right_rgb)
            axes[1].set_title("RIGHT: colored instance mask (crop)")
            axes[1].axis("off")

            plt.tight_layout()
            plt.show()

            status.value = f"<b>Preview done.</b> objects={int(masks.max())} | crop={ph}x{pw}"

    def run_batch_all(_=None):
        with log:
            clear_output(wait=True)

            model = get_cellpose_cyto3_model(use_gpu)

            processed = 0
            skipped = 0
            failed = 0

            for seg_path in tqdm(seg_files, desc="Cellpose cyto3 (ALL images)", unit="img", ncols=110):
                try:
                    out_mask_path = mask_out_root / f"{seg_path.stem}.tif"
                    out_col_path = color_out_root / f"{seg_path.stem}.png"

                    if (not overwrite) and out_mask_path.exists() and out_col_path.exists():
                        skipped += 1
                        continue

                    seg_img = read_image_any(seg_path)
                    seg_img = coerce_hw_c(seg_img)
                    if seg_img.ndim != 3:
                        raise ValueError("seg input must be (H,W,C>=2) for cyto3 batch")

                    H, W, C = seg_img.shape
                    mem_ch = int(mem_ch_w.value)
                    nuc_ch = int(nuc_ch_w.value)
                    if not (0 <= mem_ch < C and 0 <= nuc_ch < C):
                        raise ValueError(f"channel indices out of range (C={C})")

                    mem = seg_img[..., mem_ch].astype(np.float32)
                    nuc = seg_img[..., nuc_ch].astype(np.float32)
                    cp_img = np.stack([mem, nuc], axis=-1)

                    diameter = float(diameter_w.value)
                    diameter = None if diameter <= 0 else diameter

                    masks, _, _, _ = cellpose_eval_robust(
                        model,
                        cp_img,
                        diameter=diameter,
                        cellprob_th=float(cellprob_w.value),
                        flow_th=float(flow_w.value),
                    )

                    ms = int(min_size_w.value)
                    if ms > 0:
                        masks = remove_small_objects(masks, min_size=ms).astype(np.int32)

                    save_mask_tiff(out_mask_path, masks)
                    save_rgb_png(out_col_path, colorize_labels(masks, seed=abs(hash(seg_path.name)) % (2**32)))

                    processed += 1

                except Exception as e:
                    failed += 1
                    print(f"[WARN] Failed: {seg_path.name} -> {e}")

            print("\n=== Batch summary ===")
            print(f"Processed: {processed}")
            print(f"Skipped:   {skipped}")
            print(f"Failed:    {failed}")
            print(f"Masks:     {mask_out_root}")
            print(f"Colored:   {color_out_root}")

    # wiring
    def on_seg_change(*_):
        refresh_marker_list()
        run_preview()

    seg_dd.observe(on_seg_change, names="value")
    marker_dd.observe(lambda *_: run_preview(), names="value")
    crop_mode_w.observe(lambda *_: run_preview(), names="value")
    crop_size_w.observe(lambda *_: run_preview(), names="value")
    mem_ch_w.observe(lambda *_: run_preview(), names="value")
    nuc_ch_w.observe(lambda *_: run_preview(), names="value")
    diameter_w.observe(lambda *_: run_preview(), names="value")
    cellprob_w.observe(lambda *_: run_preview(), names="value")
    flow_w.observe(lambda *_: run_preview(), names="value")
    min_size_w.observe(lambda *_: run_preview(), names="value")
    outline_w.observe(lambda *_: run_preview(), names="value")

    btn_preview.on_click(run_preview)
    btn_batch.on_click(run_batch_all)

    controls = widgets.VBox(
        [
            header,
            seg_dd,
            biopsy_info,
            marker_dd,
            widgets.HTML("<b>Preview crop</b>"),
            widgets.HBox([crop_mode_w, crop_size_w]),
            widgets.HTML("<b>Channels in seg input (H,W,C)</b>"),
            widgets.HBox([mem_ch_w, nuc_ch_w]),
            widgets.HTML("<b>Cellpose parameters</b>"),
            widgets.HBox([diameter_w, min_size_w]),
            widgets.HBox([cellprob_w, flow_w]),
            outline_w,
            widgets.HBox([btn_preview, btn_batch]),
            status,
            log,
        ],
        layout=widgets.Layout(width="760px"),
    )
    preview = widgets.VBox([widgets.HTML("<b>Preview</b>"), out], layout=widgets.Layout(width="820px"))
    display(widgets.HBox([controls, preview], layout=widgets.Layout(gap="18px")))

    refresh_marker_list()
    run_preview()




In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


def export_colored_instance_masks_flat(
    *,
    path_mask: str,
    path_mask_color: str,
    seed: int = 0,
    use_all_matplotlib_colors: bool = True,
    exts_in: Tuple[str, ...] = (".tif", ".tiff", ".png"),
    progress: bool = True,
    progress_ncols: int = 110,
    background_color: Tuple[float, float, float] = (0.0, 0.0, 0.0),
) -> Dict[str, int]:
    """
    Colorize an *instance-labeled* mask folder (flat layout) WITHOUT re-labeling.

    Assumes:
      - mask pixels are integer labels
      - 0 = background
      - each object has its own label value (1..N, not necessarily contiguous)

    Output:
      - saves PNG RGB images with same stem into `path_mask_color`.

    Returns: processed / failed / output_root
    """
    path_mask = Path(path_mask)
    path_mask_color = Path(path_mask_color)
    path_mask_color.mkdir(parents=True, exist_ok=True)

    # Palette (RGB in [0,1])
    if use_all_matplotlib_colors:
        palette = [matplotlib.colors.to_rgb(hx) for hx in matplotlib.colors.cnames.values()]
    else:
        palette = [matplotlib.colors.to_rgb(hx) for hx in [
            "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
            "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
        ]]
    if not palette:
        raise RuntimeError("No colors available for palette.")
    palette = np.asarray(palette, dtype=np.float32)

    rng = np.random.default_rng(seed)

    # List mask files (flat)
    exts_l = tuple(e.lower() for e in exts_in)
    mask_files = sorted([p for p in path_mask.iterdir() if p.is_file() and p.suffix.lower() in exts_l])
    if not mask_files:
        raise RuntimeError(f"No mask files found in: {path_mask}")

    processed = 0
    failed = 0
    pbar = tqdm(total=len(mask_files), desc="Colorizing instance masks", unit="mask", ncols=progress_ncols) if progress else None

    for mask_path in mask_files:
        if pbar is not None:
            pbar.set_postfix_str(mask_path.name, refresh=False)

        try:
            mask = np.array(Image.open(mask_path))

            # Ensure integer labels
            if mask.dtype.kind not in ("u", "i"):
                # If mask was saved as float, cast safely (but ideally masks should be int)
                mask = mask.astype(np.int32)

            labels = np.unique(mask)
            labels = labels[labels != 0]  # exclude background

            H, W = mask.shape[:2]
            canvas = np.zeros((H, W, 3), dtype=np.float32)
            canvas[...] = background_color

            if labels.size > 0:
                # Pick a random color for each label present in this image (deterministic seed + RNG state)
                # We'll create a dict-like mapping using two arrays and vectorized indexing.
                # 1) draw colors for the present labels
                chosen_idx = rng.integers(0, len(palette), size=labels.size)
                chosen_colors = palette[chosen_idx]  # (n_labels, 3)

                # 2) map pixels: use an index image via searchsorted on sorted labels
                labels_sorted = np.sort(labels)
                colors_sorted = chosen_colors[np.argsort(labels)]  # align with labels_sorted

                # create an index for all pixels whose label != 0
                m = mask
                fg = (m != 0)
                idx = np.searchsorted(labels_sorted, m[fg])
                canvas[fg] = colors_sorted[idx]

            out_path = path_mask_color / f"{mask_path.stem}.png"
            plt.imsave(str(out_path), canvas)  # float in [0,1]
            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {mask_path.name}: {e}")
            else:
                print(f"[WARN] Failed {mask_path.name}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    print(f"✅ Folder with color masks available there: {path_mask_color}")

    return {"processed": processed, "failed": failed, "output_root": str(path_mask_color)}

#### Segmentation

##### Segmentation of small biopsies (<1100pixel)

In [ ]:
processing=input("Enter the image processing you want to use: ")

Enter the image processing you want to use: 2


In [ ]:
dna_marker=input("Enter the dna marker: ")

Enter the dna marker: DNA


In [ ]:
path_img_processing_biopsies=path+"images/img_processing_"+processing+"/Biopsies/"

In [ ]:
run_interactive_cellpose_cyto3_crop_ui(
    seg_root=path_img_segmentation_mesmer_cellpose,
    display_root=path_img_processing_biopsies,
    mask_out_root=path_mask_cellposev3,
    color_out_root=path_mask_color_cellposev3,
    biopsy_id_mode="prefix_before__",
    default_marker_contains=dna_marker,
    default_mem_ch=1,
    default_nuc_ch=0,
    use_gpu=True,
    overwrite=False,
)

#### 📊 Colored masks

In [ ]:
report = export_colored_instance_masks_flat(
     path_mask=path_mask_cellposev3,
     path_mask_color=path_mask_color_cellposev3,
     seed=0,
)
print(report)

Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/cellposev3
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/cellposev3'}


##### Segmentation of big biopsies (>2000 pixel)
⚠️ USER INPUT REQUIRED

In [ ]:
TILE=int(input("Enter the size of the tiles (1024 or 512): "))
OVERLAP=int(input("Enter the size of the overlap (256): "))
resolution=float(input("Enter the resolution of the images (0.5,1.5): "))

KeyboardInterrupt: Interrupted by user

In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import tifffile as tiff
import cv2

# ======================================================
# Paramètres tiling
# ======================================================
IOU_MERGE = 0.2      # seuil de fusion entre objets sur overlap
NITER = 2000         # votre paramètre cellpose
CHANNELS = [2, 1]    # votre paramètre cellpose

# ======================================================
# Utilitaires tiles
# ======================================================
def iter_tiles(H, W, tile, overlap):
    step = tile - overlap
    if step <= 0:
        raise ValueError("OVERLAP doit être < TILE")
    ys = list(range(0, max(H - tile, 0) + 1, step)) or [0]
    xs = list(range(0, max(W - tile, 0) + 1, step)) or [0]
    if ys[-1] != max(H - tile, 0):
        ys.append(max(H - tile, 0))
    if xs[-1] != max(W - tile, 0):
        xs.append(max(W - tile, 0))
    for y0 in ys:
        y1 = min(y0 + tile, H)
        for x0 in xs:
            x1 = min(x0 + tile, W)
            yield y0, y1, x0, x1

def central_region(y0, y1, x0, x1, H, W, overlap):
    half = overlap // 2
    cy0 = y0 + (0 if y0 == 0 else half)
    cx0 = x0 + (0 if x0 == 0 else half)
    cy1 = y1 - (0 if y1 == H else half)
    cx1 = x1 - (0 if x0 == W else half)
    cy1 = max(cy1, cy0)
    cx1 = max(cx1, cx0)
    return cy0, cy1, cx0, cx1

# ======================================================
# Union-Find (fusion labels)
# ======================================================
class DSU:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        p = self.parent.get(x, x)
        if p != x:
            self.parent[x] = self.find(p)
        else:
            self.parent[x] = x
        return self.parent[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra

# ======================================================
# Fusion sur overlap (IoU)
# ======================================================
def merge_overlap(global_mask, tile_mask, y0, x0, overlap):
    """
    Fusionne les instances entre global_mask et tile_mask sur la zone overlap,
    en reliant les labels qui se chevauchent avec IoU >= IOU_MERGE.
    """
    dsu = DSU()

    Ht, Wt = tile_mask.shape
    gy0, gx0 = y0, x0
    gy1, gx1 = y0 + Ht, x0 + Wt

    # zone overlap = intersection entre tile et global déjà rempli
    g_patch = global_mask[gy0:gy1, gx0:gx1]
    t_patch = tile_mask

    # pixels où les deux ont un label
    inter = (g_patch > 0) & (t_patch > 0)
    if not np.any(inter):
        return global_mask, None  # rien à fusionner

    g_ids = g_patch[inter].astype(np.int64)
    t_ids = t_patch[inter].astype(np.int64)

    # co-occurrence
    pairs = np.stack([g_ids, t_ids], axis=1)
    pairs = pairs[np.lexsort((pairs[:,1], pairs[:,0]))]

    # compter intersections par paire
    uniq, counts = np.unique(pairs, axis=0, return_counts=True)

    # aire par label dans overlap
    g_area = np.bincount(g_patch.ravel().astype(np.int64))
    t_area = np.bincount(t_patch.ravel().astype(np.int64))

    # pour chaque paire, calc iou
    for (gid, tid), inter_cnt in zip(uniq, counts):
        if gid == 0 or tid == 0:
            continue
        union = g_area[gid] + t_area[tid] - inter_cnt
        if union <= 0:
            continue
        iou = inter_cnt / union
        if iou >= IOU_MERGE:
            dsu.union(int(gid), int(tid))

    return global_mask, dsu

def relabel_with_dsu(tile_mask, dsu, tile_offset):
    """
    Applique la fusion:
    - labels tile -> labels globaux si union trouvée
    - sinon, nouveaux labels globaux (offset)
    """
    if dsu is None:
        # pas de fusion: offset classique
        out = tile_mask.copy()
        out[out > 0] += tile_offset
        return out, tile_offset + int(tile_mask.max())

    out = tile_mask.copy().astype(np.int64)
    max_tile = int(out.max())
    mapping = {}

    # on prépare un mapping tile_label -> global_label
    # dsu relie gid (global) avec tid (tile) mais tid est encore local: on map selon find()
    # On garde l'ID global le plus petit comme représentant
    # Dans dsu, les éléments sont gid ou tid; on a union(gid, tid)
    # Donc find(tid) peut renvoyer un gid.
    for tid in range(1, max_tile + 1):
        rep = dsu.find(tid)
        if rep != tid and rep > 0:
            mapping[tid] = rep  # fusion vers global id
        else:
            # nouveau label
            tile_offset += 1
            mapping[tid] = tile_offset

    # appliquer mapping
    for tid, gid in mapping.items():
        out[out == tid] = gid

    return out.astype(np.int64), tile_offset

# ======================================================
# Segmentation tiled Cellpose + stitching
# ======================================================
def segment_large_image_cellpose(img, model, tile=TILE, overlap=OVERLAP):
    """
    img: array HxWxC ou HxW
    Retourne masque global (HxW int64)
    """
    if img.ndim == 2:
        img_in = img
    else:
        img_in = img

    H, W = img_in.shape[:2]
    full_mask = np.zeros((H, W), dtype=np.int64)

    tile_offset = 0

    # on garde une zone "remplie" (utile implicitement)
    tiles = list(iter_tiles(H, W, tile, overlap))

    for (y0, y1, x0, x1) in tqdm(tiles, desc="Tiles", unit="tile", leave=False):
        tile_img = img_in[y0:y1, x0:x1]

        # pad pour avoir TILE x TILE
        pad_h = tile - (y1 - y0)
        pad_w = tile - (x1 - x0)
        if pad_h > 0 or pad_w > 0:
            if tile_img.ndim == 2:
                tile_pad = np.pad(tile_img, ((0, pad_h), (0, pad_w)), mode="constant")
            else:
                tile_pad = np.pad(tile_img, ((0, pad_h), (0, pad_w), (0, 0)), mode="constant")
        else:
            tile_pad = tile_img

        # Cellpose
        masks_pred, flows, styles = model.eval(tile_pad, diameter=None, channels=CHANNELS, niter=NITER)

        # enlever padding
        tile_mask = masks_pred[:(y1-y0), :(x1-x0)].astype(np.int64)

        # zone centrale (anti seams)
        cy0, cy1, cx0, cx1 = central_region(y0, y1, x0, x1, H, W, overlap)
        ty0, ty1 = cy0 - y0, cy1 - y0
        tx0, tx1 = cx0 - x0, cx1 - x0
        core_mask = tile_mask[ty0:ty1, tx0:tx1]

        if core_mask.max() == 0:
            continue

        # fusion overlap: on fusionne avec full_mask dans la zone où on va écrire
        gy0, gx0 = cy0, cx0
        gy1, gx1 = cy1, cx1

        # fusion sur patch correspondant
        patch_global = full_mask[gy0:gy1, gx0:gx1]
        # construire un tile_mask "local core" dans le même cadre
        tile_core = core_mask

        # merge overlap sur la zone patch
        # On fusionne avec les labels déjà présents dans patch_global
        full_mask, dsu = merge_overlap(full_mask, tile_core, gy0, gx0, overlap=overlap)

        # relabel tile core avec fusions + nouveaux IDs
        relabeled_core, tile_offset = relabel_with_dsu(tile_core, dsu, tile_offset)

        # écrire dans full_mask (priorité au nouveau, mais on ne doit pas écraser des labels existants sauf background)
        # règle: on écrit là où full_mask == 0
        write_area = (full_mask[gy0:gy1, gx0:gx1] == 0) & (relabeled_core > 0)
        full_mask[gy0:gy1, gx0:gx1][write_area] = relabeled_core[write_area]

    return full_mask

# ======================================================
# Batch processing folder
# ======================================================
list_img = sorted(os.listdir(path_img_segmentation))

for img_file in tqdm(list_img, desc="Images", unit="img"):
    if not img_file.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
        continue

    img_path = os.path.join(path_img_segmentation, img_file)
    out_path = os.path.join(path_mask_cellposev3, os.path.splitext(img_file)[0] + ".tif")
    os.makedirs(path_mask_cellposev3, exist_ok=True)

    # skip if already done
    if os.path.isfile(out_path):
        continue

    img = np.array(Image.open(img_path))

    mask_full = segment_large_image_cellpose(img, model, tile=TILE, overlap=OVERLAP)

    tiff.imwrite(out_path, mask_full.astype(np.uint32), compression="zlib")
    print(f"✅ {img_file}: {int(mask_full.max())} objects | saved: {out_path}")

print("✅ Masks available here:", path_mask_cellposev3)


### 📊 InstanSeg

In [ ]:
 path_mask_instanseg=path_segmentation_cells_mask+"instanseg/"
 path_img_segmentation_instanseg=path+"images/images_segmentation/instanseg/"
 if os.path.isdir(path_mask_instanseg)==False:
    os.mkdir(path_mask_instanseg)
    print("✅ Folder for InstanSeg mask created")
 path_mask_color_instanseg=path_segmentation_cells_mask_color+"instanseg/"
 if os.path.isdir(path_mask_color_instanseg)==False:
    os.mkdir(path_mask_color_instanseg)
    print("✅ Folder for color Instanseg masks created")

#### 🛠️ Installation of Instanseg

In [ ]:
# === Cellule 1 : installation de l'environnement InstanSeg ===

# 1) Installer micromamba (mini-conda)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!mkdir -p /root/micromamba/envs

# 2) Créer un environnement Python 3.11 pour InstanSeg
!./bin/micromamba create -y -p /root/micromamba/envs/instanseg-env python=3.11

# 3) Installer InstanSeg + dépendances dans cet environnement
!./bin/micromamba run -p /root/micromamba/envs/instanseg-env pip install -q \
    "instanseg-torch[full]" "matplotlib<3.9"


bin/micromamba
[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch     1%[+] 0.2s
conda-forge/linux-64   6%
conda-forge/noarch    16%[+] 0.3s
conda-forge/linux-64  16%
conda-forge/noarch    38%[+] 0.4s
conda-forge/linux-64  27%
conda-forge/noarch    61%[+] 0.5s
conda-forge/linux-64  38%
conda-forge/noarch    72%[+] 0.6s
conda-forge/linux-64  44%
conda-forge/noarch    96%conda-forge/noarch                                
[+] 0.7s
conda-forge/linux-64  47%[+] 0.8s
conda-forge/linux-64  67%[+] 0.9s
conda-forge/linux-64  86%[+] 1.0s
conda-forge/linux-64  95%conda-forge/linux-64                              


Transaction

  Prefix: /root/micromamba/envs/instanseg-env

  Updating specs:

   - python=3.11


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           

In [ ]:
import os, subprocess, textwrap

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

code = """
import matplotlib
matplotlib.use('Agg')

import sys
print("Python utilisé :", sys.version)

from instanseg import InstanSeg
print("InstanSeg importé OK :", InstanSeg)
"""

subprocess.run(
    ["/root/micromamba/envs/instanseg-env/bin/python", "-c", code],
    check=True,
    env=env,
)


CompletedProcess(args=['/root/micromamba/envs/instanseg-env/bin/python', '-c', '\nimport matplotlib\nmatplotlib.use(\'Agg\')\n\nimport sys\nprint("Python utilisé :", sys.version)\n\nfrom instanseg import InstanSeg\nprint("InstanSeg importé OK :", InstanSeg)\n'], returncode=0)

In [ ]:
!./bin/micromamba run -p /root/micromamba/envs/instanseg-env pip install requests


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


def export_colored_instance_masks_flat(
    *,
    path_mask: str,
    path_mask_color: str,
    seed: int = 0,
    use_all_matplotlib_colors: bool = True,
    exts_in: Tuple[str, ...] = (".tif", ".tiff", ".png"),
    progress: bool = True,
    progress_ncols: int = 110,
    background_color: Tuple[float, float, float] = (0.0, 0.0, 0.0),
) -> Dict[str, int]:
    """
    Colorize an *instance-labeled* mask folder (flat layout) WITHOUT re-labeling.

    Assumes:
      - mask pixels are integer labels
      - 0 = background
      - each object has its own label value (1..N, not necessarily contiguous)

    Output:
      - saves PNG RGB images with same stem into `path_mask_color`.

    Returns: processed / failed / output_root
    """
    path_mask = Path(path_mask)
    path_mask_color = Path(path_mask_color)
    path_mask_color.mkdir(parents=True, exist_ok=True)

    # Palette (RGB in [0,1])
    if use_all_matplotlib_colors:
        palette = [matplotlib.colors.to_rgb(hx) for hx in matplotlib.colors.cnames.values()]
    else:
        palette = [matplotlib.colors.to_rgb(hx) for hx in [
            "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
            "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
        ]]
    if not palette:
        raise RuntimeError("No colors available for palette.")
    palette = np.asarray(palette, dtype=np.float32)

    rng = np.random.default_rng(seed)

    # List mask files (flat)
    exts_l = tuple(e.lower() for e in exts_in)
    mask_files = sorted([p for p in path_mask.iterdir() if p.is_file() and p.suffix.lower() in exts_l])
    if not mask_files:
        raise RuntimeError(f"No mask files found in: {path_mask}")

    processed = 0
    failed = 0
    pbar = tqdm(total=len(mask_files), desc="Colorizing instance masks", unit="mask", ncols=progress_ncols) if progress else None

    for mask_path in mask_files:
        if pbar is not None:
            pbar.set_postfix_str(mask_path.name, refresh=False)

        try:
            mask = np.array(Image.open(mask_path))

            # Ensure integer labels
            if mask.dtype.kind not in ("u", "i"):
                # If mask was saved as float, cast safely (but ideally masks should be int)
                mask = mask.astype(np.int32)

            labels = np.unique(mask)
            labels = labels[labels != 0]  # exclude background

            H, W = mask.shape[:2]
            canvas = np.zeros((H, W, 3), dtype=np.float32)
            canvas[...] = background_color

            if labels.size > 0:
                # Pick a random color for each label present in this image (deterministic seed + RNG state)
                # We'll create a dict-like mapping using two arrays and vectorized indexing.
                # 1) draw colors for the present labels
                chosen_idx = rng.integers(0, len(palette), size=labels.size)
                chosen_colors = palette[chosen_idx]  # (n_labels, 3)

                # 2) map pixels: use an index image via searchsorted on sorted labels
                labels_sorted = np.sort(labels)
                colors_sorted = chosen_colors[np.argsort(labels)]  # align with labels_sorted

                # create an index for all pixels whose label != 0
                m = mask
                fg = (m != 0)
                idx = np.searchsorted(labels_sorted, m[fg])
                canvas[fg] = colors_sorted[idx]

            out_path = path_mask_color / f"{mask_path.stem}.png"
            plt.imsave(str(out_path), canvas)  # float in [0,1]
            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {mask_path.name}: {e}")
            else:
                print(f"[WARN] Failed {mask_path.name}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    print(f"✅ Folder with color masks available there: {path_mask_color}")

    return {"processed": processed, "failed": failed, "output_root": str(path_mask_color)}

#### Segmentation

##### Segmentation of small biopsies (<2000pixel)

In [ ]:
import os, subprocess, textwrap
from pathlib import Path

image_dir = path_img_segmentation_instanseg
output_dir = path_mask_instanseg

print("Input folder:", image_dir)
print("Output folder (masks):", output_dir)
print("-" * 70)

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

code = textwrap.dedent(f"""
import matplotlib
matplotlib.use('Agg')

from instanseg import InstanSeg
from pathlib import Path
import numpy as np
import tifffile as tiff

img_dir = Path({image_dir!r})
out_dir = Path({output_dir!r})

print("Input folder:", img_dir)
print("Output folder:", out_dir)

if not img_dir.exists():
    raise SystemExit(f"Path {{img_dir}} does not exist")

if not img_dir.is_dir():
    raise SystemExit(f"Path {{img_dir}} is not a folder")

out_dir.mkdir(parents=True, exist_ok=True)

print("Loading InstanSeg model (fluorescence_nuclei_and_cells)...")
model = InstanSeg(
    "fluorescence_nuclei_and_cells",
    image_reader="tiffslide",
    verbosity=1
)

images = sorted(list(img_dir.glob("*.tif")) + list(img_dir.glob("*.tiff")))

print("Number of images found:", len(images))
if not images:
    raise SystemExit("No .tif/.tiff images found in " + str(img_dir))

for img_path in images:
    print("\\nProcessing:", img_path.name)

    labeled_output = model.eval(
        image=str(img_path),
        save_output=False,
        save_overlay=False
    )

    mask = np.asarray(labeled_output)
    mask_path = out_dir / (img_path.stem + ".tif")

    tiff.imwrite(str(mask_path), mask)
    print("Saved mask:", mask_path)

print("Segmentation completed.")
""")

res = subprocess.run(
    ["/root/micromamba/envs/instanseg-env/bin/python", "-c", code],
    env=env,
    capture_output=True,
    text=True,
)

print("----- STDOUT -----")
print(res.stdout)
print("----- STDERR -----")
print(res.stderr)

if res.returncode != 0:
    raise RuntimeError(f"InstanSeg failed (code {res.returncode})")


Input folder: /content/gdrive/MyDrive/these/pipeline/rejection/images/images_segmentation/instanseg/
Output folder (masks): /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask/instanseg/
----------------------------------------------------------------------
----- STDOUT -----
Input folder: /content/gdrive/MyDrive/these/pipeline/rejection/images/images_segmentation/instanseg
Output folder: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask/instanseg
Loading InstanSeg model (fluorescence_nuclei_and_cells)...
Model fluorescence_nuclei_and_cells version 0.1.1 downloaded and extracted to /root/micromamba/envs/instanseg-env/lib/python3.11/site-packages/instanseg/utils/../bioimageio_models/
Requesting default device: cuda
Number of images found: 89

Processing: 17.30478 a.tif
BioImage does not support the image: '/content/gdrive/MyDrive/these/pipeline/rejection/images/images_segmentation/instanseg/17.30478 a.tif'. You may need to install an extra for

##### Segmentation of big biopsies (>2000 pixels)

In [ ]:
import os, subprocess, textwrap
from pathlib import Path

image_dir = path_img_segmentation_instanseg
output_dir = path_mask_instanseg

# --------- Tiling params ----------
TILE = 2048
OVERLAP = 512
IOU_MERGE = 0.25

print("Input folder:", image_dir)
print("Output folder (masks):", output_dir)
print("-" * 70)

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

code = textwrap.dedent(f"""
import matplotlib
matplotlib.use('Agg')

from instanseg import InstanSeg
from pathlib import Path
import numpy as np
import tifffile as tiff
from tqdm.auto import tqdm

# Lazy reader
from tiffslide import TiffSlide

img_dir = Path({str(image_dir)!r})
out_dir = Path({str(output_dir)!r})
out_dir.mkdir(parents=True, exist_ok=True)

TILE = {TILE}
OVERLAP = {OVERLAP}
IOU_MERGE = {IOU_MERGE}

SUPPORTED = (".tif", ".tiff")

def iter_tiles(H, W, tile, overlap):
    step = tile - overlap
    if step <= 0:
        raise ValueError("OVERLAP doit être < TILE")
    ys = list(range(0, max(H - tile, 0) + 1, step)) or [0]
    xs = list(range(0, max(W - tile, 0) + 1, step)) or [0]
    if ys[-1] != max(H - tile, 0):
        ys.append(max(H - tile, 0))
    if xs[-1] != max(W - tile, 0):
        xs.append(max(W - tile, 0))
    for y0 in ys:
        y1 = min(y0 + tile, H)
        for x0 in xs:
            x1 = min(x0 + tile, W)
            yield y0, y1, x0, x1

def central_region(y0, y1, x0, x1, H, W, overlap):
    half = overlap // 2
    cy0 = y0 + (0 if y0 == 0 else half)
    cx0 = x0 + (0 if x0 == 0 else half)
    cy1 = y1 - (0 if y1 == H else half)
    cx1 = x1 - (0 if x1 == W else half)
    cy1 = max(cy1, cy0)
    cx1 = max(cx1, cx0)
    return cy0, cy1, cx0, cx1

def compute_iou_mapping(global_patch, tile_patch):
    \"\"\"Map tile labels -> best global label based on IoU within the overlap patch.\"\"\"
    inter = (global_patch > 0) & (tile_patch > 0)
    if not np.any(inter):
        return {{}}

    g_ids = global_patch[inter].astype(np.int64)
    t_ids = tile_patch[inter].astype(np.int64)
    pairs = np.stack([g_ids, t_ids], axis=1)

    # Count intersections per (g,t)
    uniq, counts = np.unique(pairs, axis=0, return_counts=True)

    # Areas per id within patch
    g_area = np.bincount(global_patch.ravel().astype(np.int64))
    t_area = np.bincount(tile_patch.ravel().astype(np.int64))

    # For each tile id, keep best matching global id by IoU
    best = {{}}
    for (gid, tid), inter_cnt in zip(uniq, counts):
        if gid == 0 or tid == 0:
            continue
        union = g_area[gid] + t_area[tid] - inter_cnt
        if union <= 0:
            continue
        iou = inter_cnt / union
        if iou >= IOU_MERGE:
            prev = best.get(int(tid))
            if (prev is None) or (iou > prev[1]):
                best[int(tid)] = (int(gid), float(iou))

    return {{tid: gid for tid, (gid, _) in best.items()}}

def relabel_tile(tile_mask, mapping, next_id):
    \"\"\"Relabel tile labels: merge to mapped global labels or assign new global ids.\"\"\"
    out = tile_mask.astype(np.int64, copy=True)
    max_tile = int(out.max())
    if max_tile == 0:
        return out, next_id

    # Build final LUT for tile labels
    lut = np.zeros(max_tile + 1, dtype=np.int64)
    for tid in range(1, max_tile + 1):
        if tid in mapping:
            lut[tid] = mapping[tid]
        else:
            next_id += 1
            lut[tid] = next_id

    out = lut[out]
    return out, next_id

def ensure_2d_mask(pred):
    \"\"\"InstanSeg can output multi-channel labels (nuclei/cell). Keep 'cells' if present.\"\"\"
    arr = np.asarray(pred)
    if arr.ndim == 2:
        return arr
    # common cases: (H,W,2) or (2,H,W)
    if arr.ndim == 3:
        if arr.shape[-1] == 2:
            return arr[..., 1]  # often cells in channel 1
        if arr.shape[0] == 2 and arr.shape[1] > 16 and arr.shape[2] > 16:
            return arr[1, ...]
    # fallback: take first plane
    return arr[..., 0]

print("Loading InstanSeg model (fluorescence_nuclei_and_cells)...")
model = InstanSeg(
    "fluorescence_nuclei_and_cells",
    image_reader="tiffslide",
    verbosity=1
)

images = sorted(
    [p for p in img_dir.iterdir() if p.suffix.lower() in (".tif", ".tiff")]
)
print("Number of images found:", len(images))
if not images:
    raise SystemExit("No .tif/.tiff images found in " + str(img_dir))

for img_path in images:
    print(f"\\n=== Processing: {img_path.name} ===")

    # Open lazily
    slide = TiffSlide(str(img_path))
    W, H = slide.dimensions  # (width, height)

    full_mask = np.zeros((H, W), dtype=np.int64)
    next_id = 0

    tiles = list(iter_tiles(H, W, TILE, OVERLAP))
    for (y0, y1, x0, x1) in tqdm(tiles, desc=f"Tiles {img_path.name}", unit="tile", leave=False):

        # Read tile region (RGB or multi-channel depends on file; tiffslide returns RGBA)
        region = slide.read_region((x0, y0), 0, (x1 - x0, y1 - y0))
        tile_img = np.array(region)

        # Drop alpha if present
        if tile_img.ndim == 3 and tile_img.shape[2] == 4:
            tile_img = tile_img[..., :3]

        # Pad to TILE x TILE
        pad_h = TILE - (y1 - y0)
        pad_w = TILE - (x1 - x0)
        if pad_h > 0 or pad_w > 0:
            if tile_img.ndim == 2:
                tile_pad = np.pad(tile_img, ((0, pad_h), (0, pad_w)), mode="constant")
            else:
                tile_pad = np.pad(tile_img, ((0, pad_h), (0, pad_w), (0, 0)), mode="constant")
        else:
            tile_pad = tile_img

        # InstanSeg eval on numpy (preferred). If your version requires a path, we can add temp-file fallback.
        pred = model.eval(
            image=tile_pad,
            save_output=False,
            save_overlay=False
        )
        tile_mask = ensure_2d_mask(pred).astype(np.int64)

        # Remove padding
        tile_mask = tile_mask[:(y1 - y0), :(x1 - x0)]

        # Keep only central region to reduce seams
        cy0, cy1, cx0, cx1 = central_region(y0, y1, x0, x1, H, W, OVERLAP)
        ty0, ty1 = cy0 - y0, cy1 - y0
        tx0, tx1 = cx0 - x0, cx1 - x0
        core = tile_mask[ty0:ty1, tx0:tx1]
        if core.max() == 0:
            continue

        gy0, gy1 = cy0, cy1
        gx0, gx1 = cx0, cx1
        global_patch = full_mask[gy0:gy1, gx0:gx1]

        # Compute mapping on overlap area (where global already has labels)
        mapping = compute_iou_mapping(global_patch, core)

        # Relabel core: merge mapped ids and assign new ids for new objects
        core_relabeled, next_id = relabel_tile(core, mapping, next_id)

        # Write only where global is empty
        write = (global_patch == 0) & (core_relabeled > 0)
        global_patch[write] = core_relabeled[write]
        full_mask[gy0:gy1, gx0:gx1] = global_patch

    # Save final mask
    mask_path = out_dir / (img_path.stem + ".tif")
    tiff.imwrite(str(mask_path), full_mask.astype(np.uint32), compression="zlib")
    print("Saved stitched mask:", mask_path, "| max_id =", int(full_mask.max()))

print("✅ Segmentation completed (tiled + stitched).")
""")

res = subprocess.run(
    ["/root/micromamba/envs/instanseg-env/bin/python", "-c", code],
    env=env,
    capture_output=True,
    text=True,
)

print("----- STDOUT -----")
print(res.stdout)
print("----- STDERR -----")
print(res.stderr)

if res.returncode != 0:
    raise RuntimeError(f"InstanSeg failed (code {res.returncode})")


#### Colored masks

In [ ]:
report = export_colored_instance_masks_flat(
     path_mask=path_mask_instanseg,
     path_mask_color=path_mask_color_instanseg,
     seed=0,
)
print(report)

Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/instanseg
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/instanseg'}


### Stardist

#### Functions

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


def export_colored_instance_masks_flat(
    *,
    path_mask: str,
    path_mask_color: str,
    seed: int = 0,
    use_all_matplotlib_colors: bool = True,
    exts_in: Tuple[str, ...] = (".tif", ".tiff", ".png"),
    progress: bool = True,
    progress_ncols: int = 110,
    background_color: Tuple[float, float, float] = (0.0, 0.0, 0.0),
) -> Dict[str, int]:
    """
    Colorize an *instance-labeled* mask folder (flat layout) WITHOUT re-labeling.

    Assumes:
      - mask pixels are integer labels
      - 0 = background
      - each object has its own label value (1..N, not necessarily contiguous)

    Output:
      - saves PNG RGB images with same stem into `path_mask_color`.

    Returns: processed / failed / output_root
    """
    path_mask = Path(path_mask)
    path_mask_color = Path(path_mask_color)
    path_mask_color.mkdir(parents=True, exist_ok=True)

    # Palette (RGB in [0,1])
    if use_all_matplotlib_colors:
        palette = [matplotlib.colors.to_rgb(hx) for hx in matplotlib.colors.cnames.values()]
    else:
        palette = [matplotlib.colors.to_rgb(hx) for hx in [
            "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
            "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
        ]]
    if not palette:
        raise RuntimeError("No colors available for palette.")
    palette = np.asarray(palette, dtype=np.float32)

    rng = np.random.default_rng(seed)

    # List mask files (flat)
    exts_l = tuple(e.lower() for e in exts_in)
    mask_files = sorted([p for p in path_mask.iterdir() if p.is_file() and p.suffix.lower() in exts_l])
    if not mask_files:
        raise RuntimeError(f"No mask files found in: {path_mask}")

    processed = 0
    failed = 0
    pbar = tqdm(total=len(mask_files), desc="Colorizing instance masks", unit="mask", ncols=progress_ncols) if progress else None

    for mask_path in mask_files:
        if pbar is not None:
            pbar.set_postfix_str(mask_path.name, refresh=False)

        try:
            mask = np.array(Image.open(mask_path))

            # Ensure integer labels
            if mask.dtype.kind not in ("u", "i"):
                # If mask was saved as float, cast safely (but ideally masks should be int)
                mask = mask.astype(np.int32)

            labels = np.unique(mask)
            labels = labels[labels != 0]  # exclude background

            H, W = mask.shape[:2]
            canvas = np.zeros((H, W, 3), dtype=np.float32)
            canvas[...] = background_color

            if labels.size > 0:
                # Pick a random color for each label present in this image (deterministic seed + RNG state)
                # We'll create a dict-like mapping using two arrays and vectorized indexing.
                # 1) draw colors for the present labels
                chosen_idx = rng.integers(0, len(palette), size=labels.size)
                chosen_colors = palette[chosen_idx]  # (n_labels, 3)

                # 2) map pixels: use an index image via searchsorted on sorted labels
                labels_sorted = np.sort(labels)
                colors_sorted = chosen_colors[np.argsort(labels)]  # align with labels_sorted

                # create an index for all pixels whose label != 0
                m = mask
                fg = (m != 0)
                idx = np.searchsorted(labels_sorted, m[fg])
                canvas[fg] = colors_sorted[idx]

            out_path = path_mask_color / f"{mask_path.stem}.png"
            plt.imsave(str(out_path), canvas)  # float in [0,1]
            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {mask_path.name}: {e}")
            else:
                print(f"[WARN] Failed {mask_path.name}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    print(f"✅ Folder with color masks available there: {path_mask_color}")

    return {"processed": processed, "failed": failed, "output_root": str(path_mask_color)}

In [ ]:
! pip -q install stardist csbdeep tifffile scikit-image imageio tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.9 MB/s eta 0:00:00


In [ ]:
# ========= StarDist batch nuclei segmentation (biopsy subfolders) + dna_marker filter + progress bar =========
# Colab install (if needed):
# !pip -q install stardist csbdeep tifffile scikit-image imageio tqdm

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import tifffile as tiff
import imageio.v3 as iio
from tqdm.auto import tqdm

from stardist.models import StarDist2D
from csbdeep.utils import normalize
from skimage.morphology import remove_small_objects


IMG_EXTS = (".tif", ".tiff", ".ome.tif", ".ome.tiff", ".png", ".jpg", ".jpeg")


# -------------------------
# IO helpers
# -------------------------
def read_2d_gray(path: str | Path) -> np.ndarray:
    """Read an image as float32 grayscale (H,W). If RGB/multi-dim, take channel/page 0."""
    path = Path(path)
    ext = path.suffix.lower()

    if ext in (".tif", ".tiff", ".ome.tif", ".ome.tiff"):
        arr = tiff.imread(str(path))
    else:
        arr = iio.imread(str(path))

    arr = np.asarray(arr)

    if arr.ndim == 2:
        out = arr
    elif arr.ndim == 3:
        # (H,W,C) -> channel 0 ; (Z,H,W) -> first plane
        out = arr[..., 0] if arr.shape[-1] in (3, 4) else arr[0, ...]
    elif arr.ndim >= 4:
        # take first plane then first channel if needed
        out = arr.reshape(-1, *arr.shape[-2:])[0]
    else:
        raise ValueError(f"Unsupported ndim={arr.ndim} for {path}")

    return out.astype(np.float32)


def relabel_consecutive(lbl: np.ndarray) -> np.ndarray:
    """Relabel to consecutive 1..N."""
    lbl = lbl.astype(np.int32)
    ids = np.unique(lbl)
    ids = ids[ids != 0]
    if len(ids) == 0:
        return lbl
    out = np.zeros_like(lbl, dtype=np.int32)
    for new, old in enumerate(ids, start=1):
        out[lbl == old] = new
    return out


def list_biopsy_dirs(input_dir: str | Path) -> list[Path]:
    """Return immediate subdirectories (biopsies) under input_dir."""
    input_dir = Path(input_dir)
    return sorted([p for p in input_dir.iterdir() if p.is_dir()])


def find_dna_image_in_biopsy(
    biopsy_dir: Path,
    dna_marker: str,
    *,
    match_mode: str = "stem_contains",   # "stem_contains" | "stem_equals" | "filename_contains"
    exts: Iterable[str] = IMG_EXTS,
) -> Optional[Path]:
    """
    Find the nucleus marker image within a biopsy folder.

    Examples:
      dna_marker="Ir191" with stem_contains matches "Ir191", "Ir191_193", "DNA1_Ir191" etc.
      dna_marker="DNA"    with filename_contains matches "...DNA..." anywhere in filename.

    Returns:
      Path or None if not found / ambiguous.
    """
    dna_marker_l = dna_marker.lower().strip()
    candidates: list[Path] = []

    for p in sorted(biopsy_dir.iterdir()):
        if not p.is_file():
            continue
        if not p.name.lower().endswith(tuple(exts)):
            continue

        stem_l = p.stem.lower()
        name_l = p.name.lower()

        if match_mode == "stem_equals":
            ok = (stem_l == dna_marker_l)
        elif match_mode == "filename_contains":
            ok = (dna_marker_l in name_l)
        else:  # default: stem_contains
            ok = (dna_marker_l in stem_l)

        if ok:
            candidates.append(p)

    if len(candidates) == 1:
        return candidates[0]

    # If multiple matches, prefer an exact stem match if present
    if len(candidates) > 1:
        exact = [p for p in candidates if p.stem.lower() == dna_marker_l]
        if len(exact) == 1:
            return exact[0]

    # None if not found or ambiguous
    return None



In [ ]:
from pathlib import Path
from typing import Iterable, Optional
import numpy as np
import tifffile as tiff
from tqdm.auto import tqdm

from stardist.models import StarDist2D
from csbdeep.utils import normalize
from skimage.morphology import remove_small_objects

IMG_EXTS = (".tif", ".tiff", ".ome.tif", ".ome.tiff", ".png", ".jpg", ".jpeg")

def segment_nuclei_stardist_by_biopsy(
    input_dir: str | Path,
    output_dir: str | Path,
    *,
    dna_marker: str,
    match_mode: str = "stem_contains",
    model_name: str = "2D_versatile_fluo",
    prob_thresh: float = 0.2,
    nms_thresh: float = 0.4,
    min_size: int =None,
    normalize_pmin: float = None,
    normalize_pmax: float = None,
    overwrite: bool = False,
) -> dict:
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)          # ✅ FIX 1
    output_dir.mkdir(parents=True, exist_ok=True)

    biopsy_dirs = sorted([p for p in input_dir.iterdir() if p.is_dir()])
    if not biopsy_dirs:
        raise ValueError(f"No biopsy subfolders found in: {input_dir}")

    model = StarDist2D.from_pretrained(model_name)

    processed = 0
    skipped_exists = 0
    missing_dna, ambiguous_dna = [], []

    for bdir in tqdm(biopsy_dirs, desc="StarDist nuclei (by biopsy)", unit="biopsy", ncols=110):
        dna_path = find_dna_image_in_biopsy(bdir, dna_marker, match_mode=match_mode)
        if dna_path is None:
            # determine missing vs ambiguous
            dna_marker_l = dna_marker.lower().strip()
            cand = [p for p in bdir.iterdir()
                    if p.is_file()
                    and p.name.lower().endswith(tuple(IMG_EXTS))
                    and (dna_marker_l in p.stem.lower())]
            if len(cand) == 0:
                missing_dna.append(str(bdir))
            else:
                ambiguous_dna.append(str(bdir))
            continue


        out_path = output_dir / f"{bdir.name}.tif"


        nuc = read_2d_gray(dna_path)
        if normalize_pmin!=None or normalize_pmax!=None:
          x = normalize(nuc, pmin=normalize_pmin, pmax=normalize_pmax, axis=None)
        else:
          x=nuc
        labels, _ = model.predict_instances(
            x,
            prob_thresh=float(prob_thresh),
            nms_thresh=float(nms_thresh),
        )

        if min_size and int(min_size) > 0:
            labels = remove_small_objects(labels, min_size=int(min_size))
            labels = relabel_consecutive(labels)

        # write + compression (optional but recommended)
        tiff.imwrite(str(out_path), labels.astype(np.int32), compression="zlib")
        processed += 1

    return {
        "processed": processed,
        "skipped_exists": skipped_exists,
        "missing_dna": missing_dna,
        "ambiguous_dna": ambiguous_dna,
        "input_dir": str(input_dir),
        "output_dir": str(output_dir),
        "dna_marker": dna_marker,
        "match_mode": match_mode,
        "model_name": model_name,
        "prob_thresh": prob_thresh,
        "nms_thresh": nms_thresh,
        "min_size": min_size,
    }


#### Executions

In [ ]:
processing=input("Enter the image processing you want (1 or 2): ")

Enter the image processing you want: 2


In [ ]:
path_mask_stardist=path_segmentation_cells_mask+"stardist/"
path_mask_color_stardist=path_segmentation_cells_mask_color+"stardist/"
path_img_processing=path+"images/img_processing_"+processing+"/Biopsies/"
if os.path.isdir(path_mask_stardist)==False:
  os.mkdir(path_mask_stardist)
if os.path.isdir(path_mask_color_stardist)==False:
  os.mkdir(path_mask_color_stardist)


In [ ]:
dna_marker=input("Enter the dna marker: ")

Enter the dna marker: DNA


In [ ]:
report = segment_nuclei_stardist_by_biopsy(
    input_dir=path_img_raw,
    normalize_pmax=99.9,
    normalize_pmin=0.1,
    nms_thresh=0.6,
    prob_thresh=0.2,
    output_dir=path_mask_stardist,
    dna_marker=dna_marker,
)

Found model '2D_versatile_fluo' for 'StarDist2D'.
5320433/5320433 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


StarDist nuclei (by biopsy):   0%|                                                 | 0/89 [00:00<?, ?biopsy/s]

#### 📊 Colored masks

In [ ]:
report = export_colored_instance_masks_flat(
     path_mask=path_mask_stardist,
     path_mask_color=path_mask_color_stardist,
     seed=0,
)
print(report)

Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/stardist
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/stardist'}


### Combine algorithms to select bigest cells
Possibility to combine two predictions.
for each mask it chooses the biggest between the two algorithm

In [ ]:
list_algo=input("Enter the two algorithms you want to combine separated by a space: ").split()

Enter the two algorithms you want to combine separated by a space: mesmer cellposev3


In [ ]:
 path_mask=path_segmentation_cells+"mask/"
 path_mask_combine=path_segmentation_cells+"mask/combine_"+list_algo[0]+"_"+list_algo[1]+"/"
 if os.path.isdir(path_mask_combine)==False:
    os.mkdir(path_mask_combine)
    print("✅ Folder for combined mask created")
 path_mask_color_combine=path_segmentation_cells+"mask_color/combine_"+list_algo[0]+"_"+list_algo[1]+"/"
 if os.path.isdir(path_mask_color_combine)==False:
    os.mkdir(path_mask_color_combine)
    print("✅ Folder for color combined masks created")

✅ Folder for combined mask created
✅ Folder for color combined masks created


#### Functions

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Sequence, Dict

import numpy as np
from PIL import Image
from tqdm.auto import tqdm


def _mode_int(values: np.ndarray) -> int:
    vals, counts = np.unique(values, return_counts=True)
    return int(vals[np.argmax(counts)])


def combine_instance_masks_two_algos(
    *,
    path_mask: str,
    list_algo: Sequence[str],
    path_mask_combine: str,
    iou_thresh: float = 0.4,
    save_dtype: str = "uint16",          # "uint8" | "uint16" | "int32"
    skip_missing_in_algo2: bool = True,
    progress: bool = True,
    progress_objects: bool = False,      # NEW: inner progress per-object (slower/verbose)
) -> Dict[str, int]:
    """
    Combine instance masks from two algorithms into a single mask per image.

    Adds progress bars:
      - One clean bar over images (always recommended)
      - Optional inner bar over objects within an image (debug; slower)
    """
    if len(list_algo) != 2:
        raise ValueError("list_algo must contain exactly 2 algorithm names, e.g. ['algo1','algo2'].")

    algo1, algo2 = list_algo[0], list_algo[1]

    algo1_dir = Path(path_mask) / algo1
    algo2_dir = Path(path_mask) / algo2
    out_dir = Path(path_mask_combine)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not algo1_dir.is_dir():
        raise FileNotFoundError(f"Missing folder: {algo1_dir}")
    if not algo2_dir.is_dir():
        raise FileNotFoundError(f"Missing folder: {algo2_dir}")

    file_list = sorted([p.name for p in algo1_dir.iterdir() if p.is_file()])
    if not file_list:
        raise RuntimeError(f"No files found in: {algo1_dir}")

    dtype_map = {"uint8": np.uint8, "uint16": np.uint16, "int32": np.int32}
    if save_dtype not in dtype_map:
        raise ValueError(f"save_dtype must be one of {list(dtype_map)}, got {save_dtype}")
    out_dtype = dtype_map[save_dtype]

    processed = 0
    skipped_missing = 0
    failed = 0

    # ✅ MAIN progress bar (single line)
    iterator = tqdm(file_list, desc="Combining masks", unit="img", ncols=110) if progress else file_list

    for fname in iterator:
        try:
            p1 = algo1_dir / fname
            p2 = algo2_dir / fname
            if not p2.exists():
                if skip_missing_in_algo2:
                    skipped_missing += 1
                    if progress:
                        iterator.set_postfix_str(f"skipped (missing in {algo2})", refresh=False)
                    continue
                raise FileNotFoundError(f"Missing in algo2: {p2}")

            mask1 = np.array(Image.open(p1), dtype=np.int32)
            mask2 = np.array(Image.open(p2), dtype=np.int32)

            if mask1.shape != mask2.shape:
                raise ValueError(f"Shape mismatch for {fname}: {mask1.shape} vs {mask2.shape}")

            mask3 = np.zeros_like(mask1, dtype=np.int32)
            obj3 = 1

            objs1 = np.unique(mask1)
            objs1 = objs1[objs1 != 0]

            # Optional inner progress bar (useful for debugging; can slow things down)
            obj_iter = tqdm(objs1, desc=f"Objects {fname}", unit="obj", leave=False, ncols=110) if progress_objects else objs1

            for obj1 in obj_iter:
                region1 = (mask1 == obj1)
                overlap_labels = mask2[region1]
                overlap_labels = overlap_labels[overlap_labels > 0]

                if overlap_labels.size > 0:
                    obj2 = _mode_int(overlap_labels)

                    region2 = (mask2 == obj2)
                    inter = np.logical_and(region1, region2).sum()
                    union = np.logical_or(region1, region2).sum()
                    iou = (inter / union) if union > 0 else 0.0

                    area1 = int(region1.sum())
                    area2 = int(region2.sum())

                    if iou >= float(iou_thresh):
                        mask3[region1 if area1 >= area2 else region2] = obj3
                        obj3 += 1
                    else:
                        mask3[region1] = obj3
                        obj3 += 1
                else:
                    mask3[region1] = obj3
                    obj3 += 1

            # Pass 2: add objects only in mask2
            for obj2 in np.unique(mask2):
                if obj2 == 0:
                    continue
                region2 = (mask2 == obj2)
                if np.sum(mask1[region2]) == 0:
                    mask3[region2] = obj3
                    obj3 += 1

            # Safety: uint8 overflow
            if out_dtype == np.uint8 and mask3.max() > 255:
                raise ValueError(
                    f"{fname}: combined mask has {mask3.max()} objects > 255. "
                    "Use save_dtype='uint16' or 'int32'."
                )

            out_path = out_dir / fname
            Image.fromarray(mask3.astype(out_dtype)).save(out_path)
            processed += 1

            if progress:
                iterator.set_postfix_str(f"ok | objs={int(mask3.max())}", refresh=False)

        except Exception as e:
            failed += 1
            if progress:
                iterator.write(f"[WARN] Failed {fname}: {e}")
                iterator.set_postfix_str("failed", refresh=False)
            else:
                print(f"[WARN] Failed {fname}: {e}")

    return {
        "processed": processed,
        "skipped_missing_algo2": skipped_missing,
        "failed": failed,
        "output_dir": str(out_dir),
    }

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


def export_colored_instance_masks_flat(
    *,
    path_mask: str,
    path_mask_color: str,
    seed: int = 0,
    use_all_matplotlib_colors: bool = True,
    exts_in: Tuple[str, ...] = (".tif", ".tiff", ".png"),
    progress: bool = True,
    progress_ncols: int = 110,
    background_color: Tuple[float, float, float] = (0.0, 0.0, 0.0),
) -> Dict[str, int]:
    """
    Colorize an *instance-labeled* mask folder (flat layout) WITHOUT re-labeling.

    Assumes:
      - mask pixels are integer labels
      - 0 = background
      - each object has its own label value (1..N, not necessarily contiguous)

    Output:
      - saves PNG RGB images with same stem into `path_mask_color`.

    Returns: processed / failed / output_root
    """
    path_mask = Path(path_mask)
    path_mask_color = Path(path_mask_color)
    path_mask_color.mkdir(parents=True, exist_ok=True)

    # Palette (RGB in [0,1])
    if use_all_matplotlib_colors:
        palette = [matplotlib.colors.to_rgb(hx) for hx in matplotlib.colors.cnames.values()]
    else:
        palette = [matplotlib.colors.to_rgb(hx) for hx in [
            "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
            "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
        ]]
    if not palette:
        raise RuntimeError("No colors available for palette.")
    palette = np.asarray(palette, dtype=np.float32)

    rng = np.random.default_rng(seed)

    # List mask files (flat)
    exts_l = tuple(e.lower() for e in exts_in)
    mask_files = sorted([p for p in path_mask.iterdir() if p.is_file() and p.suffix.lower() in exts_l])
    if not mask_files:
        raise RuntimeError(f"No mask files found in: {path_mask}")

    processed = 0
    failed = 0
    pbar = tqdm(total=len(mask_files), desc="Colorizing instance masks", unit="mask", ncols=progress_ncols) if progress else None

    for mask_path in mask_files:
        if pbar is not None:
            pbar.set_postfix_str(mask_path.name, refresh=False)

        try:
            mask = np.array(Image.open(mask_path))

            # Ensure integer labels
            if mask.dtype.kind not in ("u", "i"):
                # If mask was saved as float, cast safely (but ideally masks should be int)
                mask = mask.astype(np.int32)

            labels = np.unique(mask)
            labels = labels[labels != 0]  # exclude background

            H, W = mask.shape[:2]
            canvas = np.zeros((H, W, 3), dtype=np.float32)
            canvas[...] = background_color

            if labels.size > 0:
                # Pick a random color for each label present in this image (deterministic seed + RNG state)
                # We'll create a dict-like mapping using two arrays and vectorized indexing.
                # 1) draw colors for the present labels
                chosen_idx = rng.integers(0, len(palette), size=labels.size)
                chosen_colors = palette[chosen_idx]  # (n_labels, 3)

                # 2) map pixels: use an index image via searchsorted on sorted labels
                labels_sorted = np.sort(labels)
                colors_sorted = chosen_colors[np.argsort(labels)]  # align with labels_sorted

                # create an index for all pixels whose label != 0
                m = mask
                fg = (m != 0)
                idx = np.searchsorted(labels_sorted, m[fg])
                canvas[fg] = colors_sorted[idx]

            out_path = path_mask_color / f"{mask_path.stem}.png"
            plt.imsave(str(out_path), canvas)  # float in [0,1]
            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {mask_path.name}: {e}")
            else:
                print(f"[WARN] Failed {mask_path.name}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    print(f"✅ Folder with color masks available there: {path_mask_color}")

    return {"processed": processed, "failed": failed, "output_root": str(path_mask_color)}

#### Combinaison

In [ ]:
report = combine_instance_masks_two_algos(
     path_mask=path_mask,
     list_algo=list_algo,
     path_mask_combine=path_mask_combine,
     iou_thresh=0.4,
     save_dtype="uint16",
 )
print(report)

Combining masks:   0%|                                                                | 0/89 [00:00<?, ?img/s]

{'processed': 89, 'skipped_missing_algo2': 0, 'failed': 0, 'output_dir': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask/combine_mesmer_cellposev3'}


#### Colored masks

In [ ]:
report = export_colored_instance_masks_flat(
     path_mask=path_mask_combine,
     path_mask_color=path_mask_color_combine,
     seed=0,
)
print(report)

Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/combine_mesmer_cellposev3
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color/combine_mesmer_cellposev3'}


# ⚙️ **Size filter**
⚠️ USER INPUT REQUIRED  
Removing too small or too big objects

## Functions

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


def export_colored_instance_masks_flat(
    *,
    path_mask: str,
    path_mask_color: str,
    seed: int = 0,
    use_all_matplotlib_colors: bool = True,
    exts_in: Tuple[str, ...] = (".tif", ".tiff", ".png"),
    progress: bool = True,
    progress_ncols: int = 110,
    background_color: Tuple[float, float, float] = (0.0, 0.0, 0.0),
) -> Dict[str, int]:
    """
    Colorize an *instance-labeled* mask folder (flat layout) WITHOUT re-labeling.

    Assumes:
      - mask pixels are integer labels
      - 0 = background
      - each object has its own label value (1..N, not necessarily contiguous)

    Output:
      - saves PNG RGB images with same stem into `path_mask_color`.

    Returns: processed / failed / output_root
    """
    path_mask = Path(path_mask)
    path_mask_color = Path(path_mask_color)
    path_mask_color.mkdir(parents=True, exist_ok=True)

    # Palette (RGB in [0,1])
    if use_all_matplotlib_colors:
        palette = [matplotlib.colors.to_rgb(hx) for hx in matplotlib.colors.cnames.values()]
    else:
        palette = [matplotlib.colors.to_rgb(hx) for hx in [
            "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
            "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
        ]]
    if not palette:
        raise RuntimeError("No colors available for palette.")
    palette = np.asarray(palette, dtype=np.float32)

    rng = np.random.default_rng(seed)

    # List mask files (flat)
    exts_l = tuple(e.lower() for e in exts_in)
    mask_files = sorted([p for p in path_mask.iterdir() if p.is_file() and p.suffix.lower() in exts_l])
    if not mask_files:
        raise RuntimeError(f"No mask files found in: {path_mask}")

    processed = 0
    failed = 0
    pbar = tqdm(total=len(mask_files), desc="Colorizing instance masks", unit="mask", ncols=progress_ncols) if progress else None

    for mask_path in mask_files:
        if pbar is not None:
            pbar.set_postfix_str(mask_path.name, refresh=False)

        try:
            mask = np.array(Image.open(mask_path))

            # Ensure integer labels
            if mask.dtype.kind not in ("u", "i"):
                # If mask was saved as float, cast safely (but ideally masks should be int)
                mask = mask.astype(np.int32)

            labels = np.unique(mask)
            labels = labels[labels != 0]  # exclude background

            H, W = mask.shape[:2]
            canvas = np.zeros((H, W, 3), dtype=np.float32)
            canvas[...] = background_color

            if labels.size > 0:
                # Pick a random color for each label present in this image (deterministic seed + RNG state)
                # We'll create a dict-like mapping using two arrays and vectorized indexing.
                # 1) draw colors for the present labels
                chosen_idx = rng.integers(0, len(palette), size=labels.size)
                chosen_colors = palette[chosen_idx]  # (n_labels, 3)

                # 2) map pixels: use an index image via searchsorted on sorted labels
                labels_sorted = np.sort(labels)
                colors_sorted = chosen_colors[np.argsort(labels)]  # align with labels_sorted

                # create an index for all pixels whose label != 0
                m = mask
                fg = (m != 0)
                idx = np.searchsorted(labels_sorted, m[fg])
                canvas[fg] = colors_sorted[idx]

            out_path = path_mask_color / f"{mask_path.stem}.png"
            plt.imsave(str(out_path), canvas)  # float in [0,1]
            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {mask_path.name}: {e}")
            else:
                print(f"[WARN] Failed {mask_path.name}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    print(f"✅ Folder with color masks available there: {path_mask_color}")

    return {"processed": processed, "failed": failed, "output_root": str(path_mask_color)}

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, Optional

import numpy as np
from PIL import Image
from tqdm.auto import tqdm


def filter_instance_masks_by_size(
    *,
    in_root: str,
    out_root: str,
    min_size: int,
    max_size: int,
    exts: Optional[tuple[str, ...]] = None,   # e.g. (".tif",".tiff",".png")
    keep_background: bool = True,            # kept for clarity; background always 0
    progress: bool = True,
    progress_ncols: int = 110,
) -> Dict[str, int]:
    """
    Filter instance-labeled masks by object area (pixel count).

    Expected input layout:
        in_root/
          algo1/
            img001.tif
            img002.tif
          algo2/
            ...

    Output layout:
        out_root/
          algo1/
            img001.tif
            ...
          algo2/
            ...

    Filtering rule (strict by default):
        keep labels with: min_size < area < max_size
        everything else -> 0 (background)

    Parameters
    ----------
    in_root : str
        Root folder containing one subfolder per algorithm (instance masks).
    out_root : str
        Output root folder; will be created.
    min_size : int
        Minimum object area in pixels (strictly greater than this is kept).
    max_size : int
        Maximum object area in pixels (strictly less than this is kept).
    exts : tuple[str,...] | None
        If provided, only process files with these extensions (case-insensitive).
    progress : bool
        If True, shows a single clean progress bar over all masks.

    Returns
    -------
    Dict[str,int]
        Summary: processed, skipped_ext, failed, algos.
    """
    min_size = int(min_size)
    max_size = int(max_size)
    if min_size < 0 or max_size <= 0:
        raise ValueError("min_size must be >= 0 and max_size must be > 0.")
    if max_size <= min_size:
        raise ValueError("max_size must be > min_size.")

    in_root_p = Path(in_root)
    out_root_p = Path(out_root)
    out_root_p.mkdir(parents=True, exist_ok=True)

    algos = sorted([p.name for p in in_root_p.iterdir() if p.is_dir()])
    if not algos:
        raise RuntimeError(f"No algorithm subfolders found in: {in_root_p}")

    # Build flat job list for ONE nice progress bar
    jobs = []
    for algo in algos:
        src_dir = in_root_p / algo
        for p in sorted([x for x in src_dir.iterdir() if x.is_file()]):
            if exts is not None:
                if p.suffix.lower() not in tuple(e.lower() for e in exts):
                    continue
            jobs.append((algo, p))

    if not jobs:
        raise RuntimeError("No mask files found to process (check exts filter?).")

    processed = 0
    skipped_ext = 0
    failed = 0

    pbar = tqdm(total=len(jobs), desc="Filtering masks by size", unit="mask", ncols=progress_ncols) if progress else None

    for algo, src_path in jobs:
        try:
            dst_dir = out_root_p / algo
            dst_dir.mkdir(parents=True, exist_ok=True)
            dst_path = dst_dir / src_path.name

            if pbar is not None:
                pbar.set_postfix_str(f"{algo} | {src_path.name}", refresh=False)

            mask = np.array(Image.open(src_path))

            # Compute area per label
            labels, counts = np.unique(mask, return_counts=True)

            # Keep labels strictly between thresholds (exclude background 0 automatically)
            keep = labels[(labels != 0) & (counts > min_size) & (counts < max_size)]

            if keep.size == 0:
                mask_filt = np.zeros_like(mask)
            else:
                mask_filt = np.where(np.isin(mask, keep), mask, 0).astype(mask.dtype)

            Image.fromarray(mask_filt).save(dst_path)
            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {algo}/{src_path.name}: {e}")
            else:
                print(f"[WARN] Failed {algo}/{src_path.name}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    return {
        "algos": len(algos),
        "processed": processed,
        "failed": failed,
        "output_root": str(out_root_p),
    }



In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, Optional, Tuple, Iterable

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import skimage as ski


def export_colored_masks(
    *,
    path_mask_filter: str,
    path_mask_color_filter: str,
    min_size: int = 0,
    max_size: int = 10**12,
    seed: int = 0,
    use_all_matplotlib_colors: bool = True,
    exts_in: Tuple[str, ...] = (".tif", ".tiff", ".png"),
    progress: bool = True,
    progress_ncols: int = 110,
) -> Dict[str, int]:
    """
    Colorize instance masks for visualization.

    Expected folder layout:
        path_mask_filter/
          algo1/
            img001.tif (instance mask, background=0)
            ...
          algo2/
            ...

    Output:
        path_mask_color_filter/
          algo1/
            img001.png
          algo2/
            img001.png

    What it does:
      - Reads each mask
      - (Re-)labels connected components (to ensure unique components)
      - Computes regionprops (coords, area, etc.)
      - Paints each object with a random color (deterministic via `seed`)
      - Saves a PNG image

    Parameters
    ----------
    min_size, max_size : int
        Object area filtering in pixels.
        IMPORTANT: Your original code used:
            if area > min_size OR area < max_size
        which is almost always True.
        Here we use the correct logic:
            keep if (min_size < area < max_size)
        If you really want the original (likely buggy) behavior, tell me and I’ll add a switch.
    seed : int
        Random seed for reproducible colors.
    use_all_matplotlib_colors : bool
        If True, uses matplotlib CSS named colors as palette.
        If False, uses a smaller distinct palette.
    exts_in : tuple
        Which input extensions to process.
    progress : bool
        Show one clean progress bar over all masks (not nested).

    Returns
    -------
    dict
        processed / failed / algos / output_root
    """
    path_mask_filter = Path(path_mask_filter)
    path_mask_color_filter = Path(path_mask_color_filter)
    path_mask_color_filter.mkdir(parents=True, exist_ok=True)

    min_size = int(min_size)
    max_size = int(max_size)
    if max_size <= min_size:
        raise ValueError("max_size must be > min_size")

    # --- Build palette (RGB tuples in [0,1]) ---
    if use_all_matplotlib_colors:
        rgb_colors = [matplotlib.colors.to_rgb(hx) for hx in matplotlib.colors.cnames.values()]
    else:
        # compact distinct palette
        rgb_colors = [matplotlib.colors.to_rgb(hx) for hx in [
            "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
            "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
        ]]

    if len(rgb_colors) == 0:
        raise RuntimeError("No colors available for palette.")

    rng = np.random.default_rng(seed)

    # --- Discover algorithms ---
    algos = sorted([p.name for p in path_mask_filter.iterdir() if p.is_dir()])
    if not algos:
        raise RuntimeError(f"No algorithm subfolders found in: {path_mask_filter}")

    # --- Build flat job list for ONE clean progress bar ---
    jobs: list[tuple[str, Path]] = []
    exts_l = tuple(e.lower() for e in exts_in)

    for algo in algos:
        algo_dir = path_mask_filter / algo
        for p in sorted([x for x in algo_dir.iterdir() if x.is_file()]):
            if p.suffix.lower() in exts_l:
                jobs.append((algo, p))

    if not jobs:
        raise RuntimeError("No mask files found to process.")

    processed = 0
    failed = 0

    pbar = tqdm(total=len(jobs), desc="Colorizing masks", unit="mask", ncols=progress_ncols) if progress else None

    for algo, mask_path in jobs:
        if pbar is not None:
            pbar.set_postfix_str(f"{algo} | {mask_path.name}", refresh=False)

        try:
            out_dir = path_mask_color_filter / algo
            out_dir.mkdir(parents=True, exist_ok=True)

            mask = np.array(Image.open(mask_path))

            # Re-label connected components (ensures contiguous ids for regionprops)
            lbl = ski.measure.label(mask > 0, connectivity=mask.ndim)

            props = ski.measure.regionprops_table(
                lbl,
                intensity_image=mask,
                properties=["area", "coords"],
            )
            df = pd.DataFrame(props)

            canvas = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.float32)

            # Paint each object
            for i in range(df.shape[0]):
                area = int(df.loc[i, "area"])
                if not (min_size < area < max_size):
                    continue
                color = rgb_colors[int(rng.integers(0, len(rgb_colors)))]
                coords = df.loc[i, "coords"]
                # coords is an array of (row,col)
                canvas[coords[:, 0], coords[:, 1], :] = color

            out_path = out_dir / f"{mask_path.stem}.png"
            plt.imsave(str(out_path), canvas)  # expects float in [0,1]
            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {algo}/{mask_path.name}: {e}")
            else:
                print(f"[WARN] Failed {algo}/{mask_path.name}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    print(f"✅ Folder with color masks available there: {path_mask_color_filter}")

    return {
        "algos": len(algos),
        "processed": processed,
        "failed": failed,
        "output_root": str(path_mask_color_filter),
    }


## Execution


In [ ]:
path_segmentation_cells_mask=path_segmentation_cells+"mask/"
path_mask_filter=path_segmentation_cells+"mask_filtered/"
path_mask=path_segmentation_cells_mask
if os.path.isdir(path_mask_filter)==False:
  os.mkdir(path_mask_filter)
  print("✅ Folder for filtered masks created")
path_mask_color_filter=path_segmentation_cells+"mask_color_filtered/"
path_obj_color_filter=path_segmentation_cells+"obj_color_filtered/"
if os.path.isdir(path_mask_color_filter)==False:
  os.mkdir(path_mask_color_filter)
  print("✅ Folder for colored filtered masks created")
if os.path.isdir(path_obj_color_filter)==False:
  os.mkdir(path_obj_color_filter)
  print("✅ Folder for colored filtered objects created")

In [ ]:
min_size=int(input("Enter the minimum size of the cell you want to keep: "))
max_size=int(input("Enter the maximum size of the cell you want to keep:"))

Enter the minimum size of the cell you want to keep: 15
Enter the maximum size of the cell you want to keep:300


#### Create the mask with no object smaller than the minimum size and bigger than the maximum size

In [ ]:
report = filter_instance_masks_by_size(
     in_root=path_segmentation_cells_mask,
     out_root=path_mask_filter,
     min_size=min_size,
     max_size=max_size,
     exts=(".tif", ".tiff", ".png"),
 )
print(report)

Filtering masks by size:   0%|                                                      | 0/445 [00:00<?, ?mask/s]

{'algos': 5, 'processed': 445, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_filtered'}


#### Mask colored

In [ ]:
for algo in os.listdir(path_mask_filter):
 report = export_colored_instance_masks_flat(
     path_mask=path_mask_filter+algo+"/",
     path_mask_color=path_mask_color_filter+algo+"/",
     seed=0,
 )
 print(report)

Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/instanseg
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/instanseg'}


Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/mesmer
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/mesmer'}


Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/stardist
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/stardist'}


Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/cellposev3
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/cellposev3'}


Colorizing instance masks:   0%|                                                     | 0/89 [00:00<?, ?mask/s]

✅ Folder with color masks available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/combine_mesmer_cellposev3
{'processed': 89, 'failed': 0, 'output_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/mask_color_filtered/combine_mesmer_cellposev3'}


#### Images with all the filtered objects

In [ ]:
# ------------------------- Configuration -------------------------
min_size = int(min_size)
max_size = int(max_size)

rgb_colors = {name: matplotlib.colors.to_rgb(hex_) for name, hex_ in matplotlib.colors.cnames.items()}
palette = np.array(list(rgb_colors.values()), dtype=np.float32)

rng = np.random.default_rng()

os.makedirs(path_obj_color_filter, exist_ok=True)

# ------------------------- Processing with ONE progress bar -------------------------
algos = sorted([d for d in os.listdir(path_mask)
                if os.path.isdir(os.path.join(path_mask, d))])

# Total number of images across all algorithms
total_files = sum(len(os.listdir(os.path.join(path_mask, algo))) for algo in algos)

with tqdm(total=total_files, desc="Processing masks", unit="img") as pbar:
    for file_algo in algos:
        src_dir = os.path.join(path_mask, file_algo)
        dst_dir = os.path.join(path_obj_color_filter, file_algo)
        os.makedirs(dst_dir, exist_ok=True)

        files = sorted([f for f in os.listdir(src_dir)
                        if os.path.isfile(os.path.join(src_dir, f))])

        for file_mask in files:
            src_path = os.path.join(src_dir, file_mask)
            out_path = os.path.join(dst_dir, f"{os.path.splitext(file_mask)[0]}.png")

            # Load mask
            mask = np.array(Image.open(src_path))
            H, W = mask.shape[:2]

            # Compute object sizes
            labels, counts = np.unique(mask, return_counts=True)
            valid = labels != 0
            labels = labels[valid]
            counts = counts[valid]

            # Select labels OUTSIDE the interval [min_size, max_size]
            keep_labels = labels[(counts > max_size) | (counts < min_size)]

            # Create empty RGB canvas
            color_img = np.zeros((H, W, 3), dtype=np.float32)

            if keep_labels.size > 0:
                rnd_idx = rng.integers(0, len(palette), size=keep_labels.size)
                keep_colors = palette[rnd_idx]

                for lab, col in zip(keep_labels, keep_colors):
                    color_img[mask == lab] = col

            # Save result
            plt.imsave(out_path, color_img)

            # Update ONE progress bar
            pbar.update(1)
print("✅ Folder with color objects available there: "+ path_obj_color_filter)

Processing masks:   0%|          | 0/445 [00:00<?, ?img/s]

✅ Folder with color objects available there: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/obj_color_filtered/


# 📊 **Assessment without a ground truth**
*The goal here is to assess the quality of the segmentation without ground truth. To do this, we can compare the size and number of objects, examine the amount of DNA outside the created objects, and use various visualization tools.*

In [ ]:
processing=input("Enter the image processing you want to choose (1 or 2): ")

Enter the image processing you want to choose (1 or 2): 2


In [ ]:
path_eval_quali=path_segmentation_cells+"QC_Segmentation/"
path_img_processing1=path+"images/img_processing_"+processing+"/"
path_eval_quali_without_truth=path_eval_quali+"assessment_without_ground_truth/"
path_eval_mask_outline=path_eval_quali_without_truth+"mask_outline/"
path_mask=path_segmentation_cells+"mask_filtered/"
if os.path.isdir(path_eval_quali)==False:
 os.mkdir(path_eval_quali)
if os.path.isdir(path_eval_quali_without_truth)==False:
  os.mkdir(path_eval_quali_without_truth)
if os.path.isdir(path_eval_mask_outline)==False:
  os.mkdir(path_eval_mask_outline)

### 📊 **Number of object**

#### Functions

In [ ]:
def mask_to_df(path_mask):
   mask= np.array(Image.open(path_mask))
   label=ski.measure.label(mask,connectivity=mask.ndim)
   df_mask=ski.measure.regionprops_table(label,intensity_image=mask,properties=["area","coords","equivalent_diameter_area","bbox","axis_major_length","axis_minor_length","centroid"])
   df_mask=pd.DataFrame(df_mask)
   return df_mask


In [66]:
def distribution_number_object(
    path_img: str = "./img",
    path_mask: str = "./mask",
    path_file: str = "./segmentation_quality/",
    dico_algo_min: dict = None,
    dico_algo_max: dict = None,
    font_size: int = 12,
    title: str = "",
):
    """
    Build per-image object counts for each algorithm's masks and plot:
      1) grouped bar chart (images on X, counts on Y, one bar per algorithm)
      2) distribution plot across algorithms (box + swarm)

    Improvements vs original:
    - Single clean progress bar (algo × image) instead of nested bars
    - Shows current algo + filename in postfix
    - Keeps intersection of filenames across algos for fair comparison
    """

    dico_algo_min = dico_algo_min or {}
    dico_algo_max = dico_algo_max or {}

    # --------------------------- Setup ---------------------------
    os.makedirs(path_file, exist_ok=True)

    algos = sorted([d.name for d in os.scandir(path_mask) if d.is_dir()])
    if not algos:
        raise RuntimeError(f"No algorithm subfolders found in '{path_mask}'.")

    # Intersection of filenames across algos (fair, stable)
    file_sets = []
    for algo in algos:
        algo_dir = os.path.join(path_mask, algo)
        files = {f.name for f in os.scandir(algo_dir) if f.is_file()}
        file_sets.append(files)

    common_files = sorted(set.intersection(*file_sets)) if file_sets else []
    if not common_files:
        # fallback: first algo files
        first_dir = os.path.join(path_mask, algos[0])
        common_files = sorted([f.name for f in os.scandir(first_dir) if f.is_file()])

    if not common_files:
        raise RuntimeError("No mask files found to process.")

    dico_algo_nb = {algo: [] for algo in algos}

    # --------------------------- Compute counts (single clean progress bar) ---------------------------
    total = len(algos) * len(common_files)
    pbar = tqdm(total=total, desc="Counting objects", unit="mask", ncols=110)

    for algo in algos:
        algo_dir = os.path.join(path_mask, algo)
        use_thresh = (algo != "truth") and (algo in dico_algo_min) and (algo in dico_algo_max)

        for fname in common_files:
            mask_path = os.path.join(algo_dir, fname)

            # Update progress display (nice, compact)
            pbar.set_postfix_str(f"algo={algo} | file={fname}", refresh=False)

            if use_thresh:
                df = mask_to_df(mask_path, dico_algo_min[algo], dico_algo_max[algo])
            else:
                df = mask_to_df(mask_path)

            dico_algo_nb[algo].append(int(df.shape[0]))
            pbar.update(1)

    pbar.close()

    # --------------------------- Plot 1: Grouped bar chart ---------------------------
    n_algos = len(algos)
    n_imgs = len(common_files)

    width = 0.8 / n_algos
    x_base = np.arange(n_imgs)

    plt.figure(figsize=(max(8, n_imgs * 0.5), 6))
    for i, algo in enumerate(algos):
        x_positions = x_base + i * width
        plt.bar(x_positions, dico_algo_nb[algo], label=algo, width=width)

    tick_positions = x_base + width * (n_algos - 1) / 2
    plt.title(title if title else "", fontsize=font_size)
    plt.ylabel("Number of cells", fontsize=font_size)
    plt.xlabel("Images", fontsize=font_size)
    plt.xticks(tick_positions, common_files, rotation=60, ha="right",fontsize=font_size)
    plt.legend(fontsize=font_size)
    plt.tight_layout()

    bar_out = os.path.join(path_file, (title if title else "number_of_cells_per_image") + ".png")
    plt.savefig(bar_out, dpi=200)
    plt.close()

    # --------------------------- Plot 2: Box + swarm distribution ---------------------------
    rows = [{"Algorithm": algo, "Objects": c} for algo in algos for c in dico_algo_nb[algo]]
    df_plot = pd.DataFrame(rows)

    tab20_distinct = [
        "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
        "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
    ]

    plt.figure(figsize=(max(8, n_algos * 2), 6))
    sns.swarmplot(data=df_plot, x="Algorithm", y="Objects", color="black", size=3)
    sns.boxplot(data=df_plot, x="Algorithm", y="Objects", showmeans=True, palette=tab20_distinct)
    plt.title(title if title else "", fontsize=font_size)
    plt.ylabel("Mean quantity of cells", fontsize=font_size)
    plt.xlabel("Algorithms", fontsize=font_size)
    plt.xticks(fontsize=font_size,rotation=30)
    plt.tight_layout()

    box_out = os.path.join(path_file, ("mean_" + title if title else "mean_number_of_cells") + ".png")
    plt.savefig(box_out, dpi=200)
    plt.close()

    print("✅ files created :", path_file)
    print(" -", bar_out)
    print(" -", box_out)


#### Execution

In [72]:
distribution_number_object(path_img_raw,path_mask,path_eval_quali_without_truth,title="Average number of cells per image",font_size=16)


Counting objects:   0%|                                                             | 0/445 [00:00<?, ?mask/s]

/tmp/ipykernel_176/825284896.py:108: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_plot, x="Algorithm", y="Objects", showmeans=True, palette=tab20_distinct)
/tmp/ipykernel_176/825284896.py:108: UserWarning: The palette list has more values (10) than needed (5), which may not be intended.
  sns.boxplot(data=df_plot, x="Algorithm", y="Objects", showmeans=True, palette=tab20_distinct)


✅ files created : /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/
 - /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/Average number of cells per image.png
 - /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/mean_Average number of cells per image.png


### 📊 **Object size distribution**

#### Functions

In [ ]:
def mask_to_df(path_mask):
   mask= np.array(Image.open(path_mask))
   label=ski.measure.label(mask,connectivity=mask.ndim)
   df_mask=ski.measure.regionprops_table(label,intensity_image=mask,properties=["area","coords","equivalent_diameter_area","bbox","axis_major_length","axis_minor_length","centroid"])
   df_mask=pd.DataFrame(df_mask)
   return df_mask


In [78]:
import os
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from matplotlib.patches import Patch


def size_distribution(
    path_mask: str = "./mask",
    path_file: str = "./segmentation_quality",
    *,
    min_size: Optional[float] = None,
    max_size: Optional[float] = None,
    title: str = "size_distribution.png",
    figsize: Tuple[int, int] = (10, 6),
    bins: int = 30,
    font_size: int = 12,
    intersection_only: bool = False,
    progress_ncols: int = 110,
) -> Dict[str, List[float]]:
    """
    Plot an overlaid histogram (with KDE) of object areas for each segmentation algorithm.

    If min_size and/or max_size are provided, areas are filtered with:
      - if min_size is not None: keep area > min_size
      - if max_size is not None: keep area < max_size

    Expected folder structure:
        path_mask/
            algo1/
                mask_001.tif (or png, etc.)
            algo2/
                ...
    """
    path_mask = Path(path_mask)
    path_out_dir = Path(path_file)
    path_out_dir.mkdir(parents=True, exist_ok=True)

    # ---------------------- Discover algorithms ----------------------
    algos = sorted([p.name for p in path_mask.iterdir() if p.is_dir()])
    if not algos:
        raise RuntimeError(f"No algorithm subfolders found in '{path_mask}'")

    # ---------------------- Build file list per algo ----------------------
    algo_to_files: Dict[str, List[Path]] = {}
    algo_to_nameset: Dict[str, set] = {}

    for algo in algos:
        algo_dir = path_mask / algo
        files = sorted([p for p in algo_dir.iterdir() if p.is_file()])
        algo_to_files[algo] = files
        algo_to_nameset[algo] = {p.name for p in files}

    if intersection_only:
        common = set.intersection(*(algo_to_nameset[a] for a in algos))
        if not common:
            raise RuntimeError("intersection_only=True but no common files across algorithms.")
        for algo in algos:
            algo_to_files[algo] = [p for p in algo_to_files[algo] if p.name in common]

    pairs: List[Tuple[str, Path]] = [(algo, p) for algo in algos for p in algo_to_files[algo]]
    if not pairs:
        raise RuntimeError("No mask files found to process.")

    # ---------------------- Collect areas ----------------------
    dico_algo_size: Dict[str, List[float]] = {algo: [] for algo in algos}

    pbar = tqdm(total=len(pairs), desc="Collecting areas", unit="mask", ncols=progress_ncols)
    for algo, mask_path in pairs:
        pbar.set_postfix_str(f"{algo} | {mask_path.name}", refresh=False)

        df_mask = mask_to_df(str(mask_path))  # mask_to_df must accept a single argument

        if "area" in df_mask.columns:
            areas = pd.to_numeric(df_mask["area"], errors="coerce").dropna()
            areas = areas[np.isfinite(areas)]

            # ✅ Apply filters only if provided
            if min_size is not None:
                areas = areas[areas > float(min_size)]
            if max_size is not None:
                areas = areas[areas < float(max_size)]

            dico_algo_size[algo].extend(areas.astype(float).tolist())

        pbar.update(1)
    pbar.close()

    if all(len(v) == 0 for v in dico_algo_size.values()):
        raise RuntimeError("No object areas were collected after filtering. Check thresholds or masks.")

    # ---------------------- Prepare colors ----------------------
    tab20_distinct = [
        "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
        "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
        "#AEC7E8", "#FFBB78", "#98DF8A", "#FF9896", "#C5B0D5",
        "#9EDAE5", "#DBDB8D", "#F7B6D2", "#C7C7C7", "#C49C94",
    ]
    palette = tab20_distinct[: len(algos)] if len(algos) <= len(tab20_distinct) else sns.color_palette("husl", n_colors=len(algos))
    algo_to_color = {algo: palette[i] for i, algo in enumerate(algos)}

    # ---------------------- Plot ----------------------
    plt.figure(figsize=figsize)
    ax = plt.gca()

    subtitle = ""
    if (min_size is not None) or (max_size is not None):
        subtitle = f" (filtered"
        if min_size is not None:
            subtitle += f", min>{min_size}"
        if max_size is not None:
            subtitle += f", max<{max_size}"
        subtitle += ")"

    ax.set_title(f"Mask cells size distribution{subtitle}", fontsize=font_size)
    ax.set_xlabel("Size (pixels)", fontsize=font_size)
    ax.set_ylabel("Number of cells", fontsize=font_size)

    for algo in algos:
        areas = dico_algo_size[algo]
        if not areas:
            continue
        sns.histplot(
            areas,
            kde=True,
            element="step",
            stat="count",
            common_norm=False,
            bins=bins,
            ax=ax,
            color=algo_to_color[algo],
        )

    handles = [Patch(facecolor=algo_to_color[a], edgecolor="black", label=a) for a in algos]

    if len(algos) > 10:
        ncol = 2 if len(algos) <= 20 else 3
        ax.legend(
            handles=handles,
            title="Algorithms",
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
            fontsize=font_size,
            title_fontsize=font_size,
            ncol=ncol,
        )
        plt.tight_layout(rect=[0, 0, 0.80, 1])
    else:
        ax.legend(
            handles=handles,
            title="Algorithms",
            loc="upper right",
            frameon=True,
            fontsize=font_size,
            title_fontsize=font_size,
        )
        plt.tight_layout()

    out_path = path_out_dir / title
    plt.savefig(out_path, dpi=200)
    plt.close()

    print(f"✅ file created: {out_path}")

#### Execution

In [79]:
#size_distribution(path_mask=path_mask,path_file=path_eval_quali_without_truth,figsize=(8,8),font_size=16,bins=30)
size_distribution(
    path_mask=path_mask,
    path_file=path_eval_quali_without_truth,
    min_size=15,
    max_size=300,
    figsize=(8, 8),
    font_size=16,
    bins=30,
)

✅ file created: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/size_distribution.png


### 📊 **DNA signal outside the cell**
*To assess the quality of the segmentation, it can be useful to quantify the DNA outside the objects created by the algorithm. A large amount of poorly segmented DNA indicates objects that are too small or a number of missed cells. For this, it is helpful to compare graphs showing the ratios and images showing the DNA outside the objects.*

#### Function

In [36]:
import ipywidgets as widgets
from IPython.display import display

def widget_select_object(objects: list[str], description: str = "Objet:", default: str | None = None):
    """
    Create a dropdown widget to let the user choose an object from a list.

    Parameters
    ----------
    objects : list[str]
        List of object names (e.g., ["glomerule", "tubule", "vessel"]).
    description : str
        Label shown next to the widget.
    default : str | None
        Default selected object. If None, selects the first element.

    Returns
    -------
    dropdown : ipywidgets.Dropdown
    """
    if not objects:
        raise ValueError("objects list is empty.")

    if default is None or default not in objects:
        default = objects[0]

    dropdown = widgets.Dropdown(
        options=objects,
        value=default,
        description=description,
        layout=widgets.Layout(width="420px"),
        style={"description_width": "initial"},
    )
    display(dropdown)
    return dropdown


In [69]:

def percentage_in_out(
    list_marker=("DNA",),
    path_img="./images/raw_images",
    path_mask="./mask_roi",
    font_size=12,
    path_file="./"
):
    """
    Compute the percentage of marker intensity located inside segmentation masks.
    Produces:
        1) grouped bar chart per ROI
        2) boxplot + swarmplot per algorithm (color-coded)
           - legend outside (right)
           - mean value annotated above the max whisker
    """

    # ------------------------- Setup -------------------------
    os.makedirs(path_file, exist_ok=True)

    list_name_img = sorted([d.name for d in os.scandir(path_img) if d.is_dir()])
    algos = sorted([d.name for d in os.scandir(path_mask) if d.is_dir()])

    if not list_name_img:
        raise RuntimeError(f"No image folders found in '{path_img}'.")
    if not algos:
        raise RuntimeError(f"No algorithm folders found in '{path_mask}'.")

    dico_algo_in_out = {algo: [] for algo in algos}

    # ------------------------- Single progress bar -------------------------
    total_steps = len(algos) * len(list_name_img)

    with tqdm(total=total_steps, desc="Computing % in masks", unit="image", ncols=100) as pbar:
        for algo in algos:
            algo_dir = os.path.join(path_mask, algo)

            for name_img in list_name_img:
                mask_path = os.path.join(algo_dir, f"{name_img}.tif")
                if not os.path.isfile(mask_path):
                    dico_algo_in_out[algo].append(np.nan)
                    pbar.update(1)
                    continue

                mask = np.array(Image.open(mask_path))
                mask_in = mask > 0
                mask_out = ~mask_in

                sum_in = np.uint64(0)
                sum_out = np.uint64(0)

                for marker in list_marker:
                    img_path = os.path.join(path_img, name_img, marker)
                    if not os.path.isfile(img_path):
                        continue

                    img = np.array(Image.open(img_path))

                    # arcsinh + normalize_255 doivent exister dans ton environnement
                    img = normalize_255(np.arcsinh(img))

                    img_thr = img * (img > 150)

                    sum_in += img_thr[mask_in].sum(dtype=np.uint64)
                    sum_out += img_thr[mask_out].sum(dtype=np.uint64)

                denom = int(sum_in + sum_out)
                percent_in = (100 * sum_in / denom) if denom > 0 else 0.0
                dico_algo_in_out[algo].append(float(percent_in))

                pbar.update(1)

    # ------------------------- PLOT 1 : Grouped bar chart -------------------------
    n_imgs = len(list_name_img)
    for algo in algos:
        if len(dico_algo_in_out[algo]) < n_imgs:
            dico_algo_in_out[algo].extend([np.nan] * (n_imgs - len(dico_algo_in_out[algo])))

    plt.figure(figsize=(max(18, n_imgs * 0.7), 8))
    width = 0.8 / len(algos)
    x_base = np.arange(n_imgs)

    for i, algo in enumerate(algos):
        plt.bar(x_base + i * width, dico_algo_in_out[algo], width=width, label=algo)

    tick_pos = x_base + width * (len(algos) - 1) / 2
    plt.xticks(tick_pos, list_name_img, rotation=70, ha="right", fontsize=font_size)

    plt.ylabel("Percentage (%)", fontsize=font_size)
    plt.xlabel("Images", fontsize=font_size)
    plt.title("Percentage of DNA intensity inside masks", fontsize=font_size)
    plt.legend(fontsize=font_size)
    plt.tight_layout()
    plt.savefig(os.path.join(path_file, "percentage_dna_intensity_inside_cell_masks.png"), dpi=200, bbox_inches="tight")
    plt.close()

    # ------------------------- PLOT 2 : Box + swarm -------------------------
    rows = []
    for algo in algos:
        for val in dico_algo_in_out[algo]:
            if not np.isnan(val):
                rows.append({"Algorithm": algo, "Percentage": val})

    df_plot = pd.DataFrame(rows)
    if df_plot.empty:
        print("⚠️ No data available for boxplot.")
        print("✔️ Files saved in:", path_file)
        return dico_algo_in_out

    palette = sns.color_palette("tab10", len(algos))
    algo_to_color = {algo: palette[i] for i, algo in enumerate(algos)}

    plt.figure(figsize=(max(8, len(algos) * 1.6), 7))

    sns.boxplot(
        data=df_plot,
        x="Algorithm",
        y="Percentage",
        order=algos,
        palette=algo_to_color,
        showmeans=True,
        meanprops=dict(
            marker="D",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=6,
        ),
    )

    sns.swarmplot(
        data=df_plot,
        x="Algorithm",
        y="Percentage",
        order=algos,
        color="black",
        size=3,
        alpha=0.7,
    )
    """
    # Moyenne (pour le texte) et maximum (pour la position verticale)
    stats = df_plot.groupby("Algorithm")["Percentage"].agg(["mean", "max"])
    y_min, y_max = plt.ylim()
    offset = 0.02 * (y_max - y_min)

    for i, algo in enumerate(algos):
        mean_val = stats.loc[algo, "mean"]
        max_val = stats.loc[algo, "max"]
        plt.text(
            i,
            max_val + offset,          # position au-dessus du trait de max
            f"{mean_val:.1f}%",        # valeur affichée = moyenne
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )
    """
    plt.ylabel("Percentage (%)", fontsize=font_size)
    plt.xlabel("Algorithms", fontsize=font_size)
    plt.title("Percentage of DNA intensity inside cell masks", fontsize=font_size)
    plt.xticks(fontsize=font_size,rotation=30)
    plt.yticks(fontsize=font_size)
    # Légende à l'extérieur, sans gros blanc à droite
    handles = [
        Patch(facecolor=algo_to_color[algo], edgecolor="black", label=algo)
        for algo in algos
    ]
    """
    plt.legend(
        handles=handles,
        title="Algorithms",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0,
        fontsize=11,
        title_fontsize=12,
    )
    """
    plt.tight_layout()
    plt.savefig(os.path.join(path_file, "mean_DNA_intensity_inside_cell_mask.png"), dpi=200, bbox_inches="tight")
    plt.close()

    print("✔️ Files saved in:", path_file)


#### Plot of the DNA signal percentage outside of the cells

In [38]:
roi=os.listdir(path_img_raw)
list_feature=os.listdir(path_img_raw+roi[0])
dna_w = widget_select_object(list_feature, description="Choose the DNA marker :")

Dropdown(description='Choose the DNA marker :', layout=Layout(width='420px'), options=('MPO.tif', 'Ki67.tif', …

In [73]:
dna_marker=dna_w.value
percentage_in_out(list_marker=[dna_marker],path_img=path_img_raw,path_mask=path_mask,path_file=path_eval_quali_without_truth,font_size=16)

Computing % in masks:   0%|                                              | 0/445 [00:00<?, ?image/s]

/tmp/ipykernel_176/2677183000.py:114: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/usr/local/lib/python3.12/dist-packages/seaborn/categorical.py:3399: UserWarning: 53.9% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.12/dist-packages/seaborn/categorical.py:3399: UserWarning: 42.7% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.12/dist-packages/seaborn/categorical.py:3399: UserWarning: 55.1% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.12/dist-packages/seaborn/categorical.py:3399: UserWarning: 46.1% of the points

✔️ Files saved in: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/


#### Display the Dna signal outside of the cells

##### Functions

In [40]:
import os
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm.auto import tqdm


def export_dna_mask_qc(
    *,
    path_img_raw: str,
    path_mask: str,
    path_eval_mask_dna: str,
    dna_marker: str,
    threshold: int = 150,
    out_ext: str = ".tif",
) -> None:
    """
    For each algorithm subfolder in `path_mask`, and for each mask file:
      - loads the instance mask
      - loads the corresponding DNA marker image from:
            path_img_raw/<biopsy_name>/<dna_marker>
        where biopsy_name is derived from mask filename (file_mask[:-4])
      - applies arcsinh + normalize_255 + threshold
      - builds a binary "outside-mask" QC image (255 outside mask, 0 elsewhere)
      - saves the QC image into:
            path_eval_mask_dna/<algo>/<biopsy_name>.tif

    Notes:
      - Assumes mask filenames end with a 3-letter extension (e.g. .png/.tif). If not, adapt stem extraction.
      - Requires an existing function `normalize_255`.
    """

    path_img_raw = Path(path_img_raw)
    path_mask = Path(path_mask)
    path_eval_mask_dna = Path(path_eval_mask_dna)

    path_eval_mask_dna.mkdir(parents=True, exist_ok=True)

    algos = sorted([d.name for d in path_mask.iterdir() if d.is_dir()])
    if not algos:
        raise RuntimeError(f"No algorithm folders found in: {path_mask}")

    # Build a flat list of jobs for a single clean progress bar
    jobs = []
    for algo in algos:
        algo_dir = path_mask / algo
        mask_files = sorted([p for p in algo_dir.iterdir() if p.is_file()])
        for mask_path in mask_files:
            jobs.append((algo, mask_path))

    if not jobs:
        raise RuntimeError(f"No mask files found under: {path_mask}")

    pbar = tqdm(total=len(jobs), desc="DNA mask QC export", unit="mask", ncols=110)

    for algo, mask_path in jobs:
        algo_out_dir = path_eval_mask_dna / algo
        algo_out_dir.mkdir(parents=True, exist_ok=True)

        biopsy_name = mask_path.stem  # safer than file_mask[:-4]
        dna_path = path_img_raw / biopsy_name / dna_marker

        pbar.set_postfix_str(f"{algo} | {mask_path.name}", refresh=False)

        try:
            mask = np.array(Image.open(mask_path))
            img = np.array(Image.open(dna_path))

            img = normalize_255(np.arcsinh(img))
            img = np.where(img < int(threshold), 0, img)

            # These two are computed but only img_out is saved in your original code
            # img_in  = np.where(mask > 0, img, 0)
            img_out = np.where(mask == 0, img, 0)

            img_out = np.where(img_out > 0, 255, 0).astype(np.uint8)

            out_path = algo_out_dir / f"{biopsy_name}{out_ext}"
            Image.fromarray(img_out).save(out_path)

        except Exception as e:
            # keep going but surface the error
            pbar.write(f"[WARN] Failed algo={algo} file={mask_path.name}: {e}")

        pbar.update(1)

    pbar.close()
    print(f"✅ files available : {path_eval_mask_dna}")

##### Execution

In [41]:
path_eval_mask_dna=path_eval_quali_without_truth+"mask_dna_outside/"
if os.path.isdir(path_eval_mask_dna)==False:
  os.mkdir(path_eval_mask_dna)


In [42]:
dna_marker=dna_w.value
export_dna_mask_qc(
    path_img_raw=path_img_raw,
    path_mask=path_mask,
    path_eval_mask_dna=path_eval_mask_dna,
    dna_marker=dna_marker
)

DNA mask QC export:   0%|                                                           | 0/445 [00:00<?, ?mask/s]

✅ files available : /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/mask_dna_outside


In [43]:
list_name_img=os.listdir(path_img_raw)
for algo in os.listdir(path_mask):
  if os.path.isdir(path_eval_mask_dna+algo)==False:
    os.mkdir(path_eval_mask_dna+algo)
  for file_mask in os.listdir(path_mask+"/"+algo):
            mask= np.array(Image.open(path_mask+"/"+algo+"/"+file_mask))
            img= np.array(Image.open(path_img_raw+file_mask[:-4]+"/"+dna_marker))
            img=normalize_255(np.arcsinh(img))
            img=np.where(img<150,0,img)
            img_in=np.where(mask>0,img,0)
            img_out=np.where(mask==0,img,0)
            img_out=np.where(img_out>0,255,0)
            img_in=np.where(img_in>0,255,0)
            im = Image.fromarray(img_out.astype(np.uint8))
            im.save(path_eval_mask_dna+algo+"/"+file_mask[:-4]+".tif")
print("✅ files available : "+path_eval_mask_dna)


KeyboardInterrupt: 

### 📊 **Outline the mask**
*This cell allows you to create an image of the biopsy with several markers of your choice in multiple colors, with the outline of the objects created by the algorithms.*

#### Function

In [80]:
def instance_boundaries(mask):
    m = mask
    b = np.zeros_like(m, dtype=bool)
    b[1:, :] |= (m[1:, :] != m[:-1, :]) & (m[1:, :] != 0) & (m[:-1, :] != 0)
    b[:, 1:] |= (m[:, 1:] != m[:, :-1]) & (m[:, 1:] != 0) & (m[:, :-1] != 0)
    # On inclut aussi les frontières fg/bg
    b[1:, :] |= (m[1:, :] != m[:-1, :]) & ((m[1:, :] == 0) | (m[:-1, :] == 0))
    b[:, 1:] |= (m[:, 1:] != m[:, :-1]) & ((m[:, 1:] == 0) | (m[:, :-1] == 0))
    return b

def outline_fast_instances(img_rgb, mask, color=(255,255,255), w=1):
    b = instance_boundaries(mask)
    out = img_rgb.copy()
    out[b] = color
    if w > 1:
        # épaissir les bords
        b8 = b.astype(np.uint8)*255
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*w+1, 2*w+1))
        b8 = cv2.dilate(b8, k)
        out[b8 > 0] = color
    return out


In [81]:
def filter_mask_by_area(mask, min_area=None, max_area=None):
    m = mask.astype(np.int64)
    counts = np.bincount(m.ravel())
    keep = np.ones(len(counts), dtype=bool)
    keep[0] = False
    if min_area is not None:
        keep &= (counts >= min_area)
    if max_area is not None:
        keep &= (counts <= max_area)

    # supprime labels non gardés
    out = m.copy()
    out[~keep[out]] = 0
    return out.astype(mask.dtype)


In [82]:
from tqdm.auto import tqdm
import os
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

def display_outline_mask_fast(path_img, path_mask, marker,dna_marker, path="./", dico_algo_size=None, w=1):
    algos = sorted([d for d in os.listdir(path_mask) if os.path.isdir(os.path.join(path_mask, d))])
    images = sorted([d for d in os.listdir(path_img) if os.path.isdir(os.path.join(path_img, d))])

    for algo in tqdm(algos, desc="Algorithms", unit="alg"):
        out_dir = os.path.join(path, algo)
        os.makedirs(out_dir, exist_ok=True)

        algo_dir = os.path.join(path_mask, algo)

        for name_img in tqdm(images, desc=algo, unit="img", leave=False):
            dna_path = os.path.join(path_img, name_img, dna_marker+".png")
            mask_path = os.path.join(algo_dir, f"{name_img}.tif")

            if not (os.path.isfile(dna_path) and os.path.isfile(mask_path)):
                continue

            # DNA
            img_dna = np.array(Image.open(dna_path))
            #img_dna = normalize_255(np.arcsinh(img_dna * 10))
            #img_dna = np.where(img_dna < 150, 0, img_dna)

            H, W = img_dna.shape[:2]
            img_rgb = np.zeros((H, W, 3), dtype=np.uint8)
            img_rgb[:, :, 1] = img_dna.astype(np.uint8)

            # Marker optionnel
            if marker:
                marker_path = os.path.join(path_img, name_img, f"{marker}.tif")
                if not os.path.isfile(marker_path):
                    marker_path = os.path.join(path_img, name_img, f"{marker}.tiff")
                if os.path.isfile(marker_path):
                    img_marker = np.array(Image.open(marker_path))
                    img_rgb[:, :, 1] = normalize_255(img_marker).astype(np.uint8)

            # Mask
            mask = np.array(Image.open(mask_path))

            # Filtrage taille si demandé
            if dico_algo_size is not None and algo in dico_algo_size:
                mn = dico_algo_size[algo].get("min", None)
                mx = dico_algo_size[algo].get("max", None)
                mask = filter_mask_by_area(mask, mn, mx)

            # Contours (rapide)
            img_rgb = outline_fast_instances(img_rgb, mask, color=(255,255,255), w=w)

            out_path = os.path.join(out_dir, f"{name_img}.png")
            cv2.imwrite(out_path, img_rgb[:, :, ::-1])  # cv2 = BGR
    print("✅ files created : " + path)


In [83]:
from tqdm.auto import tqdm
import os
import numpy as np
import cv2
from PIL import Image
import tifffile as tiff

import ipywidgets as widgets
from IPython.display import display

# -------------------------
# Helpers robustes
# -------------------------
IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
MASK_EXTS = (".tif", ".tiff", ".png")

def find_channel_file(folder, channel_name):
    """Find folder/channel_name.<any supported ext>"""
    for ext in IMG_EXTS:
        p = os.path.join(folder, f"{channel_name}{ext}")
        if os.path.isfile(p):
            return p
    return None

def robust_read_image(path):
    """Robust image read (TIFF via tifffile, else PIL)."""
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext in (".tif", ".tiff"):
            return tiff.imread(path)
        return np.array(Image.open(path))
    except Exception:
        return None

def to_gray_float(arr):
    """Convert array to grayscale float32 (no scaling)."""
    if arr is None:
        return None
    arr = np.asarray(arr)

    # stack (Z,H,W) -> slice 0
    if arr.ndim == 3 and arr.shape[-1] not in (3, 4):
        arr = arr[0]

    # RGB/RGBA -> gray mean
    if arr.ndim == 3 and arr.shape[-1] in (3, 4):
        arr = arr[..., :3].astype(np.float32).mean(axis=-1)

    return arr.astype(np.float32)

def robust_norm01(arr, p_low=1.0, p_high=99.8, eps=1e-8):
    """Percentile normalization to [0,1]."""
    if arr is None:
        return None
    lo, hi = np.percentile(arr, [p_low, p_high])
    if hi <= lo + eps:
        return np.zeros_like(arr, dtype=np.float32)
    x = np.clip(arr, lo, hi)
    x = (x - lo) / (hi - lo)
    return x.astype(np.float32)

def to_label_mask_2d(mask_arr):
    """Convert loaded mask to 2D int64 labels."""
    if mask_arr is None:
        return None
    m = np.asarray(mask_arr)

    # (Z,H,W) -> 0
    if m.ndim == 3 and m.shape[-1] not in (3, 4):
        m = m[0]

    # (H,W,C) -> 0
    if m.ndim == 3 and m.shape[-1] >= 1:
        m = m[..., 0]

    if np.issubdtype(m.dtype, np.floating):
        m = np.rint(m).astype(np.int64)
    else:
        m = m.astype(np.int64)

    return m

def match_size_2d(arr, target_hw):
    """Crop/pad to target (H,W) without interpolation."""
    Ht, Wt = target_hw
    H, W = arr.shape[:2]
    arr2 = arr[:min(H, Ht), :min(W, Wt)]
    out = np.zeros((Ht, Wt), dtype=arr2.dtype)
    out[:arr2.shape[0], :arr2.shape[1]] = arr2
    return out

def find_mask_for_roi(algo_dir, roi_name):
    """Find <roi>.<tif/tiff/png> in algo_dir."""
    for ext in MASK_EXTS:
        p = os.path.join(algo_dir, f"{roi_name}{ext}")
        if os.path.isfile(p):
            return p
    return None

# ======================================================
# UI Colab: select markers + pick colors (nuancier cliquable)
# ======================================================
def interactive_choose_markers_and_colors(list_all_markers, title="🎨 Select markers & colors"):
    """
    SelectMultiple markers + a color picker row per selected marker.
    Returns dict:
      state["selected_markers"] = [...]
      state["marker_to_hex"] = {marker:"#RRGGBB", ...}
    """
    list_all_markers = sorted({str(m).strip() for m in list_all_markers if str(m).strip()})

    header = widgets.HTML(f"<h3 style='margin:0 0 10px 0;'>{title}</h3>")

    marker_selector = widgets.SelectMultiple(
        options=list_all_markers,
        description="Markers:",
        rows=min(14, len(list_all_markers)),
        layout=widgets.Layout(width="520px", height="260px")
    )

    btn_build = widgets.Button(description="🎨 Create color panel", button_style="info")
    btn_confirm = widgets.Button(description="✅ Confirm selection", button_style="success")
    out = widgets.Output()

    default_palette = [
        "#e60000", "#00f504", "#0068b3", "#fff700", "#861f93", "#ffa200",
        "#00ffff", "#ff00c8", "#1000f0", "#f6fefd", "#03e212", "#aaaaaa"
    ]

    state = {"selected_markers": [], "marker_to_hex": {}}
    pickers = {}
    panel_box = widgets.VBox([])

    def make_row(marker, default_color):
        name = widgets.HTML(f"<b style='font-size:14px'>{marker}</b>",
                            layout=widgets.Layout(width="220px"))
        cp = widgets.ColorPicker(
            value=default_color,
            concise=False,
            layout=widgets.Layout(width="220px")
        )
        pickers[marker] = cp
        return widgets.HBox([name, cp], layout=widgets.Layout(gap="12px", align_items="center"))

    def on_build(_):
        sels = list(marker_selector.value)
        with out:
            out.clear_output(wait=True)
            if len(sels) == 0:
                print("Select at least one marker first.")
                panel_box.children = []
                return

            state["selected_markers"] = sels
            rows = []
            pickers.clear()

            for i, m in enumerate(sels):
                c0 = state["marker_to_hex"].get(m, default_palette[i % len(default_palette)])
                rows.append(make_row(m, c0))

            panel_box.children = rows
            print("Click the color squares to pick colors, then confirm.")

    def on_confirm(_):
        sels = list(marker_selector.value)
        if len(sels) == 0:
            with out:
                out.clear_output(wait=True)
                print("Select at least one marker, then build the panel.")
            return

        if not pickers or any(m not in pickers for m in sels):
            on_build(None)

        state["selected_markers"] = sels
        state["marker_to_hex"] = {m: pickers[m].value for m in sels}

        with out:
            out.clear_output(wait=True)
            print("✔ Selected markers:", state["selected_markers"])
            print("✔ marker_to_hex:", state["marker_to_hex"])

    btn_build.on_click(on_build)
    btn_confirm.on_click(on_confirm)

    display(header, marker_selector, widgets.HBox([btn_build, btn_confirm]), panel_box, out)
    return state

def hex_to_rgb255(hex_color):
    h = str(hex_color).strip()
    if h.startswith("#"):
        h = h[1:]
    if len(h) != 6:
        raise ValueError(f"Invalid hex color: {hex_color}")
    return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16))

# ======================================================
# Build a full composite (multi-markers) for one ROI
# ======================================================
def build_full_composite_rgb(
    roi_dir,
    selected_markers,
    marker_to_hex,
    p_low=1.0,
    p_high=99.8,
    gamma=1.0,
    weight=1.0
):
    """
    Returns uint8 RGB composite (H,W,3) by summing each marker as:
      acc += norm01(marker)*color
    """
    acc = None
    H = W = None

    for m in selected_markers:
        p = find_channel_file(roi_dir, m)
        if p is None:
            continue

        raw = robust_read_image(p)
        gray = to_gray_float(raw)
        if gray is None:
            continue

        if H is None:
            H, W = gray.shape[:2]
            acc = np.zeros((H, W, 3), dtype=np.float32)

        if gray.shape[:2] != (H, W):
            # ignore mismatched sizes
            continue

        img01 = robust_norm01(gray, p_low=p_low, p_high=p_high)
        if gamma != 1.0:
            img01 = np.power(img01, float(gamma))

        r, g, b = hex_to_rgb255(marker_to_hex.get(m, "#ffffff"))
        color01 = np.array([r, g, b], dtype=np.float32) / 255.0

        acc += (img01[..., None] * color01[None, None, :]) * float(weight)

    if acc is None:
        return None

    acc = np.clip(acc, 0, 1)
    return (acc * 255).astype(np.uint8)

# ======================================================
# MAIN: draw outlines on the full biopsy composite (no sampling)
# ======================================================
def display_outline_mask_full_composite(
    path_img,
    path_mask,
    output_dir,
    selected_markers,
    marker_to_hex,
    algos=None,
    dico_algo_size=None,   # kept for compatibility; requires your filter_mask_by_area
    w=1,
    contour_color=(255, 255, 255),
    p_low=1.0,
    p_high=99.8,
    gamma=1.0,
    weight=1.0
):
    """
    For each algo and ROI:
      1) build a full colored composite from selected markers
      2) load the ROI mask
      3) draw contours on the composite
      4) save PNG in output_dir/<algo>/<ROI>.png
    """
    os.makedirs(output_dir, exist_ok=True)

    # algos
    if algos is None:
        algos = sorted([d for d in os.listdir(path_mask) if os.path.isdir(os.path.join(path_mask, d))])
    else:
        algos = list(algos)

    rois = sorted([d for d in os.listdir(path_img) if os.path.isdir(os.path.join(path_img, d))])

    for algo in tqdm(algos, desc="Algorithms", unit="alg"):
        algo_dir = os.path.join(path_mask, algo)
        out_algo = os.path.join(output_dir, algo)
        os.makedirs(out_algo, exist_ok=True)

        for roi in tqdm(rois, desc=algo, unit="ROI", leave=False):
            roi_dir = os.path.join(path_img, roi)

            # Build composite
            composite = build_full_composite_rgb(
                roi_dir=roi_dir,
                selected_markers=selected_markers,
                marker_to_hex=marker_to_hex,
                p_low=p_low,
                p_high=p_high,
                gamma=gamma,
                weight=weight
            )
            if composite is None:
                continue

            # Mask
            mask_path = find_mask_for_roi(algo_dir, roi)
            if mask_path is None:
                continue

            mask_raw = robust_read_image(mask_path)
            mask = to_label_mask_2d(mask_raw)
            if mask is None:
                continue

            H, W = composite.shape[:2]
            if mask.shape[:2] != (H, W):
                mask = match_size_2d(mask, (H, W))

            # Optional size filtering (if you have filter_mask_by_area defined elsewhere)
            if dico_algo_size is not None and algo in dico_algo_size:
                mn = dico_algo_size[algo].get("min", None)
                mx = dico_algo_size[algo].get("max", None)
                mask = filter_mask_by_area(mask, mn, mx)

            # Draw contours
            out_img = outline_fast_instances(composite.copy(), mask, color=contour_color, w=w)

            out_path = os.path.join(out_algo, f"{roi}.png")
            cv2.imwrite(out_path, out_img[:, :, ::-1])  # cv2 expects BGR

    print(f"✅ files created in: {output_dir}")


#### Execution

In [50]:
processing=input("Enter the image processing you want (1 or 2): ")

Enter the image processing you want (1 or 2): 2


In [51]:
path_img_processing_biopsies=path+"images/img_processing_"+processing+"/Biopsies/"

In [84]:
# 1) Build available markers from one ROI folder (example)
roi0 = sorted([d for d in os.listdir(path_img_processing_biopsies) if os.path.isdir(os.path.join(path_img_processing_biopsies, d))])[0]
list_all_markers = sorted({os.path.splitext(f)[0] for f in os.listdir(os.path.join(path_img_processing_biopsies, roi0))
                            if os.path.splitext(f)[1].lower() in IMG_EXTS})
#
# 2) UI: pick markers + colors
state = interactive_choose_markers_and_colors(list_all_markers)
selected_markers = state["selected_markers"]
marker_to_hex = state["marker_to_hex"]

HTML(value="<h3 style='margin:0 0 10px 0;'>🎨 Select markers & colors</h3>")

SelectMultiple(description='Markers:', layout=Layout(height='260px', width='520px'), options=('Aquaporin1', 'C…

VBox()

Output()

In [85]:
roi0 = sorted([d for d in os.listdir(path_img_processing_biopsies) if os.path.isdir(os.path.join(path_img_processing_biopsies, d))])[0]
list_all_markers = sorted({os.path.splitext(f)[0] for f in os.listdir(os.path.join(path_img_processing_biopsies, roi0))
                            if os.path.splitext(f)[1].lower() in IMG_EXTS})

selected_markers = state["selected_markers"]
marker_to_hex = state["marker_to_hex"]

display_outline_mask_full_composite(
     path_img=path_img_processing_biopsies,
     path_mask=path_mask,
     output_dir=path_eval_mask_outline,
     selected_markers=selected_markers,
     marker_to_hex=marker_to_hex,
     w=1
 )


Algorithms:   0%|          | 0/5 [00:00<?, ?alg/s]

cellposev3:   0%|          | 0/89 [00:00<?, ?ROI/s]

combine_mesmer_cellposev3:   0%|          | 0/89 [00:00<?, ?ROI/s]

instanseg:   0%|          | 0/89 [00:00<?, ?ROI/s]

mesmer:   0%|          | 0/89 [00:00<?, ?ROI/s]

stardist:   0%|          | 0/89 [00:00<?, ?ROI/s]

✅ files created in: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/mask_outline/


### 📊 **Outline the mask of crops**
*This code interactively lets the user select multiple markers and colors, builds a colored composite image from those markers, and generates side-by-side sampled patches showing the composite image with and without cell segmentation contours to visually assess segmentation quality on large biopsies.*

#### Function

In [56]:
def instance_boundaries(mask):
    m = mask
    b = np.zeros_like(m, dtype=bool)
    b[1:, :] |= (m[1:, :] != m[:-1, :]) & (m[1:, :] != 0) & (m[:-1, :] != 0)
    b[:, 1:] |= (m[:, 1:] != m[:, :-1]) & (m[:, 1:] != 0) & (m[:, :-1] != 0)
    # On inclut aussi les frontières fg/bg
    b[1:, :] |= (m[1:, :] != m[:-1, :]) & ((m[1:, :] == 0) | (m[:-1, :] == 0))
    b[:, 1:] |= (m[:, 1:] != m[:, :-1]) & ((m[:, 1:] == 0) | (m[:, :-1] == 0))
    return b

def outline_fast_instances(img_rgb, mask, color=(255,255,255), w=1):
    b = instance_boundaries(mask)
    out = img_rgb.copy()
    out[b] = color
    if w > 1:
        # épaissir les bords
        b8 = b.astype(np.uint8)*255
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*w+1, 2*w+1))
        b8 = cv2.dilate(b8, k)
        out[b8 > 0] = color
    return out


In [57]:
def filter_mask_by_area(mask, min_area=None, max_area=None):
    m = mask.astype(np.int64)
    counts = np.bincount(m.ravel())
    keep = np.ones(len(counts), dtype=bool)
    keep[0] = False
    if min_area is not None:
        keep &= (counts >= min_area)
    if max_area is not None:
        keep &= (counts <= max_area)

    # supprime labels non gardés
    out = m.copy()
    out[~keep[out]] = 0
    return out.astype(mask.dtype)


In [58]:
from tqdm.auto import tqdm
import os
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

def display_outline_mask_fast(path_img, path_mask, marker,dna_marker, path="./", dico_algo_size=None, w=1):
    algos = sorted([d for d in os.listdir(path_mask) if os.path.isdir(os.path.join(path_mask, d))])
    images = sorted([d for d in os.listdir(path_img) if os.path.isdir(os.path.join(path_img, d))])

    for algo in tqdm(algos, desc="Algorithms", unit="alg"):
        out_dir = os.path.join(path, algo)
        os.makedirs(out_dir, exist_ok=True)

        algo_dir = os.path.join(path_mask, algo)

        for name_img in tqdm(images, desc=algo, unit="img", leave=False):
            dna_path = os.path.join(path_img, name_img, dna_marker+".png")
            mask_path = os.path.join(algo_dir, f"{name_img}.tif")

            if not (os.path.isfile(dna_path) and os.path.isfile(mask_path)):
                continue

            # DNA
            img_dna = np.array(Image.open(dna_path))
            #img_dna = normalize_255(np.arcsinh(img_dna * 10))
            #img_dna = np.where(img_dna < 150, 0, img_dna)

            H, W = img_dna.shape[:2]
            img_rgb = np.zeros((H, W, 3), dtype=np.uint8)
            img_rgb[:, :, 1] = img_dna.astype(np.uint8)

            # Marker optionnel
            if marker:
                marker_path = os.path.join(path_img, name_img, f"{marker}.tif")
                if not os.path.isfile(marker_path):
                    marker_path = os.path.join(path_img, name_img, f"{marker}.tiff")
                if os.path.isfile(marker_path):
                    img_marker = np.array(Image.open(marker_path))
                    img_rgb[:, :, 1] = normalize_255(img_marker).astype(np.uint8)

            # Mask
            mask = np.array(Image.open(mask_path))

            # Filtrage taille si demandé
            if dico_algo_size is not None and algo in dico_algo_size:
                mn = dico_algo_size[algo].get("min", None)
                mx = dico_algo_size[algo].get("max", None)
                mask = filter_mask_by_area(mask, mn, mx)

            # Contours (rapide)
            img_rgb = outline_fast_instances(img_rgb, mask, color=(255,255,255), w=w)

            out_path = os.path.join(out_dir, f"{name_img}.png")
            cv2.imwrite(out_path, img_rgb[:, :, ::-1])  # cv2 = BGR
    print("✅ files created : " + path)


In [59]:

from tqdm.auto import tqdm
import os
import numpy as np
import cv2
from PIL import Image
import tifffile as tiff

import ipywidgets as widgets
from IPython.display import display

# ======================================================
# Helpers robustes (lecture + conversion)
# ======================================================
IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

def find_channel_file(folder, channel_name):
    for ext in IMG_EXTS:
        p = os.path.join(folder, f"{channel_name}{ext}")
        if os.path.isfile(p):
            return p
    return None

def robust_read_image(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext in (".tif", ".tiff"):
            return tiff.imread(path)
        return np.array(Image.open(path))
    except Exception:
        return None

def to_gray_float(arr):
    """
    Convertit arr en grayscale float32 (sans normaliser).
    """
    if arr is None:
        return None
    arr = np.asarray(arr)

    # (Z,H,W) -> slice 0
    if arr.ndim == 3 and arr.shape[-1] not in (3, 4):
        arr = arr[0]

    # RGB/RGBA -> gray mean
    if arr.ndim == 3 and arr.shape[-1] in (3, 4):
        arr = arr[..., :3].astype(np.float32).mean(axis=-1)

    arr = arr.astype(np.float32)
    return arr

def robust_norm01(arr, p_low=1.0, p_high=99.8, eps=1e-8):
    """
    Normalisation robuste en [0,1] via percentiles (anti-voile gris).
    """
    if arr is None:
        return None
    lo, hi = np.percentile(arr, [p_low, p_high])
    if hi <= lo + eps:
        return np.zeros_like(arr, dtype=np.float32)
    x = np.clip(arr, lo, hi)
    x = (x - lo) / (hi - lo)
    return x.astype(np.float32)

def to_label_mask_2d(mask_arr):
    if mask_arr is None:
        return None
    m = np.asarray(mask_arr)

    # (Z,H,W) -> 0
    if m.ndim == 3 and m.shape[-1] not in (3, 4):
        m = m[0]

    # (H,W,C) -> 0
    if m.ndim == 3 and m.shape[-1] >= 1:
        m = m[..., 0]

    if np.issubdtype(m.dtype, np.floating):
        m = np.rint(m).astype(np.int64)
    else:
        m = m.astype(np.int64)

    return m

def match_size_2d(arr, target_hw):
    Ht, Wt = target_hw
    H, W = arr.shape[:2]
    arr2 = arr[:min(H, Ht), :min(W, Wt)]
    out = np.zeros((Ht, Wt), dtype=arr2.dtype)
    out[:arr2.shape[0], :arr2.shape[1]] = arr2
    return out


# ======================================================
# Sampling
# ======================================================
def sample_patch_coords(H, W, patch_size, n_samples, rng):
    ps = int(patch_size)
    if ps <= 0:
        raise ValueError("patch_size must be > 0")
    if ps > H or ps > W:
        ps = min(H, W)

    coords = []
    for _ in range(int(n_samples)):
        y0 = int(rng.integers(0, max(1, H - ps + 1)))
        x0 = int(rng.integers(0, max(1, W - ps + 1)))
        coords.append((y0, y0 + ps, x0, x0 + ps))
    return coords

# ======================================================
# UI Colab: choisir marqueurs + couleurs (nuancier)
# ======================================================
def interactive_choose_markers_and_colors(list_all_markers, title="🎨 Select markers & colors"):
    """
    1) SelectMultiple des marqueurs
    2) 'Create color panel' -> une ligne par marqueur, avec ColorPicker cliquable (nuancier)
    3) 'Confirm' -> state['selected_markers'], state['marker_to_hex']
    """
    list_all_markers = sorted({str(m).strip() for m in list_all_markers if str(m).strip()})

    header = widgets.HTML(f"<h3 style='margin:0 0 10px 0;'>{title}</h3>")

    marker_selector = widgets.SelectMultiple(
        options=list_all_markers,
        description="Markers:",
        rows=min(14, len(list_all_markers)),
        layout=widgets.Layout(width="520px", height="260px")
    )

    btn_build = widgets.Button(description="🎨 Create color panel", button_style="info")
    btn_confirm = widgets.Button(description="✅ Confirm selection", button_style="success")
    out = widgets.Output()

    default_palette = [
        "#e60000", "#00f504", "#0068b3", "#fff700", "#861f93", "#ffa200",
        "#00ffff", "#ff00c8", "#1000f0", "#f6fefd", "#03e212", "#aaaaaa"
    ]

    state = {"selected_markers": [], "marker_to_hex": {}}
    pickers = {}
    panel_box = widgets.VBox([])

    def make_row(marker, default_color):
        name = widgets.HTML(f"<b style='font-size:14px'>{marker}</b>",
                            layout=widgets.Layout(width="220px"))
        cp = widgets.ColorPicker(
            value=default_color,
            concise=False,
            layout=widgets.Layout(width="220px")
        )
        pickers[marker] = cp
        return widgets.HBox([name, cp], layout=widgets.Layout(gap="12px", align_items="center"))

    def on_build(_):
        sels = list(marker_selector.value)
        with out:
            out.clear_output(wait=True)
            if len(sels) == 0:
                print("Select at least one marker first.")
                panel_box.children = []
                return

            state["selected_markers"] = sels
            rows = []
            pickers.clear()

            for i, m in enumerate(sels):
                c0 = state["marker_to_hex"].get(m, default_palette[i % len(default_palette)])
                rows.append(make_row(m, c0))

            panel_box.children = rows
            print("Click the color squares to pick colors, then confirm.")

    def on_confirm(_):
        sels = list(marker_selector.value)
        if len(sels) == 0:
            with out:
                out.clear_output(wait=True)
                print("Select at least one marker, then build the panel.")
            return

        if not pickers or any(m not in pickers for m in sels):
            on_build(None)

        state["selected_markers"] = sels
        state["marker_to_hex"] = {m: pickers[m].value for m in sels}

        with out:
            out.clear_output(wait=True)
            print("✔ Selected markers:", state["selected_markers"])
            print("✔ marker_to_hex:", state["marker_to_hex"])

    btn_build.on_click(on_build)
    btn_confirm.on_click(on_confirm)

    display(header, marker_selector, widgets.HBox([btn_build, btn_confirm]), panel_box, out)
    return state

# ======================================================
# Composite RGB from many markers (colored sum)
# ======================================================
def hex_to_rgb255(hex_color):
    h = str(hex_color).strip()
    if h.startswith("#"):
        h = h[1:]
    if len(h) != 6:
        raise ValueError(f"Invalid hex color: {hex_color}")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return (r, g, b)

def make_colored_composite(roi_dir, selected_markers, marker_to_hex,
                           p_low=1.0, p_high=99.8, gamma=1.0, weight=1.0):
    """
    Construit un composite RGB float [0,1] en sommant les contributions colorées de chaque marqueur.
    """
    acc = None
    H = W = None

    for m in selected_markers:
        p = find_channel_file(roi_dir, m)
        if p is None:
            continue

        raw = robust_read_image(p)
        gray = to_gray_float(raw)
        if gray is None:
            continue

        if H is None:
            H, W = gray.shape[:2]
            acc = np.zeros((H, W, 3), dtype=np.float32)

        if gray.shape[:2] != (H, W):
            # on ignore les tailles incohérentes pour éviter erreurs
            continue

        img01 = robust_norm01(gray, p_low=p_low, p_high=p_high)
        if gamma != 1.0:
            img01 = np.power(img01, float(gamma))

        r, g, b = hex_to_rgb255(marker_to_hex.get(m, "#ffffff"))
        color01 = np.array([r, g, b], dtype=np.float32) / 255.0

        acc += (img01[..., None] * color01[None, None, :]) * float(weight)

    if acc is None:
        return None

    acc = np.clip(acc, 0, 1)
    rgb_u8 = (acc * 255).astype(np.uint8)
    return rgb_u8

# ======================================================
# Génération d'échantillons: composite multi-marqueurs + contours
# ======================================================
def generate_segmentation_samples_multimarker(
    path_img,
    path_mask,
    output_dir,
    selected_markers,
    marker_to_hex,
    algos=None,
    n_samples=6,
    patch_size=512,
    seed=0,
    contour_color=(255, 255, 255),
    w=1,
    spacer_width=18,
    spacer_color=255,
    prefer_informative=True,
    min_mask_pixels=500,
    p_low=1.0,
    p_high=99.8,
    gamma=1.0,
    weight=1.0
):
    """
    Pour évaluer la qualité de segmentation:
    - Left  = composite multi-marqueurs coloré
    - Right = même composite + contours de la segmentation
    - N patches par ROI et par algo
    """
    os.makedirs(output_dir, exist_ok=True)
    rng = np.random.default_rng(seed)

    if algos is None:
        algos = sorted([d for d in os.listdir(path_mask) if os.path.isdir(os.path.join(path_mask, d))])
    else:
        algos = list(algos)

    rois = sorted([d for d in os.listdir(path_img) if os.path.isdir(os.path.join(path_img, d))])

    for algo in tqdm(algos, desc="Algorithms", unit="alg"):
        algo_dir = os.path.join(path_mask, algo)
        out_algo = os.path.join(output_dir, algo)
        os.makedirs(out_algo, exist_ok=True)

        for roi in tqdm(rois, desc=algo, unit="ROI", leave=False):
            roi_dir = os.path.join(path_img, roi)

            # Mask file
            mask_path = None
            for ext in (".tif", ".tiff", ".png"):
                p = os.path.join(algo_dir, f"{roi}{ext}")
                if os.path.isfile(p):
                    mask_path = p
                    break
            if mask_path is None:
                continue

            # Build composite (full ROI) once
            composite = make_colored_composite(
                roi_dir=roi_dir,
                selected_markers=selected_markers,
                marker_to_hex=marker_to_hex,
                p_low=p_low,
                p_high=p_high,
                gamma=gamma,
                weight=weight
            )
            if composite is None:
                continue

            # Load mask
            mask_raw = robust_read_image(mask_path)
            mask = to_label_mask_2d(mask_raw)
            if mask is None:
                continue

            H, W = composite.shape[:2]
            if mask.shape[:2] != (H, W):
                mask = match_size_2d(mask, (H, W))

            # coords
            if prefer_informative:
                coords = []
                candidates = sample_patch_coords(H, W, patch_size, n_samples * 12, rng)
                for (y0, y1, x0, x1) in candidates:
                    if int((mask[y0:y1, x0:x1] > 0).sum()) >= int(min_mask_pixels):
                        coords.append((y0, y1, x0, x1))
                    if len(coords) >= n_samples:
                        break
                if len(coords) < n_samples:
                    coords += sample_patch_coords(H, W, patch_size, n_samples - len(coords), rng)
            else:
                coords = sample_patch_coords(H, W, patch_size, n_samples, rng)

            for k, (y0, y1, x0, x1) in enumerate(coords, start=1):
                left = composite[y0:y1, x0:x1].copy()
                mask_patch = mask[y0:y1, x0:x1]

                right = left.copy()
                right = outline_fast_instances(right, mask_patch, color=contour_color, w=w)

                spacer = np.full((left.shape[0], int(spacer_width), 3), int(spacer_color), dtype=np.uint8)
                side = np.concatenate([left, spacer, right], axis=1)

                out_path = os.path.join(out_algo, f"{roi}__sample{k}_y{y0}_x{x0}.png")
                Image.fromarray(side).save(out_path)

    print(f"✅ Samples created in: {output_dir}")



#### Execution

In [60]:
path_eval_mask_outline_crop=path_eval_mask_outline+"crop/"
path_img_processing_biopsies=path+"images/img_processing_2/Biopsies/"
if os.path.isdir(path_eval_mask_outline_crop)==False:
  os.mkdir(path_eval_mask_outline_crop)

In [61]:
# 1) Build available markers from one ROI folder (example)
roi0 = sorted([d for d in os.listdir(path_img_processing_biopsies) if os.path.isdir(os.path.join(path_img_processing_biopsies, d))])[0]
list_all_markers = sorted({os.path.splitext(f)[0] for f in os.listdir(os.path.join(path_img_processing_biopsies, roi0))
                            if os.path.splitext(f)[1].lower() in IMG_EXTS})
#
# 2) UI: pick markers + colors
state = interactive_choose_markers_and_colors(list_all_markers)
selected_markers = state["selected_markers"]
marker_to_hex = state["marker_to_hex"]

HTML(value="<h3 style='margin:0 0 10px 0;'>🎨 Select markers & colors</h3>")

SelectMultiple(description='Markers:', layout=Layout(height='260px', width='520px'), options=('Aquaporin1', 'C…

VBox()

Output()

In [62]:
# 3) Run sampling
n_samples = int(input("Number of samples per ROI? (e.g., 6): "))
patch_size = int(input("Patch size (px)? (e.g., 512 or 1024): "))
selected_markers = state["selected_markers"]
marker_to_hex = state["marker_to_hex"]

#
generate_segmentation_samples_multimarker(
     path_img=path_img_processing_biopsies,
     path_mask=path_mask,
     output_dir=path_eval_mask_outline_crop,
     selected_markers=selected_markers,
     marker_to_hex=marker_to_hex,
     n_samples=n_samples,
     patch_size=patch_size,
     contour_color=(255, 255, 255),
     w=1,
     spacer_width=20
 )


Number of samples per ROI? (e.g., 6): 2
Patch size (px)? (e.g., 512 or 1024): 300


Algorithms:   0%|          | 0/5 [00:00<?, ?alg/s]

cellposev3:   0%|          | 0/89 [00:00<?, ?ROI/s]

combine_mesmer_cellposev3:   0%|          | 0/89 [00:00<?, ?ROI/s]

instanseg:   0%|          | 0/89 [00:00<?, ?ROI/s]

mesmer:   0%|          | 0/89 [00:00<?, ?ROI/s]

stardist:   0%|          | 0/89 [00:00<?, ?ROI/s]

✅ Samples created in: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/mask_outline/crop/


### 📊 **Difference between algorithms**
*This cell allows you to create images showing the difference in segmentation between the different algorithms.*

#### Functions

In [63]:
from __future__ import annotations

import os
from pathlib import Path
from itertools import combinations
from typing import Dict, List, Tuple, Optional

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


def compare_masks_without_truth(
    *,
    base_mask_dir: str,
    out_root: str,
    threshold: int = 0,
    fig_size: Tuple[int, int] = (10, 10),
    font_scale: float = 1.0,
    thickness: int = 2,
    color_algo0_bgr: Tuple[int, int, int] = (0, 0,255),  # BGR blue
    color_algo1_bgr: Tuple[int, int, int] = (0, 255, 0),  # BGR green
    progress: bool = True,
    progress_ncols: int = 110,
    skip_missing_pairs: bool = True,
) -> Dict[str, int]:
    """
    Pairwise comparison of binary masks (presence/absence) across algorithms without ground truth.

    Input layout:
        base_mask_dir/
            algoA/
                img001.tif|png|...
            algoB/
                img001.tif|png|...
            ...

    For each pair (algo0, algo1), creates RGB difference images:
      - Red channel = pixels present only in algo0
      - Green channel = pixels present only in algo1
      - Black       = agreement (both 0 or both 1)

    Output layout:
        out_root/
            mask_dif_algo0_algo1/
                img001.png
                ...

    Notes:
      - Masks are binarized with: mask > threshold
      - Adds algo names as text on the image using OpenCV.

    Returns
    -------
    dict summary:
        algos, pairs, processed, skipped_missing, failed, out_root
    """
    base_mask_dir = Path(base_mask_dir).resolve()
    out_root = Path(out_root).resolve()
    out_root.mkdir(parents=True, exist_ok=True)

    if not base_mask_dir.is_dir():
        raise FileNotFoundError(f"base_mask_dir not found: {base_mask_dir}")

    # discover algos
    algos = sorted([p.name for p in base_mask_dir.iterdir() if p.is_dir()])
    if len(algos) < 2:
        raise RuntimeError(f"Need at least 2 algorithm folders in {base_mask_dir}")

    pairs = list(combinations(algos, 2))
    if not pairs:
        raise RuntimeError("No algorithm pairs to compare.")

    processed = 0
    skipped_missing = 0
    failed = 0

    # Build a flat job list (pair × file) -> one clean progress bar
    jobs: List[Tuple[str, str, str]] = []
    for algo0, algo1 in pairs:
        dir0 = base_mask_dir / algo0
        files0 = sorted([p.name for p in dir0.iterdir() if p.is_file()])
        for fname in files0:
            jobs.append((algo0, algo1, fname))

    pbar = tqdm(total=len(jobs), desc="Comparing masks (pairs)", unit="img", ncols=progress_ncols) if progress else None

    # OpenCV text settings
    font = cv2.FONT_HERSHEY_SIMPLEX

    for algo0, algo1, fname in jobs:
        if pbar is not None:
            pbar.set_postfix_str(f"{algo0} vs {algo1} | {fname}", refresh=False)

        try:
            dir0 = base_mask_dir / algo0
            dir1 = base_mask_dir / algo1
            p0 = dir0 / fname
            p1 = dir1 / fname

            if not p1.is_file():
                if skip_missing_pairs:
                    skipped_missing += 1
                    if pbar is not None:
                        pbar.update(1)
                    continue
                raise FileNotFoundError(f"Missing in {algo1}: {p1}")

            # output folder per pair
            out_dir = out_root / f"mask_dif_{algo0}_{algo1}"
            out_dir.mkdir(parents=True, exist_ok=True)
            out_png = out_dir / f"{Path(fname).stem}.png"

            # read masks -> binary (0/1)
            m0 = np.array(Image.open(p0))
            m1 = np.array(Image.open(p1))
            m0 = (m0 > threshold).astype(np.uint8)
            m1 = (m1 > threshold).astype(np.uint8)

            if m0.shape != m1.shape:
                raise ValueError(f"Shape mismatch {fname}: {m0.shape} vs {m1.shape}")

            H, W = m0.shape[:2]
            dif_rgb = np.zeros((H, W, 3), dtype=np.uint8)

            # Red channel = only algo0; Green channel = only algo1
            dif_rgb[..., 0] = np.where(m0 > m1, 255, 0).astype(np.uint8)  # R
            dif_rgb[..., 1] = np.where(m1 > m0, 255, 0).astype(np.uint8)  # G

            # Add labels using OpenCV (expects BGR, so convert temporarily)
            dif_bgr = dif_rgb[..., ::-1].copy()
            dif_bgr = cv2.putText(dif_bgr, algo0, (10, 25), font, font_scale, color_algo0_bgr, thickness, cv2.LINE_AA)
            dif_bgr = cv2.putText(dif_bgr, algo1, (10, 55), font, font_scale, color_algo1_bgr, thickness, cv2.LINE_AA)
            dif_rgb = dif_bgr[..., ::-1]  # back to RGB for matplotlib

            # Save borderless
            plt.figure(figsize=fig_size)
            plt.axis("off")
            plt.imshow(dif_rgb)
            plt.savefig(out_png, bbox_inches="tight", pad_inches=0)
            plt.close()

            processed += 1

        except Exception as e:
            failed += 1
            if pbar is not None:
                pbar.write(f"[WARN] Failed {algo0} vs {algo1} | {fname}: {e}")
            else:
                print(f"[WARN] Failed {algo0} vs {algo1} | {fname}: {e}")

        if pbar is not None:
            pbar.update(1)

    if pbar is not None:
        pbar.close()

    print(f"✅ Done. Outputs in: {out_root}")

    return {
        "algos": len(algos),
        "pairs": len(pairs),
        "processed": processed,
        "skipped_missing": skipped_missing,
        "failed": failed,
        "out_root": str(out_root),
    }



#### Execution

In [64]:
path_eval_diff=path_eval_quali_without_truth+"mask_difference/"
if os.path.isdir(path_eval_diff)==False:
  os.mkdir(path_eval_diff)

In [65]:
report = compare_masks_without_truth(
     base_mask_dir=path_mask,  # root containing algo subfolders
     out_root=path_eval_diff,
     threshold=0,
)
print(report)

Comparing masks (pairs):   0%|                                                       | 0/890 [00:00<?, ?img/s]

✅ Done. Outputs in: /content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/mask_difference
{'algos': 5, 'pairs': 10, 'processed': 890, 'skipped_missing': 0, 'failed': 0, 'out_root': '/content/gdrive/MyDrive/these/pipeline/rejection/segmentation/cells/QC_Segmentation/assessment_without_ground_truth/mask_difference'}


# Segmentation of the crops and assessment with a ground truth (not mandatory)

In [ ]:
path_eval=path_segmentation_cells+"Evaluation_segmentation/"
path_eval_with_truth=path_eval+"assessment_with_ground_truth/"
path_eval_quali_truth=path_eval_with_truth+"segmentation_quality/"
path_eval_mask_outline=path_eval_with_truth+"mask_outline/"
path_crop=path_eval_with_truth+"crop/"
path_mask_crop=path_eval_with_truth+"mask_crop/"
if os.path.isdir(path_eval)==False:
 os.mkdir(path_eval)
if os.path.isdir(path_eval_with_truth)==False:
  os.mkdir(path_eval_with_truth)
if os.path.isdir(path_eval_quali_truth)==False:
  os.mkdir(path_eval_quali_truth)
if os.path.isdir(path_eval_mask_outline)==False:
  os.mkdir(path_eval_mask_outline)
if os.path.isdir(path_crop)==False:
  os.mkdir(path_crop)
if os.path.isdir(path_mask_crop)==False:
  os.mkdir(path_mask_crop)

## define the crops

In [ ]:
roi=input("Enter one name of your images (If you want to choose a random one enter 'r'): ")
d=int(input(" Enter The size of your crop: "))
x=int(input(" Enter the x coordinate of the top left corner of your crop: "))
y=int(input(" Enter the y coordinate of the top left corner of your crop: "))
list_marker=input(" Enter a list of markers (separated with a commma) you want to display on the crop: ")
list_marker=list_marker.split(",")


In [ ]:

if roi == 'r':
    list_roi = os.listdir(path_img_raw)
    name_roi= list_roi[np.random.randint(len(list_roi))]

img_dna = np.arcsinh(plt.imread(path_img_raw + name_roi + "/DNA.png")*5)

img_tot = np.zeros((img_dna.shape[0], img_dna.shape[1], 3))
img_tot[:, :, 0] = img_dna

for marker in list_marker:
  img_marker = np.arcsinh(plt.imread(path_img_raw + name_roi + "/" + marker + ".png") * 20)
  img_tot[:, :, 1] += img_marker

# zone sélectionnée
crop = img_tot[y:y+d, x:x+d]

# affichage avec un carré blanc entourant la zone
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.imshow(img_tot)
# (x, y) = coin supérieur gauche ; largeur=hauteur=d
ax1.add_patch(Rectangle((x, y), d, d, linewidth=2, edgecolor='white', facecolor='none'))
ax1.set_title("Whole image")

ax2.imshow(crop)
ax2.set_title("Crop")

plt.tight_layout()
# Normalize the crop to 0-255 range and convert to uint8 before saving
im = Image.fromarray(normalize_255(crop).astype(np.uint8))
im.save(path_crop+name_roi+"_"+str(x)+"_"+str(y)+"_"+str(d)+".png")
print("✅ Crop available there: "+path_crop+name_roi+"_"+str(x)+"_"+str(y)+"_"+str(d)+".png")
plt.show()

## Segmentation of the crop

### Mesmer


In [ ]:
path_mask_crop_mesmer=path_mask_crop+"mesmer/"
path_mask_crop_geojson=path_eval_quali_with_truth+"geojson/"
if os.path.isdir(path_mask_crop_mesmer)==False:
    os.mkdir(path_mask_crop_mesmer)
    print("✅ Folder for mesmer masks created")
if os.path.isdir(path_mask_crop_geojson)==False:
    os.mkdir(path_mask_crop_geojson)
    print("✅ Folder for geojson masks created")

##### Function

In [ ]:
# Étape 1 : Installer micromamba (mini version d'Anaconda compatible Colab)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!mkdir -p /root/micromamba/envs

# Étape 2 : Créer un environnement Python 3.9 avec micromamba
!./bin/micromamba create -y -p /root/micromamba/envs/deepcell-env python=3.9

# Étape 3 : Activer l'environnement et installer DeepCell + dépendances
!./bin/micromamba run -p /root/micromamba/envs/deepcell-env pip install deepcell==0.12.6 scikit-image matplotlib

# Étape 4 : Démarrer Python dans ce nouvel environnement
import os
from IPython.display import clear_output

os.environ['PYTHONPATH'] = "/root/micromamba/envs/deepcell-env/lib/python3.9/site-packages"
os.environ['PATH'] = "/root/micromamba/envs/deepcell-env/bin:" + os.environ['PATH']
clear_output()
print("✅ Environnement Python 3.9 avec DeepCell prêt dans Colab")


##### Segmentation

In [ ]:
resolution=float(input("Enter the resolution (1.5 micrometer per pixel by default): "))

In [ ]:
code = f"""
from deepcell.applications import Mesmer
from tifffile import imsave
from PIL import Image
import numpy as np
import os
import traceback

project = "rejection"
path = "{path}"
path_crop = "{path_crop}"
path_mask_crop_mesmer = "{path_mask_crop_mesmer}"
path_mask_crop_geojson = "{path_mask_crop_geojson}"


try:
    app = Mesmer()
    print("🧠 Modèle Mesmer chargé avec succès.")
except Exception as e:
    raise RuntimeError("❌ Échec du chargement de Mesmer : " + str(e))

for i,crop in enumerate(os.listdir(path_crop)):
   img = np.array(Image.open(path_crop+crop))
   crop2=img[:,:,:2]
   crop2= np.expand_dims(crop2, axis=0)
   predictions = app.predict(crop2, image_mpp={resolution})
   mask=predictions[0,:,:,0]
   im = Image.fromarray(mask)
   im.save(path_mask_crop_mesmer+"/"+crop[:-4]+".tif")
   print("✅ Done :",crop)


"""


In [ ]:
with open("run_mesmer.py", "w") as f:
    f.write(code)

# Exécution
!./bin/micromamba run -p /root/micromamba/envs/deepcell-env python run_mesmer.py


##### Geojson file
Creation of the geojson files from the maks made by Mesmer to load it in QuPath and manually correct them

In [ ]:
for mask_file in os.listdir(path_mask_crop_mesmer):
   # Load the mask image into a NumPy array
   mask = np.array(Image.open(os.path.join(path_mask_crop_mesmer, mask_file)))
   features = labels_to_features(mask, object_type="annotation")
   geojson = json.dumps(features)
   with open(path_mask_crop_geojson+mask_file[:-4]+".geojson","w") as f:
    dump(features, f)

### Cellpose


In [ ]:
path_mask_crop_cellpose=path_mask_crop+"cellposev3/"
if os.path.isdir(path_mask_crop_cellpose)==False:
    os.mkdir(path_mask_crop_cellpose)
    print("✅ Folder for cellpose masks created")

##### Functions

In [ ]:
%pip install cellpose
from cellpose import core, utils, io, models, metrics
model = models.CellposeModel(gpu=True, model_type="cyto3")


##### Segmentation

In [ ]:
for i,crop in enumerate(os.listdir(path_crop)):
  print(crop)
  img= np.array(Image.open(path_crop+"/"+crop))
  masks_pred, flows, styles = model.eval(img, channels=[2,1],niter=2000) # using more iterations for bacteria
  im = Image.fromarray(masks_pred)
  im.save(path_mask_crop_cellpose+crop[:-4]+".tif")

## InstanSeg

In [ ]:
path_mask_crop_instanseg=path_mask_crop+"instanseg/"
if os.path.isdir(path_mask_crop_instanseg)==False:
    os.mkdir(path_mask_crop_instanseg)
    print("✅ Folder for Instanseg masks created")

#### Installation of Instanseg
(May takes few minutes)

In [ ]:
# === Cellule 1 : installation de l'environnement InstanSeg ===

# 1) Installer micromamba (mini-conda)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!mkdir -p /root/micromamba/envs

# 2) Créer un environnement Python 3.11 pour InstanSeg
!./bin/micromamba create -y -p /root/micromamba/envs/instanseg-env python=3.11

# 3) Installer InstanSeg + dépendances dans cet environnement
!./bin/micromamba run -p /root/micromamba/envs/instanseg-env pip install -q \
    "instanseg-torch[full]" "matplotlib<3.9"


In [ ]:
import os, subprocess, textwrap

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"   # évite le backend Jupyter non supporté

code = """
import matplotlib
matplotlib.use('Agg')

import sys
print("Python utilisé :", sys.version)

from instanseg import InstanSeg
print("InstanSeg importé OK :", InstanSeg)
"""

subprocess.run(
    ["/root/micromamba/envs/instanseg-env/bin/python", "-c", code],
    check=True,
    env=env,
)


In [ ]:
!./bin/micromamba run -p /root/micromamba/envs/instanseg-env pip install requests


#### Segmentation

In [ ]:
import os, subprocess, textwrap
from pathlib import Path

image_dir = path_crop
output_dir = path_mask_crop_instanseg

print("Input folder:", image_dir)
print("Output folder (masks):", output_dir)
print("-" * 70)

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"

code = textwrap.dedent(f"""
import matplotlib
matplotlib.use('Agg')

from instanseg import InstanSeg
from pathlib import Path
import numpy as np
import tifffile as tiff

img_dir = Path({image_dir!r})
out_dir = Path({output_dir!r})

print("Input folder:", img_dir)
print("Output folder:", out_dir)

if not img_dir.exists():
    raise SystemExit(f"Path {{img_dir}} does not exist")

if not img_dir.is_dir():
    raise SystemExit(f"Path {{img_dir}} is not a folder")

out_dir.mkdir(parents=True, exist_ok=True)

print("Loading InstanSeg model (fluorescence_nuclei_and_cells, reader=skimage.io)...")
model = InstanSeg(
    "fluorescence_nuclei_and_cells",
    image_reader="skimage.io",   # <--- CHANGED HERE
    verbosity=1
)

images = sorted(
    list(img_dir.glob("*.tif")) +
    list(img_dir.glob("*.tiff")) +
    list(img_dir.glob("*.png"))
)

print("Number of images found:", len(images))
if not images:
    raise SystemExit("No .tif/.tiff/.png images found in " + str(img_dir))

for img_path in images:
    print("\\nProcessing:", img_path.name)

    labeled_output = model.eval(
        image=str(img_path),
        save_output=False,
        save_overlay=False
    )

    mask = np.asarray(labeled_output)
    mask_path = out_dir / (img_path.stem + ".tif")

    tiff.imwrite(str(mask_path), mask)
    print("Saved mask:", mask_path)

print("Segmentation completed.")
""")

res = subprocess.run(
    ["/root/micromamba/envs/instanseg-env/bin/python", "-c", code],
    env=env,
    capture_output=True,
    text=True,
)

print("----- STDOUT -----")
print(res.stdout)
print("----- STDERR -----")
print(res.stderr)

if res.returncode != 0:
    raise RuntimeError(f"InstanSeg failed (code {res.returncode})")
else:
    print(DONE_ASCII)
    print(f"All outline images have been saved in:\n  {path_mask_crop_instanseg}")

## outline

#### Function

In [ ]:
def outline(img,df,w=1,color=(255,255,255)):
    for i in df.index.to_list():
        img_outline=np.zeros((img.shape[0],img.shape[1]))
        list_pixel= df.loc[i,"coords"]
        for p in list_pixel:
            img_outline[p[0],p[1]]=1
        img_outline=np.uint8(img_outline)
        contours, hierarchy = cv2.findContours(img_outline, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, contours, -1, color, w)
    return img



In [ ]:
def mask_to_df(path_mask,min_size=0,max_size=10000000):
    #img= np.array(Image.open(path_img))
    mask= np.array(Image.open(path_mask))
    label=ski.measure.label(mask,connectivity=mask.ndim)
    df_mask=ski.measure.regionprops_table(label,intensity_image=mask,properties=["area","coords","equivalent_diameter_area","bbox","axis_major_length","axis_minor_length","centroid"])
    df_mask=pd.DataFrame(df_mask)
    df=df_mask[df_mask.loc[:,"area"]>min_size].copy()
    df=df[df.loc[:,"area"]<max_size].copy()
    return df

In [ ]:
from tqdm import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt

DONE_ASCII = r"""
 ____   ___  _   _  _____
|  _ \ / _ \| \ | || ____|
| | | | | | |  \| ||  _|
| |_| | |_| | |\  || |___
|____/ \___/|_| \_||_____|
"""

def display_outline_mask(path_img, path_mask, output_root):
    os.makedirs(output_root, exist_ok=True)

    algos = [a for a in os.listdir(path_mask)
             if os.path.isdir(os.path.join(path_mask, a))]

    total = sum(len(os.listdir(os.path.join(path_mask, algo))) for algo in algos)
    pbar = tqdm(total=total, desc="Outline masks", unit="img")

    for algo in algos:
        out_algo_dir = os.path.join(output_root, algo)
        os.makedirs(out_algo_dir, exist_ok=True)

        algo_mask_dir = os.path.join(path_mask, algo)
        for name_img in sorted(os.listdir(algo_mask_dir)):
            img_name = os.path.splitext(name_img)[0]

            img_dna_path = os.path.join(path_img, img_name + ".png")
            mask_path    = os.path.join(algo_mask_dir, name_img)

            if not os.path.isfile(img_dna_path):
                pbar.update(1)
                continue

            img_dna   = plt.imread(img_dna_path)
            df_mask   = mask_to_df(mask_path)
            img_outline = outline(img_dna, df_mask)

            # --- Normalisation pour imsave ---
            img_to_save = img_outline
            if np.issubdtype(img_to_save.dtype, np.floating):
                # Si les valeurs sont typiquement 0–255 -> ramener en 0–1
                if img_to_save.max() > 1.0 or img_to_save.min() < 0.0:
                    img_to_save = img_to_save.astype(np.float32)
                    img_to_save = np.clip(img_to_save, 0, 255) / 255.0

            out_path = os.path.join(out_algo_dir, img_name + ".png")
            plt.imsave(out_path, img_to_save)

            pbar.update(1)

    pbar.close()
    print(DONE_ASCII)
    print(f"All outline images have been saved in:\n  {output_root}")


#### Execution

In [ ]:
display_outline_mask(path_crop,path_mask_crop,path_eval_mask_outline)


## Average precision and other metrics

### Functions

In [ ]:
def mask_to_df(path_mask,min_size=0,max_size=10000000):
    #img= np.array(Image.open(path_img))
    mask= np.array(Image.open(path_mask))
    label=ski.measure.label(mask,connectivity=mask.ndim)
    df_mask=ski.measure.regionprops_table(label,intensity_image=mask,properties=["area","coords","equivalent_diameter_area","bbox","axis_major_length","axis_minor_length","centroid"])
    df_mask=pd.DataFrame(df_mask)
    df=df_mask[df_mask.loc[:,"area"]>min_size].copy()
    df=df[df.loc[:,"area"]<max_size].copy()
    return df

In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def distribution_number_object(
    path_img="./img",
    path_mask="./mask",
    path_file="./segmentation_quality/",
    dico_algo_min={},
    dico_algo_max={},
    title=""
):

    # ----------------------------------------------------------------------
    # Disable warnings (clean output)
    # ----------------------------------------------------------------------
    warnings.filterwarnings("ignore")

    # ----------------------------------------------------------------------
    # Setup
    # ----------------------------------------------------------------------
    os.makedirs(path_file, exist_ok=True)

    algos = sorted([d.name for d in os.scandir(path_mask) if d.is_dir()])
    if not algos:
        raise RuntimeError(f"No algorithm subfolders found in '{path_mask}'.")

    # Compute common filenames to evaluate per algorithm
    file_sets = []
    for algo in algos:
        algo_dir = Path(path_mask) / algo
        files = {f.name for f in algo_dir.iterdir() if f.is_file()}
        file_sets.append(files)

    if file_sets:
        common_files = sorted(set.intersection(*file_sets))
    else:
        common_files = []

    if not common_files:  # fallback
        first_dir = Path(path_mask) / algos[0]
        common_files = sorted([f.name for f in first_dir.iterdir() if f.is_file()])

    if not common_files:
        raise RuntimeError("No images found in mask folders.")

    # Init result dict
    dico_algo_nb = {algo: [] for algo in algos}

    # ----------------------------------------------------------------------
    # Single progress bar over all (algo, image) pairs
    # ----------------------------------------------------------------------
    total_steps = len(algos) * len(common_files)
    pbar = tqdm(total=total_steps, desc="Counting objects", unit="img", ncols=100)

    for algo in algos:
        algo_dir = Path(path_mask) / algo

        min_size = dico_algo_min.get(algo, 0)
        max_size = dico_algo_max.get(algo, 1_000_000_00)

        for filename in common_files:
            mask_path = algo_dir / filename

            # Use thresholds only if truth not involved
            if algo != "truth" and (dico_algo_min and dico_algo_max):
                df = mask_to_df(mask_path, min_size, max_size)
            else:
                df = mask_to_df(mask_path)

            dico_algo_nb[algo].append(df.shape[0])

            pbar.update(1)

    pbar.close()

    # ----------------------------------------------------------------------
    # Plot 1: Grouped bar chart
    # ----------------------------------------------------------------------
    n_algos = len(algos)
    n_imgs = len(common_files)

    width = 0.8 / n_algos
    x_base = np.arange(n_imgs)

    plt.figure(figsize=(max(12, n_imgs * 0.5), 8))

    for i, algo in enumerate(algos):
        x_positions = x_base + i * width
        plt.bar(x_positions, dico_algo_nb[algo], label=algo, width=width)

    tick_positions = x_base + width * (n_algos - 1) / 2

    plt.title(title, fontsize=20)
    plt.ylabel("Number of objects", fontsize=16)
    plt.xlabel("Images", fontsize=16)
    plt.xticks(tick_positions, common_files, rotation=60, ha="right")
    plt.legend(fontsize=12)
    plt.tight_layout()

    bar_out = Path(path_file) / (title if title else "number_of_objects.png")
    plt.savefig(bar_out, dpi=200)
    plt.close()

    # ----------------------------------------------------------------------
    # Plot 2: Box + swarm distribution
    # ----------------------------------------------------------------------
    rows = []
    for algo in algos:
        for count in dico_algo_nb[algo]:
            rows.append({"Algorithm": algo, "Objects": count})

    df_plot = pd.DataFrame(rows)

    tab20_distinct = [
        "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
        "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B"
    ]

    plt.figure(figsize=(max(12, n_algos * 2), 6))
    sns.swarmplot(data=df_plot, x="Algorithm", y="Objects", color="black", size=3)
    sns.boxplot(data=df_plot, x="Algorithm", y="Objects", showmeans=True, palette=tab20_distinct)

    plt.title("Average " + title, fontsize=14)
    plt.ylabel("Number of objects", fontsize=12)
    plt.xlabel("Algorithm", fontsize=12)
    plt.tight_layout()

    box_out = Path(path_file) / f"mean_{title if title else 'number_of_objects'}.png"
    plt.savefig(box_out, dpi=200)
    plt.close()

    print(f"✅ Files created in: {path_file}")


In [ ]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from matplotlib.patches import Patch

def size_distribution(
    path_mask: str = "./mask",
    path_file: str = "./segmentation_quality",
    dico_algo_min: dict = {},
    dico_algo_max: dict = {},
    title: str = "size_distribution.png"
):

    # ----------------------- Clean terminal output -----------------------
    warnings.filterwarnings("ignore")               # Remove all warnings
    os.makedirs(path_file, exist_ok=True)

    # ----------------------- Scan algorithm subfolders -------------------
    algos = sorted([
        d for d in os.listdir(path_mask)
        if os.path.isdir(os.path.join(path_mask, d))
    ])
    if not algos:
        raise RuntimeError(f"No algorithm subfolders found in '{path_mask}'")

    # Prepare dict for all object areas
    dico_algo_size = {algo: [] for algo in algos}

    # ----------------------- Count total steps for single tqdm -----------
    total_images = 0
    for algo in algos:
        algo_dir = Path(path_mask) / algo
        total_images += len([f for f in algo_dir.iterdir() if f.is_file()])

    # Global_PROGRESS_BAR
    pbar = tqdm(total=total_images, desc="Size object distribution", unit="img", ncols=100)

    # ----------------------- MAIN LOOP (ONE tqdm ONLY) -------------------
    for algo in algos:
        algo_dir = Path(path_mask) / algo
        files = sorted([f.name for f in algo_dir.iterdir() if f.is_file()])

        min_size = dico_algo_min.get(algo, None)
        max_size = dico_algo_max.get(algo, None)

        for fname in files:
            mask_path = algo_dir / fname

            # Threshold logic unchanged
            if min_size is not None and max_size is not None and algo != "truth":
                df_mask = mask_to_df(mask_path, min_size, max_size)
            else:
                df_mask = mask_to_df(mask_path)

            # Extract areas
            if "area" in df_mask.columns:
                dico_algo_size[algo].extend(df_mask["area"].values.tolist())

            pbar.update(1)

    pbar.close()

    # ----------------------- Build DataFrame ------------------------------
    df_plot = pd.DataFrame(
        [(algo, area) for algo, areas in dico_algo_size.items() for area in areas],
        columns=["Algorithm", "Area"]
    )
    if df_plot.empty:
        raise RuntimeError("No areas found. Check input masks or mask_to_df output.")

    # ----------------------- Choose a palette-------------------------------
    tab20_distinct = [
        "#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
        "#17BECF", "#BCBD22", "#E377C2", "#7F7F7F", "#8C564B",
    ]
    if len(algos) <= 20:
        palette = tab20_distinct[:len(algos)]
    else:
        palette = sns.color_palette("husl", n_colors=len(algos))

    algo_to_color = {algo: palette[i] for i, algo in enumerate(algos)}

    # ----------------------- Plot histogram -------------------------------
    plt.figure(figsize=(20, 12))
    ax = plt.gca()
    ax.set_title("Mask object size distribution", fontsize=24)
    ax.set_xlabel("Size (pixels)", fontsize=18)
    ax.set_ylabel("Number of objects", fontsize=18)

    # Overlaid histograms
    for algo in algos:
        areas = dico_algo_size[algo]
        if len(areas) == 0:
            continue

        sns.histplot(
            areas,
            kde=True,
            element="step",
            stat="count",
            common_norm=False,
            bins="auto",
            ax=ax,
            color=algo_to_color[algo],
            label=algo
        )

    # Legend
    handles = [
        Patch(facecolor=algo_to_color[a], edgecolor="black", label=a)
        for a in algos
    ]

    if len(algos) > 10:
        ax.legend(handles=handles, title="Algorithms", loc="upper left",
                  bbox_to_anchor=(1.02, 1), fontsize=12, ncol=2)
        plt.tight_layout(rect=[0, 0, 0.75, 1])
    else:
        ax.legend(handles=handles, title="Algorithms", loc="upper right", fontsize=12)
        plt.tight_layout()

    # Save
    out_path = os.path.join(path_file, title)
    plt.savefig(out_path, dpi=200)
    plt.close()

    print(f"✅ Size distribution saved to: {out_path}")

    return dico_algo_size


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics
from tqdm import tqdm


def calcul_auc(path_img, path_mask, path_file, title,
               dict_min_size=None, dict_max_size=None):
    """
    Compute Average Precision (AP) per algorithm across IoU thresholds,
    and save a precision-vs-IoU plot.
    """

    path_img = Path(path_img)
    path_mask = Path(path_mask)
    path_file = Path(path_file)

    path_file.mkdir(parents=True, exist_ok=True)

    if dict_min_size is None:
        dict_min_size = {}
    if dict_max_size is None:
        dict_max_size = {}

    # IoU thresholds (x-axis)
    iou_thresholds = np.arange(0.0, 1.0, 0.1)  # [0.0, 0.1, ..., 0.9]

    plt.figure(figsize=(20, 12))
    plt.title("Average precision", fontsize=40)

    dict_algo_ap = {}

    # List algorithms (subfolders) except "truth"
    algo_dirs = [
        d for d in sorted(path_mask.iterdir())
        if d.is_dir() and d.name != "truth"
    ]
    truth_dir = path_mask / "truth"

    # --------- Pré-calcul : fichiers communs + total pour tqdm ---------
    algo_to_common_files = {}
    total_images = 0

    truth_files = {f.name for f in truth_dir.iterdir() if f.is_file()}

    for algo_dir in algo_dirs:
        algo = algo_dir.name
        algo_files = {f.name for f in algo_dir.iterdir() if f.is_file()}
        common_files = sorted(algo_files & truth_files)

        if not common_files:
            print(f"⚠️  No common files for algorithm '{algo}'. Skipping.")
        else:
            algo_to_common_files[algo] = common_files
            total_images += len(common_files)

    if total_images == 0:
        print("⚠️  No common images found for any algorithm. Nothing to do.")
        return dict_algo_ap

    # --------- Boucle principale avec barre de progression ---------
    with tqdm(total=total_images, desc="Computing Average Precision", unit="img", ncols=100) as pbar:
        for idx_algo, algo_dir in enumerate(algo_dirs):
            algo = algo_dir.name

            # Algorithme sans fichiers communs → déjà signalé, on passe
            if algo not in algo_to_common_files:
                continue

            common_files = algo_to_common_files[algo]

            min_size = int(dict_min_size.get(algo, 0))
            max_size = int(dict_max_size.get(algo, 100000000))

            precision_sum = np.zeros_like(iou_thresholds, dtype=float)
            recall_sum = np.zeros_like(iou_thresholds, dtype=float)
            count_per_thresh = np.zeros_like(iou_thresholds, dtype=int)

            for fname in common_files:
                algo_path = str(path_mask / algo / fname)
                img_path = str(path_img / (fname[:-4] + ".png"))
                truth_path = str(truth_dir / fname)

                iou, dico_objet, df_mask, df_truth = calcul_iou(
                    img_path,
                    algo_path,
                    truth_path,
                    min_size,
                    max_size
                )

                for i, thr in enumerate(iou_thresholds):
                    recall, precision, tp, fp, fn = recall_precision(
                        thr, df_mask, df_truth
                    )
                    precision_sum[i] += precision
                    recall_sum[i] += recall
                    count_per_thresh[i] += 1

                pbar.update(1)

            valid = count_per_thresh > 0
            mean_precision = np.zeros_like(precision_sum)
            mean_recall = np.zeros_like(recall_sum)

            mean_precision[valid] = precision_sum[valid] / count_per_thresh[valid]
            mean_recall[valid] = recall_sum[valid] / count_per_thresh[valid]

            ap = float(np.round(metrics.auc(iou_thresholds, mean_precision), 2))
            dict_algo_ap[algo] = ap

            plt.plot(iou_thresholds, mean_precision, label=algo)
            plt.text(
                0.05,
                0.05 + idx_algo / 20.0,
                f"Algorithm: {algo}  Average precision = {ap:.2f}",
                fontsize=14,
            )

    plt.xlabel("IoU threshold", fontsize=20)
    plt.ylabel("Precision", fontsize=20)
    plt.ylim(0, 1.05)
    plt.xlim(0, 1.0)
    plt.legend(fontsize=20)
    plt.grid(True, alpha=0.3)

    out_path = path_file / f"{title}.png"
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()

    print(f"✅ AUC plot saved to: {out_path}")
    return dict_algo_ap


In [ ]:
def calcul_iou(path_img,path_mask,path_truth,min_size=0,max_size=10000000):
    df_mask=mask_to_df(path_mask,min_size,max_size)
    df_truth=mask_to_df(path_truth)
    dico_objet,df_mask,df_truth,iou=iou_mean(df_truth,df_mask)
    return iou,dico_objet,df_mask,df_truth

In [ ]:
def recall_precision(thresh,df_mask,df_truth):
    list_true_pos=[i for i in df_mask.loc[:,"iou"] if i>thresh]
    list_false_pos=[i for i in df_mask.loc[:,"iou"] if i<thresh]
    list_false_neg=[i for i in df_truth.loc[:,"iou"] if i<thresh]

    recall=np.around(len(list_true_pos)/df_truth.shape[0],2)
    precision=np.around(len(list_true_pos)/(len(list_true_pos)+len(list_false_pos)),2)

    return recall,precision,len(list_true_pos),len(list_false_pos),len(list_false_neg)

In [ ]:
# make a plot of the mean iou according the maximum size in the predicted mask
def calcul_max_size(path_img,path_mask,path_file,step=0):
    dico_algo_size_iou={}
    for algo in os.listdir(path_mask):
        if algo!="truth":
            dico_algo_size_iou[algo]={}
            for file in os.listdir(path_mask+"/"+algo):
                df_truth=mask_to_df(path_mask+"/truth/"+file,0)
                df_mask=mask_to_df(path_mask+"/"+algo+"/"+file,0)
                if step==0:
                    step=np.max(df_mask["area"])//10
                for size in range(0,int(max(df_mask["area"])),int(step)):
                    if size not in dico_algo_size_iou.keys():
                        dico_algo_size_iou[algo][size]=[]
                    df=df_mask[df_mask.loc[:,"area"]<size].copy()
                    dico_objet,df,df_truth,iou=iou_mean(df_truth,df)
                    dico_algo_size_iou[algo][size].append(iou)
    dico_algo_mean={}
    for k in dico_algo_size_iou.keys():
        dico_algo_mean[k]=[]
        for k2 in dico_algo_size_iou[k].keys():
            dico_algo_mean[k].append(np.mean(dico_algo_size_iou[k][k2]))
    dico_algo_best={}
    for k in dico_algo_size_iou.keys():
        dico_algo_best[k]=list(dico_algo_size_iou[k].keys())[np.argmax(dico_algo_mean[k])]
    plt.figure()
    plt.title("IOU according to the maximum size of an object in the predicted mask")
    plt.ylim(0)
    plt.xlabel("Size in pixel")
    plt.ylabel("IOU")
    for n,k in enumerate(dico_algo_size_iou.keys()):
        plt.plot(list(dico_algo_size_iou[k].keys()),dico_algo_mean[k],label=k)
        plt.text(0,0.95-(n/20),"Algo: "+str(k)+" Best Iou: "+str(np.max(dico_algo_mean[k]))+" for a size of: "+str(list(dico_algo_size_iou[k].keys())[np.argmax(dico_algo_mean[k])])+" µm²")

    plt.legend(loc='upper right')
    plt.savefig(path_file+"IOU_max_size.png")
    return dico_algo_best

def calcul_min_size(path_img,path_mask,path_file,step=100):
    dico_algo_size_iou={}
    for algo in os.listdir(path_mask):
        if algo!="truth":
            dico_algo_size_iou[algo]={}
            for file in os.listdir(path_mask+"/"+algo):
                df_truth=mask_to_df(path_mask+"/truth/"+file,0)
                df_mask=mask_to_df(path_mask+"/"+algo+"/"+file,0)
                if step==0:
                    step=np.max(df_mask["area"])//10
                for size in range(0,int(max(df_truth["area"])//2),int(step)):
                    if size not in dico_algo_size_iou.keys():
                        dico_algo_size_iou[algo][size]=[]
                    df=df_mask[df_mask.loc[:,"area"]>size].copy()
                    dico_objet,df,df_truth,iou=iou_mean(df_truth,df)
                    dico_algo_size_iou[algo][size].append(iou)
    dico_algo_mean={}
    for k in dico_algo_size_iou.keys():
        dico_algo_mean[k]=[]
        for k2 in dico_algo_size_iou[k].keys():
            dico_algo_mean[k].append(np.mean(dico_algo_size_iou[k][k2]))
    dico_algo_best={}
    for k in dico_algo_size_iou.keys():
        dico_algo_best[k]=list(dico_algo_size_iou[k].keys())[np.argmax(dico_algo_mean[k])]
    plt.figure()
    plt.title("IOU according to the minimum size of an object in the predicted mask")
    plt.ylim(0)
    plt.xlabel("Size (µm²)")
    plt.ylabel("IOU")
    for n,k in enumerate(dico_algo_size_iou.keys()):
        plt.plot(list(dico_algo_size_iou[k].keys()),dico_algo_mean[k],label=k)
        plt.text(0.1,0.95-(n/20),"Algo: "+str(k)+" Best Iou: "+str(np.max(dico_algo_mean[k]))+" for a size of: "+str(list(dico_algo_size_iou[k].keys())[np.argmax(dico_algo_mean[k])])+" µm²")
    plt.legend()
    plt.savefig(path_file+"IOU_min_size.png")
    return dico_algo_best

In [ ]:
def outline_iou(path_img,path_file,name,algo,df_truth,df_mask,threshold=0.5):
    if os.path.isdir(path_file+"outline_iou")==False:
        os.mkdir(path_file+"outline_iou")
    if os.path.isdir(path_file+"outline_iou/"+algo)==False:
        os.mkdir(path_file+"outline_iou/"+algo)
    crop= np.array(Image.open(path_img))
    if len(crop.shape)==2:
        crop=cv2.cvtColor(crop,cv2.COLOR_GRAY2RGB)
    for i in df_mask.index.to_list():
        img=np.zeros((crop.shape[0],crop.shape[1]))
        if df_mask.loc[i,"iou"]>threshold:
            color=(0,255,0)
        else:
            color=(0,0,255)
        list_pixel= df_mask.loc[i,"coords"]
        for p in list_pixel:
            img[p[0],p[1]]=1
        img=np.uint8(img)
        contours, hierarchy = cv2.findContours(img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(crop, contours, -1, color, 1)
    for i in df_truth.index.to_list():
        img=np.zeros((crop.shape[0],crop.shape[1]))
        list_pixel= df_truth.loc[i,"coords"]
        for p in list_pixel:
            img[p[0],p[1]]=1
        img=np.uint8(img)
        contours, hierarchy = cv2.findContours(img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(crop, contours, -1, (255,0,0), 1)

    cv2.imwrite(path_file+"outline_iou/"+algo+"/"+name[:-4]+".png",normalize_255(crop))
    return crop


In [ ]:
# Create an image with the predicted mask in different colors and the outline of the ground truth
def outline_mask_color(path_img,path_mask,path_file,algo,threshold,dico_algo_min,dico_algo_max):
    if os.path.isdir(path_file+"outline_color")==False:
        os.mkdir(path_file+"outline_color")
    if os.path.isdir(path_file+"outline_color/"+algo)==False:
        os.mkdir(path_file+"outline_color/"+algo)

    for file_mask in os.listdir(path_mask+"/"+algo):
        img= np.array(Image.open(path_img+"/"+file_mask[:-4]+".png"))
        crop_iou=np.zeros((img.shape[0],img.shape[1],3))
        iou,dico_objet,df_mask,df_truth=calcul_iou(path_img+"/"+file_mask,path_mask+"/"+algo+"/"+file_mask,path_mask+"/truth/"+file_mask,dico_algo_min[algo],dico_algo_max[algo])
        for i in df_mask.index:
            iou_cell=df_mask.loc[i,'iou']
            if iou_cell>threshold:
                color=(0,iou_cell,0)
            elif df_mask.loc[i,'iou']==0:
                color=(0,0,1)
            elif iou<threshold:
                color=(0,iou_cell,iou_cell)

            for p in df_mask.loc[i,"coords"]:
                crop_iou[p[0],p[1]]=color
        img=outline(normalize_255(crop_iou),df_truth,1,(255,255,255))
        cv2.imwrite(path_file+"outline_color/"+algo+"/"+file_mask[:-4]+".png",normalize_255(img))
        #plt.figure()
        #plt.imshow(img)
        #plt.axis('off')
        #plt.savefig(path_file+"outline_color/"+algo+"/"+file_mask[:-4]+".png",bbox_inches = 'tight', pad_inches = 0)

In [ ]:
def iou_mean(df_truth,df_mask):
  dico_objet={}
  list_iou=[]
  df_mask["iou"]=float(0)
  df_truth["iou"]=float(0)
  for i in df_mask.index.to_list():
    centre_mask=(df_mask.loc[i,"centroid-1"],df_mask.loc[i,"centroid-0"])
    diametre_mask=df_mask.loc[i,"equivalent_diameter_area"]+5
    list_cell_mask=[]
    for p in df_mask.loc[i,"coords"]:
          list_cell_mask.append(tuple(p))
    for j in df_truth.index.to_list():
     centre_truth=(df_truth.loc[j,"centroid-1"],df_truth.loc[j,"centroid-0"])
     diametre_truth=df_truth.loc[j,"equivalent_diameter_area"]+5
     distance=np.abs(centre_mask[0]-centre_truth[0])+np.abs(centre_mask[1]-centre_truth[1])
     if distance<diametre_mask/2 or distance<diametre_truth/2:
      list_cell_truth=[]
      for p in df_truth.loc[j,"coords"]:
          list_cell_truth.append(tuple(p))
      total_mask=len(list_cell_mask)
      total_truth=len(list_cell_truth)
      intersection= len(list(set(list_cell_truth).intersection(set(list_cell_mask))))
      if intersection>0:
          iou=intersection/(total_mask+total_truth-intersection)
          if iou>df_mask.loc[i,"iou"] and iou>df_truth.loc[j,"iou"]:
            list_iou.append(iou)
            df_mask.loc[i,"iou"]=iou
            df_truth.loc[j,"iou"]=iou
            dico_objet[(i,j)]=iou

  iou=np.around(np.sum(list_iou)/(df_truth.shape[0]+df_mask.shape[0]-len(list_iou)),2)
  return dico_objet,df_mask,df_truth,iou

In [ ]:
def outline(img,df,w=1,color=(255,255,255)):
    for i in df.index.to_list():
        img_outline=np.zeros((img.shape[0],img.shape[1]))
        list_pixel= df.loc[i,"coords"]
        for p in list_pixel:
            img_outline[p[0],p[1]]=1
        img_outline=np.uint8(img_outline)
        contours, hierarchy = cv2.findContours(img_outline, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, contours, -1, color, w)
    return img

In [ ]:
import os
from pathlib import Path
from contextlib import redirect_stdout
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed


def main_evaluation_fast(
    path_crop,
    path_mask_crop,
    path_eval_quali_truth,
    path_eval_with_truth,
    threshold: float = 0.5,
    step: int = 10,
):

    path_img = Path(path_crop)
    path_mask = Path(path_mask_crop)
    out_path = Path(path_eval_quali_truth)
    out_path.mkdir(parents=True, exist_ok=True)

    summarize_path = out_path / "summarize.txt"

    with open(summarize_path, "w") as f, redirect_stdout(f):

        algos = sorted(
            d.name for d in path_mask.iterdir()
            if d.is_dir() and d.name != "truth"
        )

        print(f"Image path: {path_img}")
        print(f"Mask path:  {path_mask}")
        print(f"Output path: {out_path}")
        print(f"IoU threshold: {threshold}")
        print(f"Step for size optimization: {step}")
        print("\n" + "=" * 80 + "\n")

        # 1. Distributions sans filtre
        size_distribution(
            path_mask=str(path_mask),
            title="size_distribution.png",
            path_file=str(out_path),
        )
        distribution_number_object(
            path_img=str(path_img),
            path_mask=str(path_mask),
            title="number_objects.png",
            path_file=str(out_path),
        )

        # 2. AUC sans filtre
        print("\n\nEvaluation of algorithms without size filter\n")
        dico_algo_ap = calcul_auc(
            str(path_img),
            str(path_mask),
            str(out_path),
            title="Average_precision_without_size_filter",
        )

        # 3. Métriques par algo (sans filtre) en parallèle
        results_no_filter = {}
        with ProcessPoolExecutor() as ex:
            futures = {
                ex.submit(
                    compute_metrics_for_algo,
                    algo,
                    str(path_img),
                    str(path_mask),
                    threshold,
                    None,
                    None,
                ): algo
                for algo in algos
            }
            for fut in as_completed(futures):
                res = fut.result()
                results_no_filter[res["algo"]] = res

        for algo in algos:
            res = results_no_filter.get(algo, None)
            if res is None or not res["iou"]:
                print(f"⚠️  No valid results for algo '{algo}' (no filter).")
                continue

            ious = np.array(res["iou"])
            recalls = np.array(res["recall"])
            precisions = np.array(res["precision"])
            tps = np.array(res["tp"])
            fps = np.array(res["fp"])
            fns = np.array(res["fn"])

            print("*" * 50)
            print(algo)
            print("*" * 50)
            print(f"Mean IoU = {ious.mean():.4f} (without size filter)")
            print(f"Average precision: {dico_algo_ap.get(algo, float('nan'))}")
            print("")
            print(f"Number of predicted objects: {tps.sum() + fps.sum()}")
            print(f"Real object number:         {tps.sum() + fns.sum()}")
            print("")
            print(f"For IoU threshold = {threshold}:")
            print(f"   Recall    = {recalls.mean():.4f}")
            print(f"   Precision = {precisions.mean():.4f}")
            print(f"   True positive  = {tps.sum()}")
            print(f"   False positive = {fps.sum()}")
            print(f"   False negative = {fns.sum()}")
            print("")

        # 4. Optimisation min / max size
        print("\n\nOptimizing object size ranges per algorithm...\n")
        dico_algo_max = calcul_max_size(
            str(path_img), str(path_mask), str(path_eval_quali_truth), step
        )
        dico_algo_min = calcul_min_size(
            str(path_img), str(path_mask), str(path_eval_quali_truth), step
        )

        # 5. AUC avec filtre
        dico_algo_ap_filtered = calcul_auc(
            str(path_img),
            str(path_mask),
            str(out_path),
            title="Average_precision_with_size_filter",
            dict_min_size=dico_algo_min,
            dict_max_size=dico_algo_max,
        )

        # 6. Distributions avec filtre
        size_distribution(
            path_mask=str(path_mask),
            dico_algo_min=dico_algo_min,
            dico_algo_max=dico_algo_max,
            title="size_distribution_filtered.png",
            path_file=str(out_path),
        )
        distribution_number_object(
            path_img=str(path_img),
            path_mask=str(path_mask),
            path_file=str(out_path),
            dico_algo_max=dico_algo_max,
            dico_algo_min=dico_algo_min,
            title="number_objects_filtered.png",
        )

        print("\n\nEvaluation of the algorithms with a size filter\n")

        # 7. Métriques par algo (avec filtre) en parallèle
        results_filtered = {}
        with ProcessPoolExecutor() as ex:
            futures = {
                ex.submit(
                    compute_metrics_for_algo,
                    algo,
                    str(path_img),
                    str(path_mask),
                    threshold,
                    int(dico_algo_min[algo]),
                    int(dico_algo_max[algo]),
                ): algo
                for algo in algos
            }
            for fut in as_completed(futures):
                res = fut.result()
                results_filtered[res["algo"]] = res

        # 8. Affichage + éventuellement visuels (ici on fait visuels en séquentiel)
        for algo in algos:
            res = results_filtered.get(algo, None)
            if res is None or not res["iou"]:
                print(f"⚠️  No valid results for algo '{algo}' (with filter).")
                continue

            ious = np.array(res["iou"])
            recalls = np.array(res["recall"])
            precisions = np.array(res["precision"])
            tps = np.array(res["tp"])
            fps = np.array(res["fp"])
            fns = np.array(res["fn"])

            print("*" * 100)
            print(algo)
            print("*" * 100)
            print(
                f"Best IoU = {ious.mean():.4f} by filtering "
                f"objects < {dico_algo_min[algo]} px and > {dico_algo_max[algo]} px"
            )
            print(
                f"Average precision (with size filter): "
                f"{dico_algo_ap_filtered.get(algo, float('nan'))}"
            )
            print("")
            print(f"Number of predicted objects: {tps.sum() + fps.sum()}")
            print(f"Real object number:         {tps.sum() + fns.sum()}")
            print("")
            print(f"For IoU threshold = {threshold}:")
            print(f"   Recall       = {recalls.mean():.4f}")
            print(f"   Precision    = {precisions.mean():.4f}")
            print(f"   True positive  = {tps.sum()}")
            print(f"   False positive = {fps.sum()}")
            print(f"   False negative = {fns.sum()}")
            print("")

        print(f"\n✅ Evaluation finished. Summary written to: {summarize_path}")


In [ ]:
import os
import sys
from pathlib import Path
from contextlib import redirect_stdout

import numpy as np

# On suppose que ces fonctions existent déjà :
# - size_distribution
# - distribution_number_object
# - calcul_auc
# - calcul_iou
# - recall_precision
# - calcul_max_size
# - calcul_min_size
# - outline_iou
# - outline_mask_color


def _compute_metrics_for_algo(
    algo: str,
    path_img: Path,
    path_mask: Path,
    threshold: float,
    min_size: int | None = None,
    max_size: int | None = None,
    do_outlines: bool = False,
    path_eval_with_truth: Path | None = None,
):
    """
    Calcule les listes :
    - IoU,
    - recall,
    - precision,
    - true_pos, false_pos, false_neg
    pour un algorithme donné, avec ou sans filtre de taille.
    """

    list_iou = []
    list_recall = []
    list_precision = []
    list_true_pos = []
    list_false_pos = []
    list_false_neg = []

    algo_dir = path_mask / algo
    truth_dir = path_mask / "truth"

    # Fichiers communs -> robustesse (au lieu de zip(os.listdir(...)))
    algo_files = {f.name for f in algo_dir.iterdir() if f.is_file()}
    img_files = {f.name for f in path_img.iterdir() if f.is_file()}
    truth_files = {f.name for f in truth_dir.iterdir() if f.is_file()}

    common_files = sorted(algo_files & truth_files & img_files)
    if not common_files:
        print(f"⚠️  No common files for algorithm '{algo}'.")
        return (
            list_iou,
            list_recall,
            list_precision,
            list_true_pos,
            list_false_pos,
            list_false_neg,
        )

    for fname in common_files:
        img_path = path_img / (fname[:-4] + ".png")  # image PNG
        algo_mask_path = algo_dir / fname
        truth_mask_path = truth_dir / fname

        if min_size is None or max_size is None:
            iou, dico_objet, df_mask, df_truth = calcul_iou(
                str(img_path),
                str(algo_mask_path),
                str(truth_mask_path),
            )
        else:
            iou, dico_objet, df_mask, df_truth = calcul_iou(
                str(img_path),
                str(algo_mask_path),
                str(truth_mask_path),
                min_size,
                max_size,
            )

        list_iou.append(iou)
        recall, precision, true_pos, false_pos, false_neg = recall_precision(
            threshold, df_mask, df_truth
        )
        list_precision.append(precision)
        list_recall.append(recall)
        list_true_pos.append(true_pos)
        list_false_pos.append(false_pos)
        list_false_neg.append(false_neg)

        # Visualisations optionnelles
        if do_outlines and path_eval_with_truth is not None:
            outline_iou(
                str(path_img / (fname[:-4] + ".png")),
                str(path_eval_with_truth),
                fname,
                algo,
                df_truth,
                df_mask,
                threshold,
            )
            # selon ton implémentation, cette fonction traite souvent un set complet ;
            # si c'est le cas, la sortir de la boucle serait encore plus optimal.
            outline_mask_color(
                str(path_img),
                str(path_mask),
                str(path_eval_with_truth),
                algo,
                threshold,
                dico_algo_min={algo: min_size} if min_size is not None else {},
                dico_algo_max={algo: max_size} if max_size is not None else {},
            )

    return (
        list_iou,
        list_recall,
        list_precision,
        list_true_pos,
        list_false_pos,
        list_false_neg,
    )


def main_evaluation(
    path_crop,
    path_mask_crop,
    path_eval_quali_truth,
    path_eval_with_truth,
    threshold: float = 0.5,
    step: int = 10,
):

    path_img = Path(path_crop)
    path_mask = Path(path_mask_crop)
    out_path = Path(path_eval_quali_truth)

    out_path.mkdir(parents=True, exist_ok=True)

    summarize_path = out_path / "summarize.txt"
    with open(summarize_path, "w") as f, redirect_stdout(f):

        print(f"Image path: {path_img}")
        print(f"Mask path:  {path_mask}")
        print(f"Output path: {out_path}")
        print(f"IoU threshold for metrics: {threshold}")
        print(f"Step for size optimization: {step}")
        print("\n" + "=" * 80 + "\n")

        # ------------------------------------------------------------------
        # 1. Distributions sans filtre de taille
        # ------------------------------------------------------------------
        print("Size distribution (no size filter)")
        size_distribution(
            path_mask=str(path_mask),
            title="size_distribution.png",
            path_file=str(out_path),
        )

        print("\nNumber of objects per image (no size filter)")
        distribution_number_object(
            path_img=str(path_img),
            path_mask=str(path_mask),
            title="number_objects.png",
            path_file=str(out_path),
        )

        # ------------------------------------------------------------------
        # 2. IoU / AP sans filtre de taille
        # ------------------------------------------------------------------
        print("\n\nEvaluation of algorithms without size filter\n")

        dico_algo_ap = calcul_auc(
            str(path_img),
            str(path_mask),
            str(out_path),
            title="Average_precision_without_size_filter",
        )

        for algo in sorted(os.listdir(path_mask)):
            if algo == "truth":
                continue

            (
                list_iou,
                list_recall,
                list_precision,
                list_true_pos,
                list_false_pos,
                list_false_neg,
            ) = _compute_metrics_for_algo(
                algo=algo,
                path_img=path_img,
                path_mask=path_mask,
                threshold=threshold,
                min_size=None,
                max_size=None,
                do_outlines=False,
            )

            if not list_iou:
                continue

            print("*" * 50)
            print(algo)
            print("*" * 50)
            print(f"Mean IoU = {np.mean(list_iou):.4f} (without size filter)")
            print(f"Average precision: {dico_algo_ap.get(algo, float('nan'))}")
            print("")
            print(
                "Number of predicted objects: "
                f"{np.sum(list_true_pos) + np.sum(list_false_pos)}"
            )
            print(
                "Real object number: "
                f"{np.sum(list_true_pos) + np.sum(list_false_neg)}"
            )
            print("")
            print(f"For IoU threshold = {threshold} :")
            print(f"   Recall   = {np.mean(list_recall):.4f}")
            print(f"   Precision= {np.mean(list_precision):.4f}")
            print(f"   True positives = {np.sum(list_true_pos)}")
            print(f"   False positives= {np.sum(list_false_pos)}")
            print(f"   False negatives= {np.sum(list_false_neg)}")
            print("")

        # ------------------------------------------------------------------
        # 3. Calcul des min/max de taille par algo
        # ------------------------------------------------------------------
        print("\n\nOptimizing object size ranges per algorithm...\n")

        dico_algo_max = calcul_max_size(
            str(path_img), str(path_mask), str(path_eval_quali_truth), step
        )
        dico_algo_min = calcul_min_size(
            str(path_img), str(path_mask), str(path_eval_quali_truth), step
        )

        # ------------------------------------------------------------------
        # 4. AP avec filtre de taille
        # ------------------------------------------------------------------
        dico_algo_ap_filtered = calcul_auc(
            str(path_img),
            str(path_mask),
            str(out_path),
            title="Average_precision_with_size_filter",
            dict_min_size=dico_algo_min,
            dict_max_size=dico_algo_max,
        )

        # ------------------------------------------------------------------
        # 5. Distributions avec filtre de taille
        # ------------------------------------------------------------------
        print("\nSize distribution (with size filter)")
        size_distribution(
            path_mask=str(path_mask),
            dico_algo_min=dico_algo_min,
            dico_algo_max=dico_algo_max,
            title="size_distribution_filtered.png",
            path_file=str(out_path),
        )

        print("\nNumber of objects per image (with size filter)")
        distribution_number_object(
            path_img=str(path_img),
            path_mask=str(path_mask),
            path_file=str(out_path),
            dico_algo_max=dico_algo_max,
            dico_algo_min=dico_algo_min,
            title="number_objects_filtered.png",
        )

        # ------------------------------------------------------------------
        # 6. IoU / AP avec filtre de taille + visualisation
        # ------------------------------------------------------------------
        print("\n\nEvaluation of the algorithms with a size filter\n")

        for algo in sorted(os.listdir(path_mask)):
            if algo == "truth":
                continue

            print("*" * 100)
            print(algo)
            print("*" * 100)

            (
                list_iou,
                list_recall,
                list_precision,
                list_true_pos,
                list_false_pos,
                list_false_neg,
            ) = _compute_metrics_for_algo(
                algo=algo,
                path_img=path_img,
                path_mask=path_mask,
                threshold=threshold,
                min_size=dico_algo_min[algo],
                max_size=dico_algo_max[algo],
                do_outlines=True,
                path_eval_with_truth=Path(path_eval_with_truth),
            )

            if not list_iou:
                continue

            print(
                f"Algorithm: {algo}\n"
                f"Best IoU = {np.mean(list_iou):.4f} by filtering "
                f"objects < {dico_algo_min[algo]} px and > {dico_algo_max[algo]} px"
            )
            print(
                f"Average precision (with size filter): "
                f"{dico_algo_ap_filtered.get(algo, float('nan'))}"
            )
            print("")
            print(
                "Number of predicted objects: "
                f"{np.sum(list_true_pos) + np.sum(list_false_pos)}"
            )
            print(
                "Real object number: "
                f"{np.sum(list_true_pos) + np.sum(list_false_neg)}"
            )
            print("")
            print(f"For IoU threshold = {threshold} :")
            print(f"   Recall       = {np.around(np.mean(list_recall), 2)}")
            print(f"   Precision    = {np.around(np.mean(list_precision), 2)}")
            print(f"   True positive  = {np.sum(list_true_pos)}")
            print(f"   False positive = {np.sum(list_false_pos)}")
            print(f"   False negative = {np.sum(list_false_neg)}")
            print("")

    print(f"✅ Evaluation finished. Summary written to: {summarize_path}")


In [ ]:
from pathlib import Path
import numpy as np

def compute_metrics_for_algo(
    algo: str,
    path_img: str,
    path_mask: str,
    threshold: float,
    min_size: int | None = None,
    max_size: int | None = None,
):
    """
    Compute IoU, recall, precision, TP/FP/FN for one algorithm,
    with optional size filtering.
    Returns a dict with all lists.
    """
    path_img = Path(path_img)
    path_mask = Path(path_mask)
    algo_dir = path_mask / algo
    truth_dir = path_mask / "truth"

    list_iou = []
    list_recall = []
    list_precision = []
    list_true_pos = []
    list_false_pos = []
    list_false_neg = []

    algo_files = {f.name for f in algo_dir.iterdir() if f.is_file()}
    truth_files = {f.name for f in truth_dir.iterdir() if f.is_file()}
    img_files = {f.name for f in path_img.iterdir() if f.is_file()}

    common_files = sorted(algo_files & truth_files & img_files)
    if not common_files:
        return {
            "algo": algo,
            "iou": list_iou,
            "recall": list_recall,
            "precision": list_precision,
            "tp": list_true_pos,
            "fp": list_false_pos,
            "fn": list_false_neg,
        }

    for fname in common_files:
        img_path = path_img / (fname[:-4] + ".png")
        algo_mask_path = algo_dir / fname
        truth_mask_path = truth_dir / fname

        if min_size is None or max_size is None:
            iou, dico_objet, df_mask, df_truth = calcul_iou(
                str(img_path),
                str(algo_mask_path),
                str(truth_mask_path),
            )
        else:
            iou, dico_objet, df_mask, df_truth = calcul_iou(
                str(img_path),
                str(algo_mask_path),
                str(truth_mask_path),
                min_size,
                max_size,
            )

        list_iou.append(iou)
        recall, precision, tp, fp, fn = recall_precision(
            threshold, df_mask, df_truth
        )
        list_precision.append(precision)
        list_recall.append(recall)
        list_true_pos.append(tp)
        list_false_pos.append(fp)
        list_false_neg.append(fn)

    return {
        "algo": algo,
        "iou": list_iou,
        "recall": list_recall,
        "precision": list_precision,
        "tp": list_true_pos,
        "fp": list_false_pos,
        "fn": list_false_neg,
    }


### Execution

In [ ]:
main_evaluation_fast(
    path_crop=path_crop,
    path_mask_crop=path_mask_crop,
    path_eval_quali_truth=path_eval_quali_truth,
    path_eval_with_truth=path_eval_with_truth,
    threshold=0.5,
    step=10,
)
